# MACD mean reversion on 11 US ETFs: self-contained research notebook

**Research artifact only.** This is not financial advice, guarantees no performance, and is not production trading code. Live trading is out of scope. Any future live use would need paper trading, risk limits, kill switches, audit trails and human approval gates.

This notebook is **fully self-contained**: every function it uses is defined in the cells below, and no project Python modules are imported. Its inputs are raw data only:

| Input | Path / source |
|---|---|
| ETF 1-minute bars | `/Users/andrew/main/trading/data/etfs/{T}_clean_1min.csv` |
| NQ futures 1-minute bars | `/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/…csv` (Section 6 only) |
| Event calendar | `data/events/events.csv` (rebuilt from federalreserve.gov + ALFRED by the builder in Section 1 if missing) |
| yfinance daily (adjusted closes, ^VIX, `=F` futures) and FRED DTB3 / VIXCLS | downloaded once, cached under `data/cache/nb/` |

Derived data is cached under `data/cache/nb/` as parquet/CSV. Delete that folder, or set `RECOMPUTE_DATA = True`, to rebuild from scratch. A full run from cache takes roughly 15–30 minutes on a 6-core laptop, mostly the 12,000-variation search, placebos and Monte Carlo.

**Reading order.** hypothesis → design → backtest plan → in-sample (IS) result → out-of-sample (OOS) result → production claim. IS = 2003-01-02..2018-12-31 (train ≤ 2014, validation 2015–2018). OOS = the sealed test split 2019-01-01..2026-09-21. It is evaluated only after the candidates are registered in Section 3 (the registration cell asserts this). *Disclosure:* an earlier version of this study already evaluated this same test window, so it is no longer a pristine holdout. Treat OOS numbers here as a reproduction, not fresh evidence.

In [1]:
# ---- Run flags -------------------------------------------------------------------------------
RECOMPUTE_DATA = False     # True: re-aggregate 1-min CSVs and re-download yfinance/FRED (slow)
FAST_MODE = False          # True: smaller search/placebo/MC sizes for a quick smoke run (results NOT for reporting)

import json, math, time, zlib, hashlib, platform, sys, warnings, itertools, re
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import numba as nb
import scipy
from scipy import stats
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, log_loss, balanced_accuracy_score
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pio.renderers.default = "notebook"       # embeds plotly.js once so figures render offline
pio.templates.default = "plotly_white"
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 80)

ROOT = Path.cwd()
CACHE = ROOT / "data" / "cache" / "nb"
CACHE.mkdir(parents=True, exist_ok=True)
EVENTS_CSV = ROOT / "data" / "events" / "events.csv"
T0 = time.time()

def md(s):
    display(Markdown(s))

def show(df, caption=None, sample=None, fmt=None, n=60):
    """Display a table with an optional caption and IS/OOS sample label."""
    if caption:
        md(f"**{caption}**" + (f" · sample: `{sample}`" if sample else ""))
    d = df.head(n) if len(df) > n else df
    if fmt:
        num = [c for c in d.columns if pd.api.types.is_numeric_dtype(d[c]) and not pd.api.types.is_bool_dtype(d[c])]
        fmt = {c: fmt for c in num} if isinstance(fmt, str) else {c: v for c, v in fmt.items() if c in num}
        display(d.style.format(fmt, na_rep="n/a"))
    else:
        display(d)

def elapsed():
    return f"{(time.time() - T0) / 60:.1f} min"

## 0 · Assumptions, data contract, plan and decision gates

### Configuration (single source of truth)
Every tunable lives in `CFG`. Nothing below hard-codes a parameter that appears here.

In [2]:
CFG = {'project': {'name': 'MACD-MeanReversion-System', 'global_seed': 20261020, 'cache_dir': 'data/cache'},
       'data': {'etf_dir': '/Users/andrew/main/trading/data/etfs',
                'file_pattern': '{ticker}_clean_1min.csv',
                'nq_file': '/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv',
                'tradable': ['SPY', 'QQQ', 'DIA', 'IWM', 'GLD', 'SLV', 'USO', 'CPER', 'TLT', 'IEF', 'SHY'],
                'non_tradable': ['VXX'],
                'benchmark': 'SPY',
                'timezone': 'America/New_York',
                'rth_start': '09:30',
                'rth_last_bar': '15:59',
                'late_open_flag_minutes': 5,
                'early_close_flag_time': '15:55',
                'splice_date': '2022-03-07',
                'price_basis': 'yf_total_return_daily_factor',
                'bad_bar_residual_threshold': 0.01,
                'extreme_return_threshold': 0.15,
                'volume_normalization': 'per_source_segment',
                'volume_warmup_days_after_splice': 20,
                'start_date': '2003-01-02',
                'end_date': '2026-09-21',
                'vix_ticker': '^VIX',
                'rf_fred_series': 'DTB3',
                'yf_futures_tickers': {'ES': 'ES=F',
                                       'NQ': 'NQ=F',
                                       'YM': 'YM=F',
                                       'RTY': 'RTY=F',
                                       'GC': 'GC=F',
                                       'SI': 'SI=F',
                                       'CL': 'CL=F',
                                       'HG': 'HG=F',
                                       'ZN': 'ZN=F',
                                       'ZB': 'ZB=F',
                                       'ZT': 'ZT=F'},
                'iex_high_low': 'union_yf'},
       'events': {'source': 'public_schedules',
                  'fomc_source': 'federalreserve.gov fomchistorical{year}.htm + fomccalendars.htm',
                  'alfred_series': {'CPI': {'series': 'CPIAUCSL', 'impact': 'high'},
                                    'NFP': {'series': 'PAYEMS', 'impact': 'high'},
                                    'GDP': {'series': 'GDP', 'impact': 'high'},
                                    'PCE': {'series': 'PCEPI', 'impact': 'medium'},
                                    'RETAIL_SALES': {'series': 'RSAFS', 'impact': 'medium'},
                                    'PPI': {'series': 'PPIFIS', 'impact': 'medium', 'predecessor': 'PPIFGS'},
                                    'JOBLESS_CLAIMS': {'series': 'ICSA', 'impact': 'low'}},
                  'fomc_impact': 'high',
                  'windows': [-1, 0, 1],
                  'keep_cancelled_scheduled_meetings': True},
       'strategy_base': {'fast': 12,
                         'slow': 26,
                         'signal': 9,
                         'signal_kind': 'hist_z',
                         'z_window': 60,
                         'entry_z': 1.5,
                         'exit_z': 0.0,
                         'require_turn': True,
                         'stop_atr': 2.0,
                         'tp_atr': 0.0,
                         'max_hold': 10,
                         'atr_window': 14,
                         'trend_filter': 'none',
                         'vol_filter': 'none',
                         'volume_filter': 'none',
                         'event_filter': 'none',
                         'side': 'both'},
       'volume_indicator': {'kind': 'relvol', 'relvol_window': 20, 'obv_slope_window': 10},
       'regimes': {'vix_bins': [0, 15, 20, 30, 1000],
                   'vix_labels': ['<15', '15-20', '20-30', '>30'],
                   'spy_trend_sma': 200,
                   'adx_window': 14,
                   'adx_range_threshold': 25,
                   'relvol_bins': [0, 0.7, 1.0, 1.5, 1000],
                   'relvol_labels': ['<0.7', '0.7-1.0', '1.0-1.5', '>1.5'],
                   'realized_vol_window': 20,
                   'rv_pct_lookback': 252,
                   'min_bucket_trades': 30},
       'costs': {'commission_bps': 0.5,
                 'slippage_bps': 1.0,
                 'half_spread_bps': {'SPY': 0.15,
                                     'QQQ': 0.25,
                                     'DIA': 0.5,
                                     'IWM': 0.4,
                                     'GLD': 0.4,
                                     'SLV': 1.5,
                                     'USO': 1.5,
                                     'CPER': 10.0,
                                     'TLT': 0.5,
                                     'IEF': 0.6,
                                     'SHY': 0.6},
                 'borrow_bps_annual': {'default': 50.0, 'USO': 150.0, 'CPER': 300.0, 'SLV': 75.0},
                 'financing_spread_bps_annual': 150.0,
                 'cash_interest': True,
                 'cost_multipliers': [0.0, 0.5, 1.0, 2.0, 3.0, 5.0]},
       'risk': {'init_equity': 100000.0,
                'risk_pct': 0.01,
                'max_leverage': 2.0,
                'portfolio_aggregation': 'equal_weight_daily_rebalanced_available_sleeves',
                'stop_fill': 'stop_price_or_open_if_gapped',
                'same_bar_stop_and_target': 'stop_first'},
       'splits': {'train': ['2003-01-02', '2014-12-31'],
                  'validation': ['2015-01-01', '2018-12-31'],
                  'test': ['2019-01-01', '2026-09-21'],
                  'multi_split_is_years': [1, 2, 3, 4, 5],
                  'multi_split_half': True,
                  'warmup_days': 260},
       'search': {'n_variations': 12000,
                  'seed_offset': 1,
                  'space': {'fast': [5, 8, 10, 12, 15, 19],
                            'slow': [17, 21, 26, 30, 35, 39, 50],
                            'signal': [5, 7, 9, 12],
                            'min_fast_slow_gap': 4,
                            'signal_kind': ['hist_z', 'macd_z'],
                            'z_window': [40, 60, 120, 252],
                            'entry_z': [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0],
                            'exit_z': [1.0, 0.5, 0.0, -0.5],
                            'require_turn': [True, False],
                            'stop_atr': [1.0, 1.5, 2.0, 2.5, 3.0, 4.0],
                            'tp_atr': [0.0, 1.0, 1.5, 2.0, 3.0, 4.0],
                            'max_hold': [3, 5, 10, 15, 20, 30],
                            'trend_filter': ['none', 'own_sma200_align', 'spy_sma200_align', 'adx_range'],
                            'vol_filter': ['none', 'vix_lt_20', 'vix_lt_30', 'vix_gt_20', 'rv_pct_lt_80'],
                            'volume_filter': ['none', 'relvol_gt_1', 'relvol_gt_1.5', 'relvol_lt_1', 'obv_divergence'],
                            'event_filter': ['none',
                                             'skip_high_impact_window',
                                             'skip_any_event_window',
                                             'only_event_window'],
                            'side': ['long_only', 'short_only', 'both']},
                  'selection': {'min_trades_train': 100,
                                'min_trades_validation': 30,
                                'require_train_sharpe_positive': True,
                                'neighborhood_k': 25,
                                'rank_metric': 'validation_sharpe_shrunk_by_neighborhood',
                                'n_candidates': 5}},
       'bootstrap': {'n_mc': 10000,
                     'n_wrc_spa': 1000,
                     'stationary_mean_block': 10,
                     'ci_level': 0.95,
                     'n_ci_boot': 2000,
                     'seed_stability_seeds': [1, 2, 3, 4, 5],
                     'cscv_partitions': 16,
                     'newey_west_lags': 'auto'},
       'placebo': {'n_random_signal': 500,
                   'n_shuffled_returns': 200,
                   'signal_lags': [1, 2, 5, 10],
                   'n_random_params': 2000,
                   'n_random_event_flags': 500},
       'ml': {'model': 'RandomForestClassifier',
              'n_estimators': 500,
              'max_depth': 6,
              'min_samples_leaf': 50,
              'max_features': 'sqrt',
              'class_weight': 'balanced_subsample',
              'label': 'triple_barrier_R',
              'label_r_hi': 1.0,
              'walk_forward_first_test_year': 2008,
              'retrain_every_years': 1,
              'embargo_days': 10,
              'size_multipliers': {'out': 0.0, 'base': 1.0, 'up': 1.5},
              'max_risk_pct_per_trade': 0.015,
              'threshold_grid_out': [0.4, 0.45, 0.5, 0.55, 0.6, 0.65],
              'threshold_grid_up': [0.3, 0.35, 0.4, 0.45, 0.5, 1.01],
              'n_jobs': 2},
       'prop': {'account': '50K',
                'n_mc': 10000,
                'max_days': 252,
                'risk_per_trade_usd_grid': [100, 200, 300, 400, 500, 750],
                'use_micros': True,
                'slippage_ticks_per_side': 1,
                'futures_ratio_window': 5,
                'session': 'rth_intraday',
                'carry_rf': True,
                'assume_micros_listed': False}}

if FAST_MODE:   # smoke-run sizes only
    CFG["search"]["n_variations"] = 1500
    CFG["bootstrap"].update(n_mc=2000, n_wrc_spa=300, n_ci_boot=500)
    CFG["placebo"].update(n_random_signal=100, n_shuffled_returns=30, n_random_params=300, n_random_event_flags=100)
    CFG["prop"]["n_mc"] = 2000

def cfg_hash(obj) -> str:
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:12]

def rng(name: str) -> np.random.Generator:
    """Deterministic named RNG stream: PCG64(SeedSequence(global_seed, spawn_key=crc32(name)))."""
    ss = np.random.SeedSequence(CFG["project"]["global_seed"], spawn_key=(zlib.crc32(name.encode()),))
    return np.random.Generator(np.random.PCG64(ss))

def seed_for(name: str) -> int:
    return int(rng(name).integers(0, 2**31 - 1))

CONFIG_HASH = cfg_hash(CFG)
print("config hash", CONFIG_HASH, "| FAST_MODE", FAST_MODE)
print(json.dumps(CFG, indent=1)[:3000], "…")

config hash 753c8c440127 | FAST_MODE False
{
 "project": {
  "name": "MACD-MeanReversion-System",
  "global_seed": 20261020,
  "cache_dir": "data/cache"
 },
 "data": {
  "etf_dir": "/Users/andrew/main/trading/data/etfs",
  "file_pattern": "{ticker}_clean_1min.csv",
  "nq_file": "/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv",
  "tradable": [
   "SPY",
   "QQQ",
   "DIA",
   "IWM",
   "GLD",
   "SLV",
   "USO",
   "CPER",
   "TLT",
   "IEF",
   "SHY"
  ],
  "non_tradable": [
   "VXX"
  ],
  "benchmark": "SPY",
  "timezone": "America/New_York",
  "rth_start": "09:30",
  "rth_last_bar": "15:59",
  "late_open_flag_minutes": 5,
  "early_close_flag_time": "15:55",
  "splice_date": "2022-03-07",
  "price_basis": "yf_total_return_daily_factor",
  "bad_bar_residual_threshold": 0.01,
  "extreme_return_threshold": 0.15,
  "volume_normalization": "per_source_segment",
  "volume_warmup_days_after_splice": 20,
  "start_date": "2003-01-02",
  "end_date": "2026

### Assumptions & data contract

| ID | Assumption | Handling |
|----|------------|----------|
| A1 | ETF data = daily OHLCV, adjustable | **Modified:** files are **1-minute** RTH bars (`datetime,Open,High,Low,Close,Volume,source`), naive US/Eastern. They are resampled to daily RTH bars (09:30–15:59). |
| A2 | Event data with timestamps/impact/actual/forecast | **Modified:** no event file was supplied. "myforexfunds" is a defunct prop firm, not a FRED dataset (ambiguity flagged). The calendar is built from public schedules: FOMC from federalreserve.gov and CPI/NFP/GDP/PCE/Retail Sales/PPI/Jobless Claims from ALFRED vintage dates. It has **dates and rule-based impact only; no actual/forecast/surprise values.** |
| A3 | VIX via yfinance, comparison only | ^VIX from yfinance, cross-checked against FRED VIXCLS. VXX is audited but never traded. |
| A4 | 1% risk to the stop, ATR(14) stop | shares = 1%·sleeve equity / (stop_atr·ATR14). Notional is capped at 2× sleeve equity, and the cap binds on low-volatility SHY/IEF. Gaps can lose more than 1R. |
| A5 | Base MACD 12/26/9 | `CFG["strategy_base"]`; variations in `CFG["search"]["space"]` |
| A6 | Signal at close t, fill at open t+1 | Engine semantics; a perturbation test in Section 2 verifies causality |
| A7 | Costs incl. commission, spread, slippage, borrow, financing | Per-ETF bps are **assumptions, not measured**. Cost multipliers 0–5× are reported. |
| A8 | Prop rules configurable; provisional if absent | Topstep values transcribed from public help pages (2026-10-20) into `PROP_RULES` in Section 6. **PROVISIONAL.** |
| A9 | Price basis | The vendor's pitrading segment (≤2022-03-04) and IEX segment (≥2022-03-07) are dividend-adjusted as of different dates, which creates fake splice gaps (SPY −5.8%, TLT −14%). Daily O/H/L/C are rebased to the yfinance total-return close by a per-day factor, and inconsistent bars are replaced by yfinance (flagged). |
| A10 | Volume | IEX volume is single-venue (~1–3% of consolidated), so relvol and OBV are computed **within each source segment**. After 2022-03-07 they are a noisier proxy for market volume. |
| A11 | IEX high/low | Single-venue ranges are too narrow. Using the union with the yfinance consolidated high/low is conservative for stop touches. |
| A12 | Portfolio | Equal-weight, daily-rebalanced mean of available sleeves (one per ETF). Cash earns DTB3, and Sharpe is computed on excess returns. Fractional shares. |
| A13 | Event timing | Scheduled dates are known in advance. Shutdown-delayed releases (2013, 2018-19, 2025) were announced only days ahead, a small point-in-time caveat. Unscheduled FOMC actions and cancelled meetings are never used as forward flags. |
| A14 | Prop admissibility | Topstep requires being flat by 15:10 CT. The multi-day swing strategy is **inadmissible as-is**, so Section 6 simulates an intraday (entry-day) adaptation. |
| A15 | Universe | 11 ETFs that exist today, chosen ex post: **survivorship and selection bias cannot be excluded**. |

| Data | Range | Fields | TZ | Adjustment | Missing data | PIT caveats |
|---|---|---|---|---|---|---|
| `{T}_clean_1min.csv` ×11 tradable (SPY QQQ DIA IWM GLD SLV USO CPER TLT IEF SHY) + VXX | 2002-12-30 → 2026-09-21 (CPER from 2011, SLV/USO 2006, GLD 2004, VXX 2022) | OHLCV + source ∈ {pitrading, iex} | naive America/New_York | vendor back-adjustment with two anchors → rebased (A9) | calendar = yfinance SPY days; local gaps filled from yfinance inside the local span (flagged) | the 15:59 bar ≠ official close (6–13 bp pre-2009) |
| NQ 1-min | 2010-06-06 → 2026-01-23 | ts_event UTC, OHLCV, symbol | UTC → ET | raw contracts | front = max-volume outright per day | Section 6 mapping check only |
| yfinance / FRED | full | Adj Close, VIX, `=F` futures, DTB3 | exchange dates | yfinance auto-adjust | cached | re-downloads change adjusted *levels* only |
| `events.csv` | 2003 → 2026 | date, event, impact, release_time_et, scheduled, cancelled, cancel_announced, source_url | trading date of the release | — | non-trading days → next NYSE day | Jobless Claims only from 2009-05-28 |

### Plan and decision gates
Each gate's outcome is computed at run time and summarised in Section 7.

| Gate | Criterion (fixed before the run) |
|---|---|
| G0 Data integrity | All files import; splice repaired (output gap within 1% of yfinance's); no unexplained extreme returns |
| G1 IS baseline edge | Base 12/26/9 portfolio IS Sharpe > 0 with NW p < 0.05 |
| G2 Search significance | Primary DSR ≥ 0.95, Hansen SPA_c p < 0.05, PBO < 0.5, ≥ 1 Holm discovery |
| G3 Sealed OOS | Registration precedes test access; primary OOS Sharpe > 0 with block-bootstrap 95% CI lower bound > 0 |
| G4 Multi-split robustness | Re-selected #1 positive and HAC-significant OOS in ≥ 4 of 6 splits, and beats SPY in ≥ 4 of 6 |
| G5 ML improvement | ML Sharpe > base Sharpe OOS and paired-bootstrap Sharpe-difference CI > 0 |
| G6 Prop viability | OOS P(pass) ≥ 25% and P(pass) > P(fail), under PROVISIONAL rules |
| G7 Paper trading | Only if G1–G6 pass; never reached otherwise |

### Leakage / look-ahead design checklist
- Indicators are causal (rolling/ewm on data ≤ t). A perturbation test in Section 2 changes all data after k and asserts identical signals, fills and equity up to k.
- Fills happen at open t+1; stops/targets are checked intrabar with stop-first; gaps fill at the open.
- Event filters use scheduled dates only. The entry window around t+1 is legitimate because schedules are published in advance.
- Selection reads train/validation only (the grid for selection is simulated **only up to 2018-12-31**). Candidates are hashed and registered before any test access.
- ML features are timestamped ≤ signal date. Labels are purged (training exits < fold start) with a 10-day embargo and no shuffling.

### Reproducibility

In [3]:
def env_report() -> pd.DataFrame:
    rows = [("python", sys.version.split()[0]), ("platform", platform.platform())]
    for name, mod in [("numpy", np), ("pandas", pd), ("scipy", scipy), ("numba", nb), ("scikit-learn", sklearn),
                      ("statsmodels", statsmodels), ("plotly", plotly)]:
        rows.append((name, mod.__version__))
    try:
        import yfinance as yf
        rows.append(("yfinance", yf.__version__))
    except Exception as e:
        rows.append(("yfinance", f"missing: {e}"))
    rows += [("global_seed", CFG["project"]["global_seed"]), ("config_hash", CONFIG_HASH),
             ("run_utc", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")),
             ("rng scheme", "named streams: PCG64(SeedSequence(global_seed, spawn_key=crc32(name)))"),
             ("bootstrap indices", "deterministic: every resampling call takes its own named stream")]
    return pd.DataFrame(rows, columns=["item", "value"])

show(env_report(), "Environment and seeds")

**Environment and seeds**

,item,value
0,python,3.13.9
1,platform,macOS-27.0.1-arm64-arm-64bit-Mach-O
2,numpy,2.3.5
3,pandas,2.3.3
4,scipy,1.16.3
5,numba,0.62.1
6,scikit-learn,1.7.2
7,statsmodels,0.14.5
8,plotly,6.3.0
9,yfinance,1.5.1


## 1 · Data import and audit

### 1.1 Raw 1-minute bars → daily RTH bars
Each CSV is read once, de-duplicated, sorted and restricted to 09:30–15:59 ET, then aggregated to daily bars (open = first, high = max, low = min, close = last, volume = sum). Per-file integrity counts are kept for the audit.

In [4]:
DCFG = CFG["data"]
TRADABLE, ALL_ETF = DCFG["tradable"], DCFG["tradable"] + DCFG["non_tradable"]
SPLICE = pd.Timestamp(DCFG["splice_date"])
START, END = pd.Timestamp(DCFG["start_date"]), pd.Timestamp(DCFG["end_date"])

def aggregate_1min(ticker: str) -> tuple[pd.DataFrame, dict]:
    out, meta_f = CACHE / f"daily_raw_{ticker}.parquet", CACHE / f"daily_raw_{ticker}.json"
    if out.exists() and meta_f.exists() and not RECOMPUTE_DATA:
        return pd.read_parquet(out), json.loads(meta_f.read_text())
    path = Path(DCFG["etf_dir"]) / DCFG["file_pattern"].format(ticker=ticker)
    df = pd.read_csv(path, engine="pyarrow")
    df["datetime"] = pd.to_datetime(df["datetime"])
    meta = dict(file=path.name, n_minute_rows=len(df), out_of_order=int((df["datetime"].diff().dt.total_seconds() < 0).sum()))
    meta["duplicate_minutes"] = int(df.duplicated("datetime").sum())
    df = df.drop_duplicates("datetime").sort_values("datetime")
    px_cols = ["Open", "High", "Low", "Close"]
    nonpos = (df[px_cols] <= 0).any(axis=1)
    meta["zero_or_negative_prices"] = int(nonpos.sum())
    df = df[~nonpos]
    tod = df["datetime"].dt.hour * 60 + df["datetime"].dt.minute
    meta["outside_rth_rows"] = int(((tod < 570) | (tod > 959)).sum())
    df = df[(tod >= 570) & (tod <= 959)].assign(tod=tod)
    g = df.groupby(df["datetime"].dt.normalize())
    d = pd.DataFrame({"open": g["Open"].first(), "high": g["High"].max(), "low": g["Low"].min(),
                      "close": g["Close"].last(), "volume": g["Volume"].sum().astype(float), "n_bars": g.size(),
                      "first_min": g["tod"].first(), "last_min": g["tod"].last(), "source": g["source"].last()})
    d.index.name = "date"
    d.to_parquet(out)
    meta_f.write_text(json.dumps(meta))
    return d, meta

def yf_daily(ticker: str, name: str | None = None) -> pd.DataFrame:
    import yfinance as yf
    f = CACHE / f"yf_{(name or ticker).replace('^', '').replace('=', '_')}.parquet"
    if f.exists() and not RECOMPUTE_DATA:
        return pd.read_parquet(f)
    d = yf.download(ticker, start="2002-11-01", end=(END + pd.Timedelta(days=30)).strftime("%Y-%m-%d"),
                    auto_adjust=False, actions=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    d.index = pd.to_datetime(d.index).tz_localize(None).normalize()
    d = d[~d.index.duplicated()].sort_index()
    d.to_parquet(f)
    return d

def fred_series(sid: str) -> pd.Series:
    f = CACHE / f"fred_{sid}.csv"
    if not f.exists() or RECOMPUTE_DATA:
        s = pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}")
        s.to_csv(f, index=False)
    s = pd.read_csv(f)
    s.columns = ["date", sid]
    s["date"] = pd.to_datetime(s["date"])
    return pd.to_numeric(s.set_index("date")[sid], errors="coerce")

raw, raw_meta = {}, {}
for t in ALL_ETF:
    raw[t], raw_meta[t] = aggregate_1min(t)
yfd = {t: yf_daily(t) for t in ALL_ETF}
vix_yf = yf_daily(DCFG["vix_ticker"], "VIX")
fut_yf = {k: yf_daily(v, f"{k}_F") for k, v in DCFG["yf_futures_tickers"].items()}

CAL = yfd["SPY"].index[(yfd["SPY"].index >= START) & (yfd["SPY"].index <= END)]
NCAL = len(CAL)
print(f"master calendar: {CAL[0].date()} → {CAL[-1].date()} ({NCAL} trading days) · {elapsed()}")

master calendar: 2003-01-02 → 2026-09-21 (5967 trading days) · 0.0 min


### 1.2 Price basis: splice repair and bad-bar replacement (A9, A11)
For every day, factor `f_t = yfAdjClose_t / localClose_t` and output `O/H/L/C = local × f_t`. Closes therefore equal the yfinance total-return close, and the local intraday shape is kept. A local bar is **replaced by yfinance adjusted OHLC** when any of these holds:
- it is structurally invalid;
- its close residual against the trailing 21-day median factor of its source segment exceeds 1%;
- its open differs from a credible yfinance open by more than 1%;
- its high/low lies more than 1% outside the consolidated range.

Calendar days with no local bar are filled from yfinance inside the local span. yfinance rows with zero volume are untrusted and never used. On the IEX segment, high/low use the union with the consolidated range.

In [5]:
def build_bars(t: str) -> tuple[pd.DataFrame, dict]:
    thr = DCFG["bad_bar_residual_threshold"]
    r = raw[t].reindex(CAL)
    y = yfd[t].reindex(CAL)
    trusted = (y["Volume"] > 0) & (y["Close"] > 0)
    adj = y["Adj Close"] / y["Close"]
    ya = pd.DataFrame({c: y[c.capitalize()] * adj for c in ["open", "high", "low", "close"]})
    has_local = r["close"].notna()
    span = (CAL >= raw[t].index.min()) & (CAL <= raw[t].index.max())
    seg = pd.Series(np.where(CAL >= SPLICE, "iex", "pitrading"), index=CAL)
    f = (ya["close"] / r["close"]).where(trusted & has_local)
    logf = np.log(f)
    base = logf.groupby(seg).transform(lambda s: s.shift(1).rolling(21, min_periods=5).median())
    resid = (logf - base).abs()
    structural = (r["high"] < r["low"]) | (r["open"] > r["high"] * (1 + 1e-9)) | (r["open"] < r["low"] * (1 - 1e-9)) | \
                 (r["close"] > r["high"] * (1 + 1e-9)) | (r["close"] < r["low"] * (1 - 1e-9))
    credible_open = (y["Open"] != y["Close"]) & (y["Open"] != y["Close"].shift(1))
    open_bad = credible_open & (np.log(r["open"] * f / ya["open"]).abs() > thr)
    hl_bad = (r["high"] * f > ya["high"] * (1 + thr)) | (r["low"] * f < ya["low"] * (1 - thr))
    bad = has_local & trusted & (structural | (resid > thr) | open_bad | hl_bad)
    f_used = f.groupby(seg).ffill()                       # local bar on an untrusted yf day: previous factor
    out = pd.DataFrame(index=CAL)
    for c in ["open", "high", "low", "close"]:
        out[c] = r[c] * f_used
    iex_ok = has_local & trusted & (seg == "iex") & ~bad
    out.loc[iex_ok, "high"] = np.maximum(out.loc[iex_ok, "high"], ya.loc[iex_ok, "high"])
    out.loc[iex_ok, "low"] = np.minimum(out.loc[iex_ok, "low"], ya.loc[iex_ok, "low"])
    fill = (~has_local) & span & trusted
    repl = bad | fill
    for c in ["open", "high", "low", "close"]:
        out.loc[repl, c] = ya.loc[repl, c]
    out["volume"] = r["volume"].where(has_local & ~repl)
    out["source"] = seg.where(out["close"].notna())
    out["bad_bar"], out["filled_from_yf"] = bad, fill
    raw_ret = r["close"].pct_change()
    pre, post = CAL[CAL < SPLICE][-1], CAL[CAL >= SPLICE][0]
    info = dict(n_bad_replaced=int(bad.sum()), n_filled=int(fill.sum()),
                splice_jump_raw_pct=100 * (r.at[post, "open"] / r.at[pre, "close"] - 1) if span[CAL.get_loc(post)] and has_local[pre] else np.nan,
                splice_gap_yf_pct=100 * (ya.at[post, "open"] / ya.at[pre, "close"] - 1),
                splice_gap_out_pct=100 * (out.at[post, "open"] / out.at[pre, "close"] - 1),
                median_resid_bp=1e4 * resid[has_local & trusted].median(),
                corr_local_vs_yf=raw_ret.corr(ya["close"].pct_change()),
                iex_volume_vs_yf=float((r["volume"] / y["Volume"])[(seg == "iex") & has_local & trusted].median()),
                pit_volume_vs_yf=float((r["volume"] / y["Volume"])[(seg == "pitrading") & has_local & trusted].median()))
    return out, info

BARS, BAR_INFO = {}, {}
for t in ALL_ETF:
    BARS[t], BAR_INFO[t] = build_bars(t)

### 1.3 Data-audit table

In [6]:
def audit_row(t):
    m, i, b, r = raw_meta[t], BAR_INFO[t], BARS[t], raw[t]
    ret = b["close"].pct_change()
    local_days = r.index[(r.index >= START) & (r.index <= END)]
    span = (CAL >= r.index.min()) & (CAL <= r.index.max())
    return dict(ticker=t, tradable=t in TRADABLE, file=m["file"], first_date=r.index.min().date(), last_date=r.index.max().date(),
                minute_rows=m["n_minute_rows"], out_of_order=m["out_of_order"], duplicate_minutes=m["duplicate_minutes"],
                zero_or_neg=m["zero_or_negative_prices"], days_local=len(local_days), calendar_days_in_span=int(span.sum()),
                missing_local_days=int(span.sum() - len(local_days)), late_open_days=int((r["first_min"] > 570 + DCFG["late_open_flag_minutes"]).sum()),
                early_close_days=int((r["last_min"] < 955).sum()), pitrading_days=int((r["source"] == "pitrading").sum()),
                iex_days=int((r["source"] == "iex").sum()), bad_bars_replaced=i["n_bad_replaced"], filled_from_yf=i["n_filled"],
                extreme_returns=int((ret.abs() > DCFG["extreme_return_threshold"]).sum()),
                splice_jump_raw_pct=i["splice_jump_raw_pct"], splice_gap_yf_pct=i["splice_gap_yf_pct"], splice_gap_out_pct=i["splice_gap_out_pct"],
                median_resid_bp=i["median_resid_bp"], corr_local_ret_vs_yf=i["corr_local_vs_yf"],
                vol_ratio_pitrading_vs_yf=i["pit_volume_vs_yf"], vol_ratio_iex_vs_yf=i["iex_volume_vs_yf"])

AUDIT = pd.DataFrame([audit_row(t) for t in ALL_ETF])
show(AUDIT, "Data audit (all files sorted, de-duplicated and aligned to the master calendar)", "n/a",
     fmt={c: "{:.3f}" for c in AUDIT.columns if AUDIT[c].dtype == float})
md("**Adjusted vs unadjusted.** Both vendor segments are dividend back-adjusted, but as of different dates (≈2022-03 and ≈2026-03). The raw splice jumps above equal the dividends paid between the two anchors. After rebasing, every splice gap matches yfinance's adjusted gap. "
   "**Survivorship.** The universe consists of 11 ETFs that still trade today, chosen ex post. Delisted ETFs are absent, so survivorship and selection bias cannot be excluded (A15).")

ext = []
for t in ALL_ETF:
    r_ = BARS[t]["close"].pct_change()
    for d_, v in r_[r_.abs() > DCFG["extreme_return_threshold"]].items():
        ext.append(dict(ticker=t, date=d_.date(), ret_pct=100 * v, yf_adj_ret_pct=100 * (yfd[t]["Adj Close"].pct_change().get(d_, np.nan))))
show(pd.DataFrame(ext), "Extreme daily returns (|r| > 15%) cross-checked against yfinance", "n/a", fmt={"ret_pct": "{:.2f}", "yf_adj_ret_pct": "{:.2f}"})

**Data audit (all files sorted, de-duplicated and aligned to the master calendar)** · sample: `n/a`

,ticker,tradable,file,first_date,last_date,minute_rows,out_of_order,duplicate_minutes,zero_or_neg,days_local,calendar_days_in_span,missing_local_days,late_open_days,early_close_days,pitrading_days,iex_days,bad_bars_replaced,filled_from_yf,extreme_returns,splice_jump_raw_pct,splice_gap_yf_pct,splice_gap_out_pct,median_resid_bp,corr_local_ret_vs_yf,vol_ratio_pitrading_vs_yf,vol_ratio_iex_vs_yf
0,SPY,True,SPY_clean_1min.csv,2002-12-30,2026-09-21,2250959,0,0,0,5966,5967,1,114,141,4828,1140,17,1,0,-5.826,-0.143,-0.148,1.867,0.988,0.772,0.016
1,QQQ,True,QQQ_clean_1min.csv,2002-12-30,2026-09-21,2245015,0,0,0,5966,5967,1,127,120,4828,1140,12,1,0,-2.779,-0.003,-0.037,2.308,0.994,0.814,0.013
2,DIA,True,DIA_clean_1min.csv,2002-12-30,2026-09-21,2067170,0,0,0,5964,5967,3,136,125,4826,1140,11,3,0,-7.368,-0.377,-0.372,2.216,0.986,0.871,0.013
3,IWM,True,IWM_clean_1min.csv,2002-12-30,2026-09-21,2217669,0,0,0,5964,5967,3,149,105,4826,1140,16,3,0,-5.075,0.282,0.167,2.771,0.992,0.789,0.020
4,GLD,True,GLD_clean_1min.csv,2004-11-18,2026-09-21,1924936,0,0,0,5490,5493,3,189,71,4350,1140,5,3,0,0.414,0.419,0.403,1.141,0.998,0.823,0.015
5,SLV,True,SLV_clean_1min.csv,2006-04-28,2026-09-21,1752691,0,0,0,5128,5131,3,162,63,3988,1140,7,3,3,-0.444,-0.423,-0.444,3.694,0.997,0.913,0.017
6,USO,True,USO_clean_1min.csv,2006-04-10,2026-09-21,1707848,0,0,0,5141,5144,3,335,57,4001,1140,14,3,5,0.567,0.591,0.559,2.702,0.997,0.888,0.012
7,CPER,True,CPER_clean_1min.csv,2011-11-15,2026-09-21,109059,0,0,0,3315,3732,417,1709,1901,2215,1100,107,92,1,-4.078,-3.445,-4.013,0.116,0.736,0.932,0.014
8,TLT,True,TLT_clean_1min.csv,2002-12-30,2026-09-21,2005004,0,0,0,5964,5967,3,173,126,4826,1140,1,3,0,-14.248,-0.799,-0.905,2.332,0.978,0.848,0.025
9,IEF,True,IEF_clean_1min.csv,2002-12-30,2026-09-21,1560831,0,0,0,5964,5967,3,297,193,4826,1140,0,3,0,-12.206,-0.469,-0.527,1.612,0.935,0.891,0.025


**Adjusted vs unadjusted.** Both vendor segments are dividend back-adjusted, but as of different dates (≈2022-03 and ≈2026-03). The raw splice jumps above equal the dividends paid between the two anchors. After rebasing, every splice gap matches yfinance's adjusted gap. **Survivorship.** The universe consists of 11 ETFs that still trade today, chosen ex post. Delisted ETFs are absent, so survivorship and selection bias cannot be excluded (A15).

**Extreme daily returns (|r| > 15%) cross-checked against yfinance** · sample: `n/a`

,ticker,date,ret_pct,yf_adj_ret_pct
0,SLV,2008-10-10,-17.99,-17.99
1,SLV,2026-01-30,-28.54,-28.54
2,SLV,2026-02-05,-15.77,-15.77
3,USO,2020-03-09,-25.32,-25.32
4,USO,2020-03-18,-17.51,-17.51
5,USO,2020-04-02,16.67,16.67
6,USO,2020-04-03,15.46,15.46
7,USO,2020-04-21,-25.07,-25.07
8,CPER,2025-07-30,-19.31,-19.31
9,VXX,2024-08-02,24.09,24.09


In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["SPY: raw local close vs repaired (log)", "TLT: raw local close vs repaired (log)"])
for j, t in enumerate(["SPY", "TLT"], start=1):
    rc = raw[t]["close"].reindex(CAL)
    fig.add_trace(go.Scatter(x=CAL, y=rc / rc.dropna().iloc[0], name=f"{t} raw local", line=dict(width=1)), 1, j)
    oc = BARS[t]["close"]
    fig.add_trace(go.Scatter(x=CAL, y=oc / oc.dropna().iloc[0], name=f"{t} repaired", line=dict(width=1)), 1, j)
    fig.add_vline(x=SPLICE, line_dash="dot", line_color="red", row=1, col=j)
fig.update_yaxes(type="log")
fig.update_layout(height=380, title="Splice repair: the raw vendor series drops at 2022-03-07 (red), the repaired series does not")
fig.show()

vr = []
for t in TRADABLE:
    v = (raw[t]["volume"] / yfd[t]["Volume"].reindex(raw[t].index)).replace([np.inf, -np.inf], np.nan)
    vr.append(v.resample("QE").median().rename(t))
vr = pd.concat(vr, axis=1)
fig = px.line(vr, log_y=True, title="Local volume / consolidated (yfinance) volume, quarterly median: the IEX segment is single-venue (A10)")
fig.add_vline(x=SPLICE, line_dash="dot", line_color="red")
fig.update_layout(height=360, yaxis_title="ratio")
fig.show()

### 1.4 VIX, risk-free rate and event calendar
VIX close (16:15 ET) on day t is known before the open at t+1, so it may drive decisions filled at t+1. DTB3 (percent, annualised) becomes a daily rate (1+r)^(1/252)−1.

In [8]:
VIX = vix_yf["Close"].reindex(CAL).ffill(limit=3)
vix_fred = fred_series("VIXCLS").reindex(CAL)
vdiff = (VIX - vix_fred).abs()
rf_ann = (fred_series(DCFG["rf_fred_series"]) / 100).reindex(CAL.union(fred_series(DCFG["rf_fred_series"]).index)).ffill().reindex(CAL)
RF = ((1 + rf_ann.fillna(0)) ** (1 / 252) - 1).values
show(pd.DataFrame([dict(series="^VIX yf vs FRED VIXCLS", common_days=int(vdiff.notna().sum()),
                        days_absdiff_gt_0_05=int((vdiff > 0.05).sum()), median_absdiff=vdiff.median(), max_absdiff=vdiff.max(),
                        max_date=vdiff.idxmax().date() if vdiff.notna().any() else None, vix_nan_after_ffill=int(VIX.isna().sum()),
                        rf_mean_annual_pct=100 * rf_ann.mean())]), "VIX cross-check and risk-free rate", "n/a")

def build_event_calendar() -> pd.DataFrame:
    """Rebuild events.csv from public schedules (used only if the CSV is missing).
    FOMC: federalreserve.gov historical pages (≤2020) + current calendar (2021+); day 0 = last meeting day.
    Macro: ALFRED vintage dates (vintage date == publication date). Simplified relative to the original
    build (which separated revision-only vintages using initial-release downloads); documented as such."""
    import requests
    from bs4 import BeautifulSoup
    hdr = {"User-Agent": "Mozilla/5.0 (research; macd-mr notebook)"}
    months = {m: i for i, m in enumerate(["january", "february", "march", "april", "may", "june", "july", "august",
                                          "september", "october", "november", "december"], start=1)}
    rows = []
    def add_meeting(year, label):
        m = re.match(r"(\w+)(?:/(\w+))?\s+(\d+)(?:-(\d+))?", label.strip())
        if not m or m.group(1).lower() not in months:
            return
        mon1, mon2, d1, d2 = m.group(1).lower(), (m.group(2) or m.group(1)).lower(), int(m.group(3)), int(m.group(4) or m.group(3))
        end_month = months.get(mon2, months[mon1])
        rows.append(dict(date=pd.Timestamp(year, end_month, d2), event="FOMC", impact=CFG["events"]["fomc_impact"],
                         release_time_et="14:00", scheduled="unscheduled" not in label.lower(),
                         cancelled="cancel" in label.lower(), source_url="federalreserve.gov", note=label.strip()))
    for year in range(2003, 2021):
        html = requests.get(f"https://www.federalreserve.gov/monetarypolicy/fomchistorical{year}.htm", headers=hdr, timeout=30).text
        for lab in re.findall(r"((?:January|February|March|April|May|June|July|August|September|October|November|December)[^<]{0,40}?Meeting)", html):
            add_meeting(year, lab.replace("Meeting", ""))
        time.sleep(0.3)
    soup = BeautifulSoup(requests.get("https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm", headers=hdr, timeout=30).text, "html.parser")
    for panel in soup.select("div.panel"):
        h = panel.find(["h4", "h5"])
        ym = re.search(r"(20\d\d)", h.get_text() if h else "")
        if not ym:
            continue
        for row in panel.select("div.fomc-meeting"):
            mo = row.select_one(".fomc-meeting__month")
            dt = row.select_one(".fomc-meeting__date")
            if mo and dt:
                add_meeting(int(ym.group(1)), f"{mo.get_text(strip=True).split('/')[0]} {dt.get_text(strip=True).replace('*', '')}")
    for name, spec in CFG["events"]["alfred_series"].items():
        for sid in [spec["series"]] + ([spec["predecessor"]] if "predecessor" in spec else []):
            html = requests.get(f"https://alfred.stlouisfed.org/series/downloaddata?seid={sid}", headers=hdr, timeout=60).text
            for d_ in sorted(set(re.findall(r"(20\d\d-\d\d-\d\d)", html))):
                rows.append(dict(date=pd.Timestamp(d_), event=name, impact=spec["impact"], release_time_et="08:30", scheduled=True,
                                 cancelled=False, source_url=f"https://alfred.stlouisfed.org/series/downloaddata?seid={sid}", note="ALFRED vintage"))
            time.sleep(0.3)
    ev = pd.DataFrame(rows).drop_duplicates(["date", "event"]).sort_values("date")
    ev["cancel_announced"] = None
    EVENTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    ev.to_csv(EVENTS_CSV, index=False)
    return ev

if not EVENTS_CSV.exists():
    print("events.csv missing → rebuilding from public schedules")
    build_event_calendar()
EVENTS = pd.read_csv(EVENTS_CSV, parse_dates=["date"])
for c in ["scheduled", "cancelled"]:
    EVENTS[c] = EVENTS[c].astype(str).str.lower().isin(["true", "1"])
ev_tab = EVENTS.assign(year=EVENTS["date"].dt.year).pivot_table(index="event", columns="year", values="date", aggfunc="count", fill_value=0)
show(ev_tab, "Event calendar: rows per type and year (scheduled releases; *_REVISION and FOMC_NO_STATEMENT rows are low impact and never used in 'any/high' flags)", "n/a")
md(f"Impact counts: {EVENTS['impact'].value_counts().to_dict()}. Cancelled scheduled meetings kept for reference: "
   f"{EVENTS.loc[EVENTS['cancelled'], 'date'].dt.date.tolist()} (excluded from realised flags; the 2020-03-18 cancellation was announced 2020-03-15, before any affected decision).")

**VIX cross-check and risk-free rate** · sample: `n/a`

,series,common_days,days_absdiff_gt_0_05,median_absdiff,max_absdiff,max_date,vix_nan_after_ffill,rf_mean_annual_pct
0,^VIX yf vs FRED VIXCLS,5967,9,3.433228e-07,2.610001,2026-02-06,0,1.724439


**Event calendar: rows per type and year (scheduled releases; *_REVISION and FOMC_NO_STATEMENT rows are low impact and never used in 'any/high' flags)** · sample: `n/a`

year,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
event,,,,,,,,,,,,,,,,,,,,,,,,
CPI,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,11,10
CPI_REVISION,0,0,1,1,1,1,1,1,1,1,1,1,1,0,1,0,1,1,1,1,1,1,0,0
FOMC,8,8,8,8,10,11,8,9,8,8,8,8,8,8,8,8,9,14,8,8,8,8,9,8
FOMC_NO_STATEMENT,5,0,0,0,1,3,3,1,2,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
GDP,13,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,11,12,12,12,12,12,10,10
JOBLESS_CLAIMS,0,0,0,0,0,0,32,52,52,52,52,53,52,52,52,52,52,53,52,52,52,52,46,41
JOBLESS_CLAIMS_REVISION,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0
NFP,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,12,11,10
NFP_REVISION,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


Impact counts: {'high': 1060, 'low': 969, 'medium': 854}. Cancelled scheduled meetings kept for reference: [datetime.date(2020, 3, 18)] (excluded from realised flags; the 2020-03-18 cancellation was announced 2020-03-15, before any affected decision).

### 1.5 NQ futures (used only to validate the QQQ→NQ mapping in Section 6)
Calendar spreads (symbols containing '-') are dropped. Timestamps are converted UTC → ET and restricted to RTH, and the front contract is the highest-volume outright each day.

In [9]:
def load_nq_daily() -> pd.DataFrame:
    f = CACHE / "nq_daily.parquet"
    if f.exists() and not RECOMPUTE_DATA:
        return pd.read_parquet(f)
    import pyarrow as pa, pyarrow.csv as pacsv, pyarrow.compute as pc
    tbl = pacsv.read_csv(DCFG["nq_file"], convert_options=pacsv.ConvertOptions(
        include_columns=["ts_event", "open", "high", "low", "close", "volume", "symbol"],
        column_types={"ts_event": pa.string(), "open": pa.float64(), "high": pa.float64(), "low": pa.float64(),
                      "close": pa.float64(), "volume": pa.float64(), "symbol": pa.string()}))
    tbl = tbl.filter(pc.invert(pc.match_substring(tbl["symbol"], "-")))
    df = tbl.to_pandas()
    ts = pd.to_datetime(df["ts_event"].str.slice(0, 19), utc=True).dt.tz_convert("America/New_York")
    tod = ts.dt.hour * 60 + ts.dt.minute
    keep = (tod >= 570) & (tod <= 959)
    df = df.loc[keep].assign(date=ts[keep].dt.tz_localize(None).dt.normalize(), ts=ts[keep])
    df = df.sort_values("ts")
    g = df.groupby(["date", "symbol"])
    d = pd.DataFrame({"open": g["open"].first(), "high": g["high"].max(), "low": g["low"].min(), "close": g["close"].last(), "volume": g["volume"].sum()}).reset_index()
    d = d.sort_values(["date", "volume"]).groupby("date").tail(1).set_index("date").sort_index()
    d.to_parquet(f)
    return d

NQ = load_nq_daily()
show(pd.DataFrame([dict(first=NQ.index.min().date(), last=NQ.index.max().date(), days=len(NQ), contracts_used=NQ["symbol"].nunique(),
                        rolls=int((NQ["symbol"] != NQ["symbol"].shift()).sum() - 1))]), "NQ daily front-month coverage", "n/a")
EXT = pd.DataFrame(ext)
G0_PASS = bool((AUDIT["splice_gap_out_pct"] - AUDIT["splice_gap_yf_pct"]).abs().max() < 1.0
               and (EXT.empty or (EXT["ret_pct"] - EXT["yf_adj_ret_pct"]).abs().max() < 0.5))
print("G0 data integrity:", "PASS" if G0_PASS else "FAIL")
print("Section 1 done ·", elapsed())

**NQ daily front-month coverage** · sample: `n/a`

,first,last,days,contracts_used,rolls
0,2010-06-07,2026-01-23,4016,40,63


G0 data integrity: PASS
Section 1 done · 0.0 min


## 2 · Baseline MACD mean-reversion examination (in-sample only)

**Hypothesis.** MACD extremes revert, especially in range-bound regimes. Trend, volatility, volume and event filters may improve or degrade the edge.

**Exact signal rules** (evaluated at close t, filled at open t+1):
- MACD = EMA_fast(C) − EMA_slow(C), signal = EMA_sig(MACD), hist = MACD − signal.
- x_t = rolling z-score (window `z_window`) of the histogram (`hist_z`) or of the MACD line (`macd_z`).
- **Long entry:** x_t ≤ −entry_z, and if `require_turn`, hist_t > hist_{t−1}. **Short entry:** mirror image.
- **Exit:** a long exits when x_t ≥ −exit_z, a short when x_t ≤ +exit_z. Otherwise exit after `max_hold` bars, at the stop (`stop_atr`·ATR14 from the entry price) or at the target (`tp_atr`·ATR14, 0 = none).
- Stops and targets are checked intrabar from the entry bar. A gap through the stop fills at the open. If stop and target are touched in the same bar, the stop is assumed first.
- One position per ETF sleeve. Exit-and-reverse on the same open is allowed.
- **Sizing:** risk 1% of sleeve equity at t to the stop, notional ≤ 2× equity.
- **Costs per fill:** commission + half-spread + slippage (bps), plus short borrow and a financing spread on negative cash. Cash earns DTB3.

**Volume indicator (configurable, documented choice):** `relvol` = volume / rolling 20-day median volume, computed **within each data-source segment** (pitrading vs single-venue IEX, A10). The alternative, `obv_slope`, is the OLS slope of on-balance volume over 10 days divided by the 20-day median volume. Both are in the search space as filters.

The base strategy (A5) is 12/26/9, hist_z(60), entry 1.5, exit 0, turn required, 2 ATR stop, no target, 10-bar max hold, long and short, no filters. **Every signal is taken.** This section uses **IS data only** (≤ 2018-12-31).

In [10]:
SB, RCFG, CC, RK = CFG["strategy_base"], CFG["regimes"], CFG["costs"], CFG["risk"]
IS_END = pd.Timestamp(CFG["splits"]["validation"][1])
TR_END = pd.Timestamp(CFG["splits"]["train"][1])
TEST_START = pd.Timestamp(CFG["splits"]["test"][0])
I_IS_END = int(CAL.searchsorted(IS_END, side="right"))            # exclusive end index of IS
I_TR_END = int(CAL.searchsorted(TR_END, side="right"))
I_TEST = int(CAL.searchsorted(TEST_START))
WARM = CFG["splits"]["warmup_days"]
CHECKS = []                                                          # consolidated checklist for Section 7

def check(section, item, ok, evidence, status=None):
    CHECKS.append(dict(section=section, item=item, status=status or ("PASS" if ok else "FAIL"), evidence=evidence))

# ---------- causal indicators (computed on valid bars only, then re-aligned to the calendar) ----------
def _valid(b):
    return b["close"].notna() & b["open"].notna()

def wilder(x: pd.Series, n: int) -> pd.Series:
    return x.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()

def atr_adx(b: pd.DataFrame, n: int) -> tuple[pd.Series, pd.Series]:
    h, l, c = b["high"], b["low"], b["close"]
    c1 = c.shift(1)
    tr = pd.concat([h - l, (h - c1).abs(), (l - c1).abs()], axis=1).max(axis=1)
    atr = wilder(tr, n)
    up, dn = h.diff(), -l.diff()
    pdm = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=b.index)
    mdm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=b.index)
    pdi, mdi = 100 * wilder(pdm, n) / atr, 100 * wilder(mdm, n) / atr
    dx = 100 * (pdi - mdi).abs() / (pdi + mdi)
    return atr, wilder(dx, n)

def seg_relvol(v: pd.Series, seg: pd.Series, w: int) -> pd.Series:
    out = pd.Series(np.nan, index=v.index)
    for s in seg.dropna().unique():
        vv = v[(seg == s) & v.notna() & (v > 0)]
        out.loc[vv.index] = vv / vv.rolling(w, min_periods=w).median()
    return out

def seg_obv_slope(c: pd.Series, v: pd.Series, seg: pd.Series, w: int, wmed: int) -> pd.Series:
    out = pd.Series(np.nan, index=c.index)
    k = np.arange(w) - (w - 1) / 2
    for s in seg.dropna().unique():
        m = (seg == s) & v.notna() & c.notna()
        cc, vv = c[m], v[m]
        obv = (np.sign(cc.diff()).fillna(0) * vv).cumsum()
        slope = pd.Series(np.convolve(obv.values, k[::-1] / (k ** 2).sum(), mode="full")[: len(obv)], index=obv.index)
        slope.iloc[: w - 1] = np.nan
        out.loc[obv.index] = slope / vv.rolling(wmed, min_periods=wmed).median()
    return out

def static_features(b: pd.DataFrame, spy_close: pd.Series) -> dict:
    """Parameter-independent features of one ETF, all known at the close of t."""
    ok = _valid(b)
    bv = b[ok]
    atr, adx = atr_adx(bv, SB["atr_window"])
    lr = np.log(bv["close"]).diff()
    rv = lr.rolling(RCFG["realized_vol_window"]).std() * np.sqrt(252)
    rv_pct = rv.rolling(RCFG["rv_pct_lookback"], min_periods=RCFG["rv_pct_lookback"]).apply(lambda w: (w <= w[-1]).mean(), raw=True)
    vw = CFG["volume_indicator"]
    f = dict(atr=atr, adx=adx, sma200=bv["close"].rolling(200).mean(), rv=rv, rv_pct=rv_pct,
             relvol=seg_relvol(bv["volume"], bv["source"], vw["relvol_window"]),
             obv_slope=seg_obv_slope(bv["close"], bv["volume"], bv["source"], vw["obv_slope_window"], vw["relvol_window"]))
    out = {k: v.reindex(CAL).values for k, v in f.items()}
    spy_sma = spy_close.rolling(RCFG["spy_trend_sma"]).mean()
    out.update(open=b["open"].values, high=b["high"].values, low=b["low"].values, close=b["close"].values,
               valid=ok.values, spy_up=(spy_close > spy_sma).values, spy_trend=(spy_close / spy_sma - 1).values,
               vix=VIX.values)
    return out

def macd_features(close: np.ndarray, valid: np.ndarray, fast, slow, signal, kind, zw) -> dict:
    c = pd.Series(close[valid])
    m = c.ewm(span=fast, adjust=False).mean() - c.ewm(span=slow, adjust=False).mean()
    s = m.ewm(span=signal, adjust=False).mean()
    hist = m - s
    base = hist if kind == "hist_z" else m
    x = (base - base.rolling(zw).mean()) / base.rolling(zw).std()
    x.iloc[: slow + signal] = np.nan                                  # EMA warm-up
    out = {}
    for k, v in dict(x=x, hist=hist, hist_prev=hist.shift(1), macd=m).items():
        a = np.full(len(close), np.nan)
        a[valid] = v.values
        out[k] = a
    return out

# ---------- events (scheduled, not cancelled) → calendar flags ----------
def event_flags(ev: pd.DataFrame, cal: pd.DatetimeIndex = CAL) -> pd.DataFrame:
    real = ev[ev["scheduled"] & ~ev["cancelled"] & ~ev["event"].str.endswith("_REVISION")]
    idx = cal.searchsorted(real["date"].values)                      # non-trading day → next trading day
    keep = idx < len(cal)
    real, idx = real[keep], idx[keep]
    out = pd.DataFrame(index=cal)
    for name, imps in [("any", ("high", "medium")), ("high", ("high",)), ("low", ("low",))]:
        d0 = np.zeros(len(cal), bool)
        d0[idx[real["impact"].isin(imps).values]] = True
        out[f"{name}_d0"] = d0
    for typ in ["FOMC", "CPI", "NFP", "GDP", "PCE", "RETAIL_SALES", "PPI", "JOBLESS_CLAIMS"]:
        d0 = np.zeros(len(cal), bool)
        d0[idx[(real["event"] == typ).values]] = True
        out[f"{typ}_d0"] = d0
    for name in ["any", "high", "low"]:
        d0 = out[f"{name}_d0"]
        out[f"{name}_dm1"] = d0.shift(-1, fill_value=False)           # event tomorrow
        out[f"{name}_dp1"] = d0.shift(1, fill_value=False)            # event yesterday
        out[f"{name}_win"] = out[f"{name}_d0"] | out[f"{name}_dm1"] | out[f"{name}_dp1"]
    return out

EVF = event_flags(EVENTS)
# Window around the ENTRY day t+1, read at signal time t (legitimate: schedules are published in advance).
EV_NEXT = {k: np.append(EVF[f"{k}_win"].values[1:], False) for k in ["any", "high"]}

FEAT = {t: static_features(BARS[t], BARS["SPY"]["close"]) for t in TRADABLE}
FIRST_IDX = {t: int(np.argmax(FEAT[t]["valid"])) for t in TRADABLE}
print("features ready ·", elapsed())

features ready · 0.0 min


In [11]:
# ---------- signals ----------
def build_signals(F: dict, M: dict, p: dict, ev_next=EV_NEXT) -> tuple:
    x, h, hp = M["x"], M["hist"], M["hist_prev"]
    with np.errstate(invalid="ignore"):
        le, se = x <= -p["entry_z"], x >= p["entry_z"]
        if p["require_turn"]:
            le, se = le & (h > hp), se & (h < hp)
        tf = p["trend_filter"]
        if tf == "own_sma200_align":
            le, se = le & (F["close"] > F["sma200"]), se & (F["close"] < F["sma200"])
        elif tf == "spy_sma200_align":
            le, se = le & F["spy_up"], se & ~F["spy_up"] & ~np.isnan(F["spy_trend"])
        elif tf == "adx_range":
            ok = F["adx"] < RCFG["adx_range_threshold"]
            le, se = le & ok, se & ok
        vf = p["vol_filter"]
        if vf != "none":
            ok = {"vix_lt_20": F["vix"] < 20, "vix_lt_30": F["vix"] < 30, "vix_gt_20": F["vix"] > 20,
                  "rv_pct_lt_80": F["rv_pct"] < 0.8}[vf]
            le, se = le & ok, se & ok
        uf = p["volume_filter"]
        if uf == "obv_divergence":
            le, se = le & (F["obv_slope"] > 0), se & (F["obv_slope"] < 0)
        elif uf != "none":
            ok = {"relvol_gt_1": F["relvol"] > 1, "relvol_gt_1.5": F["relvol"] > 1.5, "relvol_lt_1": F["relvol"] < 1}[uf]
            le, se = le & ok, se & ok
        ef = p["event_filter"]
        if ef != "none":
            ok = {"skip_high_impact_window": ~ev_next["high"], "skip_any_event_window": ~ev_next["any"],
                  "only_event_window": ev_next["any"]}[ef]
            le, se = le & ok, se & ok
        if p["side"] == "long_only":
            se = np.zeros_like(se)
        elif p["side"] == "short_only":
            le = np.zeros_like(le)
        lx, sx = x >= -p["exit_z"], x <= p["exit_z"]
    return le, se, lx, sx

# ---------- backtest engine (numba) ----------
TRADE_COLS = ["signal_idx", "entry_idx", "exit_idx", "side", "entry_px", "exit_px", "shares", "risk_usd",
              "pnl_net", "costs", "eq_entry", "reason", "notional"]
REASONS = {1: "stop", 2: "gap_stop", 3: "signal", 4: "time", 5: "target", 6: "eod"}

@nb.njit(cache=False)
def simulate(o, h, l, c, atr, rf, le, se, lx, sx, smult, stop_atr, tp_atr, max_hold, risk_pct, max_risk_pct,
             max_lev, cost, borrow, fin, eq0, start, end):
    n = len(c)
    eq = np.full(n, np.nan)
    expo = np.zeros(n)
    traded = np.zeros(n)
    tr = np.zeros((end - start + 2, 13))
    ntr = 0
    cash, pos, side = eq0, 0.0, 0
    entry_px = stop = tgt = risk_usd = eq_entry = e_cost = 0.0
    held, e_idx, s_idx, pend_exit, pend_entry, pend_sig = 0, -1, -1, 0, 0, -1
    last_c, eq_prev = np.nan, eq0
    for t in range(start, end):
        if cash > 0:                                   # carry on yesterday's balances
            cash += cash * rf[t]
        else:
            cash += cash * (rf[t] + fin / 252.0)
        if pos < 0:
            cash -= (-pos) * last_c * borrow / 252.0
        valid = (c[t] == c[t]) and (o[t] == o[t])
        if valid:
            if pend_exit != 0 and pos != 0:            # exit at the open
                px = o[t]
                cst = abs(pos) * px * cost
                cash += pos * px - cst
                traded[t] += abs(pos) * px
                tr[ntr, 0] = s_idx; tr[ntr, 1] = e_idx; tr[ntr, 2] = t; tr[ntr, 3] = side
                tr[ntr, 4] = entry_px; tr[ntr, 5] = px; tr[ntr, 6] = abs(pos); tr[ntr, 7] = risk_usd
                tr[ntr, 8] = side * abs(pos) * (px - entry_px) - e_cost - cst; tr[ntr, 9] = e_cost + cst
                tr[ntr, 10] = eq_entry; tr[ntr, 11] = pend_exit; tr[ntr, 12] = abs(pos) * entry_px
                ntr += 1
                pos, side = 0.0, 0
            pend_exit = 0
            if pend_entry != 0 and pos == 0:           # entry at the open
                a, m = atr[pend_sig], smult[pend_sig]
                if a == a and a > 0 and m == m and m > 0:
                    rk = min(risk_pct * m, max_risk_pct) * eq_prev
                    dist = stop_atr * a
                    sh = rk / dist
                    cap = max_lev * eq_prev / o[t]
                    if sh > cap:
                        sh = cap
                    px = o[t]
                    cst = sh * px * cost
                    side = pend_entry
                    pos = side * sh
                    cash -= pos * px + cst
                    traded[t] += sh * px
                    entry_px, stop = px, px - side * dist
                    tgt = px + side * tp_atr * a if tp_atr > 0 else np.nan
                    risk_usd, eq_entry, e_cost, e_idx, s_idx, held = sh * dist, eq_prev, cst, t, pend_sig, 0
            pend_entry = 0
            if pos != 0:                               # intrabar stop / target (stop first)
                xp, rs = np.nan, 0
                if side == 1:
                    if o[t] <= stop:
                        xp, rs = o[t], 2
                    elif l[t] <= stop:
                        xp, rs = stop, 1
                    elif tp_atr > 0 and o[t] >= tgt:
                        xp, rs = o[t], 5
                    elif tp_atr > 0 and h[t] >= tgt:
                        xp, rs = tgt, 5
                else:
                    if o[t] >= stop:
                        xp, rs = o[t], 2
                    elif h[t] >= stop:
                        xp, rs = stop, 1
                    elif tp_atr > 0 and o[t] <= tgt:
                        xp, rs = o[t], 5
                    elif tp_atr > 0 and l[t] <= tgt:
                        xp, rs = tgt, 5
                if rs != 0:
                    cst = abs(pos) * xp * cost
                    cash += pos * xp - cst
                    traded[t] += abs(pos) * xp
                    tr[ntr, 0] = s_idx; tr[ntr, 1] = e_idx; tr[ntr, 2] = t; tr[ntr, 3] = side
                    tr[ntr, 4] = entry_px; tr[ntr, 5] = xp; tr[ntr, 6] = abs(pos); tr[ntr, 7] = risk_usd
                    tr[ntr, 8] = side * abs(pos) * (xp - entry_px) - e_cost - cst; tr[ntr, 9] = e_cost + cst
                    tr[ntr, 10] = eq_entry; tr[ntr, 11] = rs; tr[ntr, 12] = abs(pos) * entry_px
                    ntr += 1
                    pos, side = 0.0, 0
            last_c = c[t]
            if pos != 0:                               # decisions at the close
                held += 1
                if (side == 1 and lx[t]) or (side == -1 and sx[t]):
                    pend_exit = 3
                elif held >= max_hold:
                    pend_exit = 4
            if t + 1 < end:
                if le[t] and (pos == 0 or (pend_exit != 0 and side == -1)):
                    pend_entry, pend_sig = 1, t
                elif se[t] and (pos == 0 or (pend_exit != 0 and side == 1)):
                    pend_entry, pend_sig = -1, t
        else:
            pend_entry = 0
        eq[t] = cash + (pos * last_c if pos != 0 else 0.0)
        expo[t] = abs(pos * last_c) / eq[t] if pos != 0 else 0.0
        eq_prev = eq[t]
    if pos != 0:                                       # end of window: close at the last close
        px = last_c
        cst = abs(pos) * px * cost
        cash += pos * px - cst
        traded[end - 1] += abs(pos) * px
        tr[ntr, 0] = s_idx; tr[ntr, 1] = e_idx; tr[ntr, 2] = end - 1; tr[ntr, 3] = side
        tr[ntr, 4] = entry_px; tr[ntr, 5] = px; tr[ntr, 6] = abs(pos); tr[ntr, 7] = risk_usd
        tr[ntr, 8] = side * abs(pos) * (px - entry_px) - e_cost - cst; tr[ntr, 9] = e_cost + cst
        tr[ntr, 10] = eq_entry; tr[ntr, 11] = 6; tr[ntr, 12] = abs(pos) * entry_px
        ntr += 1
        eq[end - 1] = cash
    return eq, expo, traded, tr[:ntr]

def cost_frac(t, mult=1.0):
    return (CC["commission_bps"] + CC["half_spread_bps"][t] + CC["slippage_bps"]) * mult / 1e4

def run_sleeve(t, p, sig, start, end, cost_mult=1.0, smult=None, F=None):
    F = F or FEAT[t]
    le, se, lx, sx = sig
    sm_ = np.ones(NCAL) if smult is None else smult
    return simulate(F["open"], F["high"], F["low"], F["close"], F["atr"], RF, le, se, lx, sx, sm_,
                    float(p["stop_atr"]), float(p["tp_atr"]), int(p["max_hold"]), RK["risk_pct"],
                    CFG["ml"]["max_risk_pct_per_trade"], RK["max_leverage"], cost_frac(t, cost_mult),
                    CC["borrow_bps_annual"].get(t, CC["borrow_bps_annual"]["default"]) / 1e4,
                    CC["financing_spread_bps_annual"] / 1e4, RK["init_equity"], start, end)

RF_CUM = np.concatenate([[0.0], np.cumsum(RF)])

def trades_frame(t, tr) -> pd.DataFrame:
    d = pd.DataFrame(tr, columns=TRADE_COLS)
    for c_ in ["signal_idx", "entry_idx", "exit_idx", "side", "reason"]:
        d[c_] = d[c_].astype(int)
    d.insert(0, "ticker", t)
    d["signal_date"], d["entry_date"], d["exit_date"] = CAL[d["signal_idx"]], CAL[d["entry_idx"]], CAL[d["exit_idx"]]
    d["exit_reason"] = d["reason"].map(REASONS)
    d["ret_net"] = d["pnl_net"] / d["eq_entry"]
    d["R_net"] = d["pnl_net"] / d["risk_usd"]
    rf_hold = RF_CUM[d["exit_idx"].values + 1] - RF_CUM[d["entry_idx"].values + 1]
    d["ret_excess"] = d["ret_net"] - d["side"] * rf_hold * d["notional"] / d["eq_entry"]   # vs T-bill on the financed notional
    d["bars_held"] = d["exit_idx"] - d["entry_idx"] + 1
    d["gross_ret"] = (d["pnl_net"] + d["costs"]) / d["eq_entry"]
    return d

MACD_CACHE = {}
def macd_for(t, p):
    key = (t, p["fast"], p["slow"], p["signal"], p["signal_kind"], p["z_window"])
    if key not in MACD_CACHE:
        if len(MACD_CACHE) > 400:
            MACD_CACHE.clear()
        MACD_CACHE[key] = macd_features(FEAT[t]["close"], FEAT[t]["valid"], *key[1:])
    return MACD_CACHE[key]

def sleeve(t, p, start, end, cost_mult=1.0, smult=None, F=None, ev_next=None, sig_fn=None, M=None):
    """One ETF sleeve on [start, end): returns (ret, exposure, turnover, raw trade array)."""
    F = F if F is not None else FEAT[t]
    M = M if M is not None else (macd_for(t, p) if F is FEAT[t] else
                                 macd_features(F["close"], F["valid"], p["fast"], p["slow"], p["signal"], p["signal_kind"], p["z_window"]))
    sig = build_signals(F, M, p, EV_NEXT if ev_next is None else ev_next)
    if sig_fn is not None:
        sig = sig_fn(t, sig)
    eq, ex, trd, tr = run_sleeve(t, p, sig, max(start, WARM), end, cost_mult, smult, F)
    e = eq[start:end]
    ss = max(start, WARM) - start                                     # days before the simulation starts hold cash
    prev = np.concatenate([[RK["init_equity"]], e[:-1]])
    prev[: ss + 1] = RK["init_equity"]
    r = e / prev - 1
    r[:ss] = RF[start:start + ss]
    avail = np.arange(start, end) >= FIRST_IDX[t]
    return np.where(avail, r, np.nan), np.where(avail, ex[start:end], np.nan), np.where(avail, trd[start:end] / prev, np.nan), tr

def port_fast(p, start, end, cost_mult=1.0, ev_next=None, feat=None, sig_fn=None, tickers=TRADABLE, smults=None):
    """Portfolio daily EXCESS returns and number of trade entries per day (arrays; no pandas)."""
    R = np.empty((len(tickers), end - start))
    ne = np.zeros(end - start, np.int16)
    for j, t in enumerate(tickers):
        r, _, _, tr = sleeve(t, p, start, end, cost_mult, None if smults is None else smults.get(t), None if feat is None else feat[t], ev_next, sig_fn)
        R[j] = r
        if len(tr):
            np.add.at(ne, tr[:, 1].astype(np.int64) - start, 1)
    return np.nanmean(R, axis=0) - RF[start:end], ne

def run_portfolio(p, start, end, cost_mult=1.0, smults=None, tickers=TRADABLE, keep_trades=True, ev_next=None, feat=None, sig_fn=None):
    """Equal-weight daily-rebalanced portfolio of per-ETF sleeves on [start, end)."""
    rets, expos, turns, trades = [], [], [], []
    for t in tickers:
        r, ex, tu, tr = sleeve(t, p, start, end, cost_mult, None if smults is None else smults.get(t),
                               None if feat is None else feat[t], ev_next, sig_fn)
        rets.append(r); expos.append(ex); turns.append(tu)
        if keep_trades and len(tr):
            trades.append(trades_frame(t, tr))
    R = np.vstack(rets)
    port = np.nanmean(R, axis=0)
    idx = CAL[start:end]
    daily = pd.DataFrame({"ret": port, "excess": port - RF[start:end], "exposure": np.nanmean(np.vstack(expos), axis=0),
                          "turnover": np.nanmean(np.vstack(turns), axis=0)}, index=idx)
    tdf = pd.concat(trades, ignore_index=True) if trades else pd.DataFrame(columns=["ticker"] + TRADE_COLS)
    return daily, tdf, pd.DataFrame(R.T, index=idx, columns=list(tickers))

### 2.1 Causality check of the engine (no look-ahead)
All SPY prices after index k are replaced with noise. Features, signals, fills and equity up to k must be bit-identical.

In [12]:
def lookahead_test(t="SPY", k_date="2012-06-29", seed="lookahead"):
    k = int(CAL.searchsorted(pd.Timestamp(k_date)))
    g = rng(seed)
    b2 = BARS[t].copy()
    noise = np.exp(g.normal(0, 0.05, NCAL - k - 1))
    for c_ in ["open", "high", "low", "close"]:
        b2.iloc[k + 1:, b2.columns.get_loc(c_)] *= noise
    b2.iloc[k + 1:, b2.columns.get_loc("volume")] *= g.uniform(0.2, 5, NCAL - k - 1)
    F1, F2 = FEAT[t], static_features(b2, BARS["SPY"]["close"] if t != "SPY" else b2["close"])
    M1 = macd_features(F1["close"], F1["valid"], SB["fast"], SB["slow"], SB["signal"], SB["signal_kind"], SB["z_window"])
    M2 = macd_features(F2["close"], F2["valid"], SB["fast"], SB["slow"], SB["signal"], SB["signal_kind"], SB["z_window"])
    s1, s2 = build_signals(F1, M1, SB), build_signals(F2, M2, SB)
    e1 = run_sleeve(t, SB, s1, WARM, NCAL, F=F1)[0]
    e2 = run_sleeve(t, SB, s2, WARM, NCAL, F=F2)[0]
    same_feat = all(np.array_equal(F1[f][: k + 1], F2[f][: k + 1], equal_nan=True) for f in ["atr", "adx", "sma200", "rv_pct", "relvol", "obv_slope"])
    same_sig = all(np.array_equal(a[: k + 1], b[: k + 1]) for a, b in zip(s1, s2))
    same_eq = np.allclose(e1[WARM: k + 1], e2[WARM: k + 1], equal_nan=True, rtol=0, atol=1e-9)
    return dict(ticker=t, cut=CAL[k].date(), features_identical=same_feat, signals_identical=same_sig, equity_identical_to_k=same_eq)

la = pd.DataFrame([lookahead_test("SPY", "2012-06-29"), lookahead_test("TLT", "2016-03-31", "lookahead2")])
show(la, "No-look-ahead perturbation test", "n/a")
check("2", "No look-ahead: perturbing data after k leaves features/signals/equity ≤ k unchanged",
      bool(la[["features_identical", "signals_identical", "equity_identical_to_k"]].all().all()), la.to_dict("records"))

**No-look-ahead perturbation test** · sample: `n/a`

,ticker,cut,features_identical,signals_identical,equity_identical_to_k
0,SPY,2012-06-29,True,True,True
1,TLT,2016-03-31,True,True,True


### 2.2 Baseline backtest (IS 2003–2018, every signal taken)

In [13]:
def hac_lags(n):
    return int(np.floor(4 * (n / 100) ** (2 / 9)))

def nw_ttest(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 10 or np.std(x) == 0:
        return dict(mean=np.nan, t=np.nan, p=np.nan, n=len(x))
    r = sm.OLS(x, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags(len(x))})
    return dict(mean=float(r.params[0]), t=float(r.tvalues[0]), p=float(r.pvalues[0]), n=len(x))

def sharpe(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    return np.nan if len(x) < 60 or x.std(ddof=1) == 0 else x.mean() / x.std(ddof=1) * np.sqrt(252)

def max_dd(ret):
    eq = np.cumprod(1 + np.nan_to_num(ret))
    return float((eq / np.maximum.accumulate(eq) - 1).min())

@nb.njit(cache=False)
def _sb_counts(T, n, p, seed):
    np.random.seed(seed)
    C = np.zeros((n, T), np.float32)
    for b in range(n):
        i = np.random.randint(T)
        for k in range(T):
            if k > 0 and np.random.random() < p:
                i = np.random.randint(T)
            else:
                i = (i + 1) % T if k > 0 else i
            C[b, i] += 1.0
    return C

@nb.njit(cache=False)
def _sb_indices(T, n, p, seed):
    np.random.seed(seed)
    I = np.empty((n, T), np.int32)
    for b in range(n):
        i = np.random.randint(T)
        for k in range(T):
            if k > 0:
                i = np.random.randint(T) if np.random.random() < p else (i + 1) % T
            I[b, k] = i
    return I

def sb_indices(T, n, name, block=None):
    """Stationary (Politis–Romano) bootstrap indices; deterministic given the named stream."""
    return _sb_indices(T, n, 1.0 / (block or CFG["bootstrap"]["stationary_mean_block"]), seed_for(name))

def sharpe_ci(x, name, n=None, block=None):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 60:
        return (np.nan, np.nan)
    I = sb_indices(len(x), n or CFG["bootstrap"]["n_ci_boot"], name, block)
    xs = x[I]
    s = xs.mean(1) / xs.std(1, ddof=1) * np.sqrt(252)
    return tuple(np.nanpercentile(s, [2.5, 97.5]))

SPY_RET = BARS["SPY"]["close"].pct_change().values

def perf_stats(daily: pd.DataFrame, trades: pd.DataFrame | None = None, name="", ci=True) -> dict:
    r, ex = daily["ret"].values, daily["excess"].values
    yrs = len(r) / 252
    eq_end = np.prod(1 + np.nan_to_num(r))
    neg = ex[ex < 0]
    out = dict(subject=name, start=daily.index[0].date(), end=daily.index[-1].date(),
               CAGR=eq_end ** (1 / yrs) - 1, ann_excess=np.nanmean(ex) * 252, ann_vol=np.nanstd(r, ddof=1) * np.sqrt(252),
               sharpe=sharpe(ex), sortino=np.nanmean(ex) / np.sqrt(np.mean(neg ** 2)) * np.sqrt(252) if len(neg) else np.nan,
               max_dd=max_dd(r))
    out["calmar"] = out["CAGR"] / abs(out["max_dd"]) if out["max_dd"] < 0 else np.nan
    if ci:
        out["sharpe_lo"], out["sharpe_hi"] = sharpe_ci(ex, f"ci.{name}.{out['start']}")
    nw = nw_ttest(ex)
    out["nw_t"], out["nw_p"] = nw["t"], nw["p"]
    spy_ex = SPY_RET[CAL.get_indexer(daily.index)] - RF[CAL.get_indexer(daily.index)]
    ok = np.isfinite(ex) & np.isfinite(spy_ex)
    if ok.sum() > 60 and np.nanstd(ex[ok]) > 0 and name != "SPY buy&hold":
        reg = sm.OLS(ex[ok], sm.add_constant(spy_ex[ok])).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags(ok.sum())})
        out.update(alpha_ann=reg.params[0] * 252, alpha_t=reg.tvalues[0], beta=reg.params[1])
    else:
        out.update(alpha_ann=np.nan, alpha_t=np.nan, beta=1.0 if name == "SPY buy&hold" else np.nan)
    if "exposure" in daily:
        out["exposure"] = np.nanmean(daily["exposure"])
        out["turnover_ann"] = np.nansum(daily["turnover"]) / yrs
    if trades is not None and len(trades):
        out.update(n_trades=len(trades), trades_per_year=len(trades) / yrs, hit_rate=(trades["pnl_net"] > 0).mean(),
                   profit_per_trade_pct=100 * trades["ret_net"].mean(), mean_R=trades["R_net"].mean())
    else:
        out.update(n_trades=0)
    return out

def buy_and_hold(t, start, end):
    """1× notional, entry and exit costs once, dividends via the total-return basis."""
    c = BARS[t]["close"].values
    r = np.nan_to_num(np.concatenate([[0.0], c[1:] / c[:-1] - 1]))[start:end].copy()
    r[0] -= cost_frac(t)
    r[-1] -= cost_frac(t)
    return pd.DataFrame({"ret": r, "excess": r - RF[start:end], "exposure": 1.0, "turnover": 0.0}, index=CAL[start:end])

base_daily, base_trades, base_sleeves = run_portfolio(SB, 0, I_IS_END)
base_trades = base_trades[base_trades["entry_idx"] < I_IS_END]
BASE_IS = perf_stats(base_daily.iloc[WARM:], base_trades, "base 12/26/9")
SPY_IS = perf_stats(buy_and_hold("SPY", WARM, I_IS_END), None, "SPY buy&hold")
per_etf = []
for t in TRADABLE:
    d = pd.DataFrame({"ret": base_sleeves[t].values, "excess": base_sleeves[t].values - RF[:I_IS_END]}, index=CAL[:I_IS_END]).iloc[max(WARM, FIRST_IDX[t]):]
    per_etf.append(perf_stats(d.assign(exposure=np.nan, turnover=0.0), base_trades[base_trades["ticker"] == t], t, ci=False))
tbl2 = pd.DataFrame([BASE_IS, SPY_IS] + per_etf).assign(sample="IS")
show(tbl2[["subject", "sample", "start", "end", "CAGR", "ann_excess", "ann_vol", "sharpe", "sharpe_lo", "sharpe_hi", "sortino", "max_dd",
           "calmar", "nw_t", "nw_p", "alpha_ann", "beta", "n_trades", "hit_rate", "profit_per_trade_pct", "mean_R"]],
     "Baseline 12/26/9: portfolio, SPY and per-ETF sleeves", "IS", fmt="{:.4g}")
gross = base_trades["gross_ret"].mean() * 100
md(f"Pooled mean **gross** return per trade before costs: {gross:.4f}% of sleeve equity; mean cost {100 * (base_trades['costs'] / base_trades['eq_entry']).mean():.4f}%. "
   f"Share of SHY trades at the 2× leverage cap: {(base_trades.query('ticker == \"SHY\"')['notional'] >= 0.999 * RK['max_leverage'] * base_trades.query('ticker == \"SHY\"')['eq_entry']).mean():.0%}.")
G1_PASS = bool(BASE_IS["sharpe"] > 0 and BASE_IS["nw_p"] < 0.05)
check("2", "G1 IS baseline edge (Sharpe > 0 and NW p < 0.05)", G1_PASS, f"Sharpe {BASE_IS['sharpe']:.3f}, NW p {BASE_IS['nw_p']:.3f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=base_daily.index[WARM:], y=np.cumprod(1 + base_daily["ret"].values[WARM:]), name="base 12/26/9 (portfolio)"))
spy_bh = buy_and_hold("SPY", WARM, I_IS_END)
fig.add_trace(go.Scatter(x=spy_bh.index, y=np.cumprod(1 + spy_bh["ret"].values), name="SPY buy&hold"))
fig.update_layout(title="Baseline vs SPY, growth of 1 (IS)", yaxis_type="log", height=380)
fig.show()

**Baseline 12/26/9: portfolio, SPY and per-ETF sleeves** · sample: `IS`

,subject,sample,start,end,CAGR,ann_excess,ann_vol,sharpe,sharpe_lo,sharpe_hi,sortino,max_dd,calmar,nw_t,nw_p,alpha_ann,beta,n_trades,hit_rate,profit_per_trade_pct,mean_R
0,base 12/26/9,IS,2004-01-14,2018-12-31,0.008292,-0.003964,0.01145,-0.3473,-0.8204,0.1187,-0.2926,-0.02981,0.2781,-1.408,0.159,-0.004496,0.006827,1027,0.4713,-0.04661,-0.06534
1,SPY buy&hold,IS,2004-01-14,2018-12-31,0.07624,0.07791,0.1829,0.426,-0.02125,0.9041,0.4033,-0.5519,0.1381,1.931,0.05343,n/a,1,0,n/a,n/a,n/a
2,SPY,IS,2004-01-14,2018-12-31,0.009973,-0.002028,0.02595,-0.07823,n/a,n/a,-0.03636,-0.08022,0.1243,-0.3223,0.7472,-0.003132,0.01417,112,0.4911,-0.02488,-0.02488
3,QQQ,IS,2004-01-14,2018-12-31,0.01092,-0.001111,0.02526,-0.04403,n/a,n/a,-0.02043,-0.06945,0.1572,-0.1727,0.8629,-0.002454,0.01723,110,0.4909,-0.01235,-0.01235
4,DIA,IS,2004-01-14,2018-12-31,0.008609,-0.003404,0.02494,-0.1367,n/a,n/a,-0.06058,-0.08266,0.1041,-0.5369,0.5914,-0.004087,0.008759,102,0.451,-0.04399,-0.04399
5,IWM,IS,2004-01-14,2018-12-31,0.009115,-0.002941,0.02333,-0.1262,n/a,n/a,-0.05556,-0.08625,0.1057,-0.4975,0.6189,-0.003905,0.01237,104,0.4904,-0.03999,-0.03999
6,GLD,IS,2004-11-18,2018-12-31,0.007162,-0.004807,0.02511,-0.1914,n/a,n/a,-0.08338,-0.09386,0.07631,-0.6912,0.4894,-0.005348,0.006914,95,0.4737,-0.06773,-0.06773
7,SLV,IS,2006-04-28,2018-12-31,0.004796,-0.004801,0.02528,-0.1901,n/a,n/a,-0.08148,-0.1331,0.03604,-0.6445,0.5192,-0.005014,0.002646,84,0.4762,-0.0691,-0.0691
8,USO,IS,2006-04-10,2018-12-31,0.0131,0.003264,0.02436,0.134,n/a,n/a,0.05768,-0.07572,0.173,0.4661,0.6411,0.003247,0.0002141,80,0.525,0.05626,0.05626
9,CPER,IS,2011-11-15,2018-12-31,-0.00601,-0.01037,0.02879,-0.3603,n/a,n/a,-0.151,-0.07419,-0.081,-1.138,0.2549,-0.0103,-0.000579,32,0.4062,-0.2179,-0.2179


Pooled mean **gross** return per trade before costs: -0.0196% of sleeve equity; mean cost 0.0270%. Share of SHY trades at the 2× leverage cap: 96%.

### 2.3 Where does the edge thrive or decay? Bucket analysis (IS trades)
Each trade is tagged with information known at decision time:
- day-of-week and month of the entry;
- VIX regime and SPY trend (close vs SMA200) at the signal close;
- volume regime (relvol at the signal close);
- event window relative to the entry day (−1 = event tomorrow, 0 = event day, +1 = event yesterday);
- highest event impact in that window.

Per-bucket statistics use the trade's **excess** return (net of costs and of T-bill carry on the financed notional). The bucket Sharpe is mean/std × √(trades per year), descriptive only. CIs come from an entry-date cluster bootstrap. NW t/p are computed on trades in entry order. Buckets with n < 30 are flagged.

In [14]:
def tag_trades(tr: pd.DataFrame) -> pd.DataFrame:
    d = tr.copy()
    s, e = d["signal_idx"].values, d["entry_idx"].values
    d["day_of_week"] = d["entry_date"].dt.day_name().str[:3]
    d["month"] = d["entry_date"].dt.month
    d["vix_regime"] = pd.cut(VIX.values[s], RCFG["vix_bins"], labels=RCFG["vix_labels"]).astype(str)
    d["spy_trend"] = np.where(np.isnan(FEAT["SPY"]["spy_trend"][s]), "n/a", np.where(FEAT["SPY"]["spy_up"][s], "up", "down"))
    rv_ = np.array([FEAT[t]["relvol"][i] for t, i in zip(d["ticker"], s)])
    d["volume_regime"] = pd.cut(rv_, RCFG["relvol_bins"], labels=RCFG["relvol_labels"]).astype(str)
    any0, anym1, anyp1 = EVF["any_d0"].values[e], EVF["any_dm1"].values[e], EVF["any_dp1"].values[e]
    d["event_window"] = np.select([any0, anym1, anyp1], ["0", "-1", "+1"], "none")
    hi = EVF["high_win"].values[e]
    d["event_impact"] = np.select([hi, EVF["any_win"].values[e], EVF["low_win"].values[e]], ["high", "medium", "low"], "none")
    return d

def bucket_stats(g: pd.DataFrame, name: str, years: float) -> dict:
    x = g["ret_excess"].values
    n = len(x)
    out = dict(bucket=name, n_trades=n, mean_excess_pct=100 * x.mean() if n else np.nan, hit_rate=(g["pnl_net"] > 0).mean() if n else np.nan,
               sharpe_trade=x.mean() / x.std(ddof=1) * np.sqrt(n / years) if n > 2 and x.std() > 0 else np.nan,
               max_dd_pct=100 * (np.cumsum(x) - np.maximum.accumulate(np.cumsum(x))).min() if n else np.nan,
               min_sample_warning=n < RCFG["min_bucket_trades"])
    if n >= 10:
        dates = g["entry_date"].values
        ud, inv = np.unique(dates, return_inverse=True)
        sums, cnts = np.bincount(inv, weights=x), np.bincount(inv)
        G = rng(f"bucket.{name}").integers(0, len(ud), (CFG["bootstrap"]["n_ci_boot"], len(ud)))
        bm = sums[G].sum(1) / cnts[G].sum(1)
        out["ci_lo_pct"], out["ci_hi_pct"] = 100 * np.percentile(bm, 2.5), 100 * np.percentile(bm, 97.5)
        nw = nw_ttest(x)
        out["nw_t"], out["nw_p"] = nw["t"], nw["p"]
    return out

DIMS = ["day_of_week", "month", "vix_regime", "spy_trend", "volume_regime", "event_window", "event_impact"]
tagged = tag_trades(base_trades)
yrs_is = (I_IS_END - WARM) / 252
rows = []
for dim in DIMS:
    for b, g in tagged.groupby(dim):
        rows.append(dict(dim=dim, ticker="ALL", **bucket_stats(g, str(b), yrs_is)))
    for (tk, b), g in tagged.groupby(["ticker", dim]):
        rows.append(dict(dim=dim, ticker=tk, **bucket_stats(g, str(b), yrs_is)))
BUCKETS = pd.DataFrame(rows).assign(sample="IS")
fam = BUCKETS["nw_p"].notna() & ~BUCKETS["min_sample_warning"]
BUCKETS.loc[fam, "bh_q"] = multipletests(BUCKETS.loc[fam, "nw_p"], method="fdr_bh")[1]
pooled = BUCKETS[BUCKETS["ticker"] == "ALL"]
show(pooled.drop(columns=["ticker"]), "Pooled bucket statistics (all ETFs)", "IS", fmt={c: "{:.3f}" for c in pooled.columns if pooled[c].dtype == float})
md(f"**Multiple testing:** {len(BUCKETS)} bucket tests ({int(fam.sum())} with n ≥ 30 in the BH family). Nominal p < 0.05: "
   f"{int((BUCKETS.loc[fam, 'nw_p'] < 0.05).sum())} vs {0.05 * fam.sum():.1f} expected by chance. BH q < 0.05: {int((BUCKETS['bh_q'] < 0.05).sum())}. "
   "Treat individual bucket 'edges' as hypotheses, not findings.")

for dim in DIMS:
    sub = BUCKETS[(BUCKETS["dim"] == dim)]
    z = sub.pivot(index="ticker", columns="bucket", values="mean_excess_pct")
    nn = sub.pivot(index="ticker", columns="bucket", values="n_trades")
    zm = z.where(nn >= RCFG["min_bucket_trades"])
    fig = go.Figure(go.Heatmap(z=zm.values * 100, x=[str(c) for c in z.columns], y=z.index, colorscale="RdBu", zmid=0,
                               text=np.where(nn.values >= 0, nn.fillna(0).astype(int).astype(str), ""), texttemplate="n=%{text}",
                               colorbar=dict(title="bp/trade")))
    fig.update_layout(title=f"Mean excess return per trade (bp) by {dim} · IS · cells with n<30 blank", height=360)
    fig.show()

**Pooled bucket statistics (all ETFs)** · sample: `IS`

,dim,bucket,n_trades,mean_excess_pct,hit_rate,sharpe_trade,max_dd_pct,min_sample_warning,ci_lo_pct,ci_hi_pct,nw_t,nw_p,sample,bh_q
0,day_of_week,Fri,214,-0.081,0.449,-0.344,-23.548,False,-0.227,0.065,-1.563,0.118,IS,0.558
1,day_of_week,Mon,178,0.006,0.466,0.021,-13.237,False,-0.172,0.195,0.083,0.934,IS,0.981
2,day_of_week,Thu,205,-0.001,0.507,-0.003,-15.878,False,-0.159,0.153,-0.012,0.991,IS,0.993
3,day_of_week,Tue,198,0.009,0.480,0.037,-9.520,False,-0.172,0.188,0.161,0.872,IS,0.965
4,day_of_week,Wed,232,-0.140,0.457,-0.657,-33.171,False,-0.269,-0.010,-2.935,0.003,IS,0.075
60,month,1,77,-0.229,0.403,-0.611,-18.312,False,-0.443,0.002,-2.487,0.013,IS,0.134
61,month,2,102,-0.167,0.363,-0.487,-17.791,False,-0.378,0.059,-2.350,0.019,IS,0.161
62,month,3,96,0.043,0.531,0.121,-6.935,False,-0.173,0.268,0.502,0.615,IS,0.965
63,month,4,59,0.178,0.559,0.374,-9.028,False,-0.080,0.446,1.219,0.223,IS,0.799
64,month,5,75,-0.288,0.347,-0.776,-22.065,False,-0.531,-0.006,-2.903,0.004,IS,0.075


**Multiple testing:** 425 bucket tests (104 with n ≥ 30 in the BH family). Nominal p < 0.05: 15 vs 5.2 expected by chance. BH q < 0.05: 0. Treat individual bucket 'edges' as hypotheses, not findings.

### 2.4 High-impact events: does the edge change around them? (associational, not causal)
For each event type, the statistic is the mean trade excess return with the entry inside the [−1, +1] window minus the mean outside it. The null permutes that event type's dates within each year (500 draws). A difference is an association. A causal claim would need a design such as pre-registered event-day rules evaluated out of sample.

In [15]:
def event_effect(tr: pd.DataFrame, n_perm=None) -> pd.DataFrame:
    n_perm = n_perm or CFG["placebo"]["n_random_event_flags"]
    e = tr["entry_idx"].values
    x = tr["ret_excess"].values
    years = CAL.year.values
    out = []
    groups = [(typ, EVENTS["event"] == typ) for typ in ["FOMC", "CPI", "NFP", "GDP", "PCE", "RETAIL_SALES", "PPI"]] + \
             [(f"impact:{imp}", ~EVENTS["event"].str.endswith("_REVISION") & (EVENTS["impact"] == imp)) for imp in ["high", "medium"]]
    for name, mask in groups:
        ev = EVENTS[mask & EVENTS["scheduled"] & ~EVENTS["cancelled"]]
        idx = np.unique(CAL.searchsorted(ev["date"].values))
        idx = idx[idx < NCAL]
        def stat(ix):
            w = np.zeros(NCAL + 2, bool)
            for k in (-1, 0, 1):
                w[np.clip(ix + k, 0, NCAL - 1)] = True
            inw = w[e]
            return x[inw].mean() - x[~inw].mean() if inw.any() and (~inw).any() else np.nan, int(inw.sum())
        actual, n_in = stat(idx)
        g = rng(f"event_effect.{name}")
        null = []
        for _ in range(n_perm):
            perm = []
            for y in np.unique(years[idx]):
                pool = np.where(years == y)[0]
                perm.append(g.choice(pool, (years[idx] == y).sum(), replace=False))
            null.append(stat(np.concatenate(perm))[0])
        null = np.array(null)
        out.append(dict(event=name, n_trades_in_window=n_in, diff_mean_excess_pct=100 * actual,
                        null_q05_pct=100 * np.nanpercentile(null, 5), null_q95_pct=100 * np.nanpercentile(null, 95),
                        perm_p_two_sided=float((np.abs(null) >= abs(actual)).mean())))
    d = pd.DataFrame(out)
    d["bh_q"] = multipletests(d["perm_p_two_sided"], method="fdr_bh")[1]
    return d.assign(sample="IS", interpretation="associational")

EV_EFFECT = event_effect(tagged)
show(EV_EFFECT, "Event-window effect on the baseline edge", "IS", fmt={c: "{:.4f}" for c in EV_EFFECT.columns if EV_EFFECT[c].dtype == float})
print("Section 2 done ·", elapsed())

**Event-window effect on the baseline edge** · sample: `IS`

,event,n_trades_in_window,diff_mean_excess_pct,null_q05_pct,null_q95_pct,perm_p_two_sided,bh_q,sample,interpretation
0,FOMC,84,0.0335,-0.2147,0.2340,0.8020,0.9023,IS,associational
1,CPI,160,-0.0407,-0.1638,0.1860,0.6940,0.9023,IS,associational
2,NFP,145,-0.0434,-0.1867,0.1894,0.7560,0.9023,IS,associational
3,GDP,152,-0.0523,-0.1774,0.1766,0.6280,0.9023,IS,associational
4,PCE,140,-0.0670,-0.1749,0.1865,0.5720,0.9023,IS,associational
5,RETAIL_SALES,147,-0.0698,-0.1994,0.1905,0.5540,0.9023,IS,associational
6,PPI,154,0.0095,-0.1877,0.1968,0.9340,0.9340,IS,associational
7,impact:high,488,-0.0596,-0.1385,0.1165,0.4660,0.9023,IS,associational
8,impact:medium,365,-0.0989,-0.1290,0.1339,0.2560,0.9023,IS,associational


Section 2 done · 0.1 min


## 3 · Bootstrap / grid parameter search, pre-registration and overfitting controls

**Design.** `n_variations` unique configurations are drawn at random (seeded) from the parameter space below; row 0 is the A5 base. Each variation is simulated as the equal-weight 11-ETF portfolio **only up to 2018-12-31**, so the selection grid never touches the test window.

**Selection rule (fixed before the run).**
- Eligible: train trades ≥ 100, validation trades ≥ 30, and train Sharpe > 0.
- score = ½ · validation Sharpe + ½ · median validation Sharpe of the 25 nearest neighbours in normalised parameter space. This rewards stable regions, not isolated peaks.
- The top 5 by score are the candidates; rank 1 is the **primary**.

The candidates are hashed and **registered** before the sealed test is evaluated.

In [16]:
SP = CFG["search"]["space"]
NUMERIC = ["fast", "slow", "signal", "z_window", "entry_z", "exit_z", "stop_atr", "tp_atr", "max_hold"]
CATEG = ["signal_kind", "trend_filter", "vol_filter", "volume_filter", "event_filter", "side"]
PARAM_COLS = NUMERIC + ["require_turn"] + CATEG
MACD_KEY = ["fast", "slow", "signal", "signal_kind", "z_window"]
show(pd.DataFrame([(k, v) for k, v in SP.items() if isinstance(v, list)], columns=["parameter", "values"]), "Search space", "n/a")

def sample_space(n: int, name: str) -> pd.DataFrame:
    g = rng(name)
    rows, seen = [dict((k, SB[k]) for k in PARAM_COLS)], set()
    seen.add(tuple(rows[0][k] for k in PARAM_COLS))
    while len(rows) < n:
        r = {k: SP[k][g.integers(len(SP[k]))] for k in PARAM_COLS}
        if r["slow"] < r["fast"] + SP["min_fast_slow_gap"] or r["exit_z"] >= r["entry_z"]:
            continue
        key = tuple(r[k] for k in PARAM_COLS)
        if key not in seen:
            seen.add(key)
            rows.append(r)
    return pd.DataFrame(rows)

def grid_run(P: pd.DataFrame, end: int, label: str, start: int = 0):
    """Simulate every row of P on [start, end). Returns (excess T×K float32, entries T×K int16)."""
    K = len(P)
    X = np.empty((end - start, K), np.float32)
    NE = np.zeros((end - start, K), np.int16)
    order = P.sort_values(MACD_KEY, kind="stable").index.values
    t0 = time.time()
    recs = P.to_dict("records")
    for i, k in enumerate(order):
        X[:, k], NE[:, k] = port_fast(recs[k], start, end)
        if (i + 1) % 2000 == 0:
            print(f"  {label}: {i + 1}/{K} variations · {time.time() - t0:.0f}s")
    MACD_CACHE.clear()
    return X, NE

def col_sharpe(X):
    m, s = X.mean(0), X.std(0, ddof=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(s > 0, m / s * np.sqrt(252), np.nan)

def encode_params(P: pd.DataFrame) -> np.ndarray:
    cols = [P[c].map({v: i / max(1, len(SP[c]) - 1) for i, v in enumerate(SP[c])}).values for c in NUMERIC]
    cols.append(P["require_turn"].astype(float).values)
    for c in CATEG:
        for v in SP[c]:
            cols.append((P[c] == v).astype(float).values)
    return np.column_stack(cols)

def grid_metrics(P, X, NE, tr_rows, va_rows) -> pd.DataFrame:
    m = P.copy()
    m["train_sharpe"], m["val_sharpe"] = col_sharpe(X[tr_rows]), col_sharpe(X[va_rows])
    m["train_trades"], m["val_trades"] = NE[tr_rows].sum(0), NE[va_rows].sum(0)
    m["val_ann_excess"] = X[va_rows].mean(0) * 252
    return m

def select(m: pd.DataFrame, k_nn=None, min_tr=None, min_va=None, n_cand=None) -> pd.DataFrame:
    sc = CFG["search"]["selection"]
    k_nn, n_cand = k_nn or sc["neighborhood_k"], n_cand or sc["n_candidates"]
    min_tr = sc["min_trades_train"] if min_tr is None else min_tr
    min_va = sc["min_trades_validation"] if min_va is None else min_va
    ok = np.isfinite(m["val_sharpe"].values)
    E = encode_params(m)
    nn = NearestNeighbors(n_neighbors=min(k_nn + 1, ok.sum())).fit(E[ok])
    _, ind = nn.kneighbors(E[ok])
    vs = m["val_sharpe"].values[ok]
    nbr = np.full(len(m), np.nan)
    nbr[ok] = np.median(vs[ind[:, 1:]], axis=1)
    m = m.assign(nbr_median=nbr, score=0.5 * m["val_sharpe"] + 0.5 * nbr)
    m["eligible"] = (m["train_trades"] >= min_tr) & (m["val_trades"] >= min_va) & (m["train_sharpe"] > 0) & m["score"].notna()
    cand = m[m["eligible"]].sort_values("score", ascending=False).head(n_cand)
    return m, cand

P_MAIN = sample_space(CFG["search"]["n_variations"], "search")
tr_rows, va_rows = slice(WARM, I_TR_END), slice(I_TR_END, I_IS_END)
X_IS, NE_IS = grid_run(P_MAIN, I_IS_END, "selection grid (≤2018)")
GM, CAND = select(grid_metrics(P_MAIN, X_IS, NE_IS, tr_rows, va_rows))
print(f"grid: {len(P_MAIN)} variations, {int(GM['eligible'].sum())} eligible · {elapsed()}")
show(CAND[["train_sharpe", "val_sharpe", "nbr_median", "score", "train_trades", "val_trades"] + PARAM_COLS],
     "Candidates (selected on train/validation only)", "IS", fmt={c: "{:.3f}" for c in ["train_sharpe", "val_sharpe", "nbr_median", "score"]})

**Search space** · sample: `n/a`

,parameter,values
0,fast,"[5, 8, 10, 12, 15, 19]"
1,slow,"[17, 21, 26, 30, 35, 39, 50]"
2,signal,"[5, 7, 9, 12]"
3,signal_kind,"[hist_z, macd_z]"
4,z_window,"[40, 60, 120, 252]"
5,entry_z,"[1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]"
6,exit_z,"[1.0, 0.5, 0.0, -0.5]"
7,require_turn,"[True, False]"
8,stop_atr,"[1.0, 1.5, 2.0, 2.5, 3.0, 4.0]"
9,tp_atr,"[0.0, 1.0, 1.5, 2.0, 3.0, 4.0]"


  selection grid (≤2018): 2000/12000 variations · 6s


  selection grid (≤2018): 4000/12000 variations · 12s


  selection grid (≤2018): 6000/12000 variations · 18s


  selection grid (≤2018): 8000/12000 variations · 24s


  selection grid (≤2018): 10000/12000 variations · 29s


  selection grid (≤2018): 12000/12000 variations · 35s


grid: 12000 variations, 1765 eligible · 0.7 min


**Candidates (selected on train/validation only)** · sample: `IS`

,train_sharpe,val_sharpe,nbr_median,score,train_trades,val_trades,fast,slow,signal,z_window,entry_z,exit_z,stop_atr,tp_atr,max_hold,require_turn,signal_kind,trend_filter,vol_filter,volume_filter,event_filter,side
4115,0.247,1.582,0.720,1.151,291,96,10,30,7,40,1.500000,1.000000,2.500000,4.000000,20,False,macd_z,own_sma200_align,vix_lt_30,relvol_lt_1,only_event_window,long_only
2395,0.358,1.324,0.720,1.022,330,108,19,26,9,40,1.500000,1.000000,3.000000,2.000000,20,False,macd_z,own_sma200_align,none,relvol_lt_1,only_event_window,long_only
4783,0.518,1.241,0.629,0.935,159,44,5,30,5,40,2.000000,-0.500000,2.000000,3.000000,20,False,macd_z,own_sma200_align,none,relvol_lt_1,only_event_window,long_only
3206,0.008,1.404,0.421,0.913,233,79,15,50,12,120,1.750000,1.000000,1.500000,1.000000,20,True,macd_z,own_sma200_align,none,relvol_lt_1,only_event_window,both
9484,0.284,1.120,0.626,0.873,397,129,12,21,5,40,1.250000,0.500000,2.000000,4.000000,20,False,macd_z,own_sma200_align,none,relvol_lt_1,only_event_window,long_only


### 3.1 Pre-registration (before any test-period evaluation)

In [17]:
TEST_ACCESS_LOG = []
REGISTRATION = dict(registered_utc=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), config_hash=CONFIG_HASH,
                    grid_hash=cfg_hash(P_MAIN.to_dict("records")), selection_rule=CFG["search"]["selection"],
                    candidates=[dict(rank=i + 1, id=int(k), **{c: (v.item() if hasattr(v, "item") else v) for c, v in CAND.loc[k, PARAM_COLS].items()})
                                for i, k in enumerate(CAND.index)])
REGISTRATION["sha1"] = hashlib.sha1(json.dumps(REGISTRATION, sort_keys=True, default=str).encode()).hexdigest()[:12]
(CACHE / "registration.json").write_text(json.dumps(REGISTRATION, indent=1, default=str))
PRIMARY_ID = int(CAND.index[0])
PRIMARY = {c: (v.item() if hasattr(v, "item") else v) for c, v in P_MAIN.loc[PRIMARY_ID, PARAM_COLS].items()}
CAND_PARAMS = {int(k): {c: (v.item() if hasattr(v, "item") else v) for c, v in P_MAIN.loc[k, PARAM_COLS].items()} for k in CAND.index}
assert not TEST_ACCESS_LOG, "test window accessed before registration"
md(f"Registered **{len(CAND)} candidates** at {REGISTRATION['registered_utc']} · sha1 `{REGISTRATION['sha1']}` · primary id **{PRIMARY_ID}**: `{PRIMARY}`")

def test_access(what: str):
    """Every sealed-test evaluation goes through here (asserts registration first)."""
    assert REGISTRATION.get("sha1"), "register candidates before touching the test window"
    TEST_ACCESS_LOG.append(dict(utc=datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), what=what, registration=REGISTRATION["sha1"]))

Registered **5 candidates** at 2026-10-25T21:58:13Z · sha1 `7616018ed24b` · primary id **4115**: `{'fast': 10, 'slow': 30, 'signal': 7, 'z_window': 40, 'entry_z': 1.5, 'exit_z': 1.0, 'stop_atr': 2.5, 'tp_atr': 4.0, 'max_hold': 20, 'require_turn': False, 'signal_kind': 'macd_z', 'trend_filter': 'own_sma200_align', 'vol_filter': 'vix_lt_30', 'volume_filter': 'relvol_lt_1', 'event_filter': 'only_event_window', 'side': 'long_only'}`

### 3.2 Overfitting controls on the selection window (IS only)
- **Deflated Sharpe** (Bailey & López de Prado): probability that the primary's validation Sharpe beats the expected maximum Sharpe of N trials under the null. It uses the cross-sectional variance of the trial Sharpes and the primary's skew and kurtosis.
- **White Reality Check** and **Hansen SPA** (lower / consistent / upper p): H0 is that no variation beats cash on validation excess returns. Stationary bootstrap, mean block 10.
- **CSCV PBO** (S = 16) over train + validation: probability that the in-sample best is below the median out of sample.
- **Holm / BH:** one-sided NW t-tests of validation excess returns across all variations.

In [18]:
def psr(sr, n, skew, kurt, sr0=0.0):
    return stats.norm.cdf((sr - sr0) * np.sqrt(n - 1) / np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr ** 2))

def deflated_sharpe(x, n_trials, var_trials):
    """x: daily excess returns of the selected strategy; var_trials: variance of per-period trial Sharpes."""
    sr = x.mean() / x.std(ddof=1)
    g = 0.5772156649
    e_max = np.sqrt(var_trials) * ((1 - g) * stats.norm.ppf(1 - 1 / n_trials) + g * stats.norm.ppf(1 - 1 / (n_trials * np.e)))
    return psr(sr, len(x), stats.skew(x), stats.kurtosis(x, fisher=False), e_max), e_max * np.sqrt(252)

def nw_t_cols(X):
    T = X.shape[0]
    L = hac_lags(T)
    Xc = X - X.mean(0)
    v = (Xc ** 2).sum(0) / T
    for l in range(1, L + 1):
        v += 2 * (1 - l / (L + 1)) * (Xc[l:] * Xc[:-l]).sum(0) / T
    with np.errstate(invalid="ignore", divide="ignore"):
        return X.mean(0) / np.sqrt(v / T)

def wrc_spa(X, name, n_boot=None, block=None):
    X = X[:, X.std(0) > 0].astype(np.float32)
    T = X.shape[0]
    n_boot = n_boot or CFG["bootstrap"]["n_wrc_spa"]
    C = _sb_counts(T, n_boot, 1.0 / (block or CFG["bootstrap"]["stationary_mean_block"]), seed_for(name))
    m = X.mean(0)
    Mb = (C @ X) / T
    p_wrc = float(((np.sqrt(T) * (Mb - m).max(1)) >= np.sqrt(T) * m.max()).mean())
    omega = np.sqrt(T) * Mb.std(0)
    omega = np.where(omega > 0, omega, np.nan)
    t_obs = max(np.nanmax(np.sqrt(T) * m / omega), 0.0)
    A = omega / np.sqrt(T) * np.sqrt(2 * np.log(np.log(T)))
    out = dict(K=X.shape[1], T=T, wrc_p=p_wrc)
    for lab, mu in [("spa_lower_p", np.minimum(m, 0)), ("spa_consistent_p", np.where(m <= -A, m, 0.0)), ("spa_upper_p", np.zeros_like(m))]:
        Z = np.sqrt(T) * (Mb - m + mu) / omega
        out[lab] = float((np.maximum(np.nanmax(Z, axis=1), 0) >= t_obs).mean())
    return out

def cscv_pbo(X, S=None):
    S = S or CFG["bootstrap"]["cscv_partitions"]
    X = X[:, X.std(0) > 0].astype(np.float64)
    blocks = np.array_split(np.arange(X.shape[0]), S)
    s1 = np.stack([X[b].sum(0) for b in blocks]); s2 = np.stack([(X[b] ** 2).sum(0) for b in blocks])
    nb_ = np.array([len(b) for b in blocks], float)
    combos = np.array(list(itertools.combinations(range(S), S // 2)))
    W = np.zeros((len(combos), S)); W[np.arange(len(combos))[:, None], combos] = 1
    logits = []
    for ch in np.array_split(np.arange(len(combos)), max(1, len(combos) // 500)):
        def shp(Wc):
            n = Wc @ nb_; a = Wc @ s1; b = Wc @ s2
            mu = a / n[:, None]
            return mu / np.sqrt(np.maximum(b / n[:, None] - mu ** 2, 1e-18))
        si, so = shp(W[ch]), shp(1 - W[ch])
        best = si.argmax(1)
        so_best = so[np.arange(len(ch)), best]
        w = ((so < so_best[:, None]).sum(1) + 1) / (X.shape[1] + 1)
        logits.append(np.log(w / (1 - w)))
    lg = np.concatenate(logits)
    return dict(pbo=float((lg <= 0).mean()), n_combos=len(lg), K=X.shape[1]), lg

xv = X_IS[va_rows]
prim_val = xv[:, PRIMARY_ID].astype(float)
sr_trials = GM["val_sharpe"].values / np.sqrt(252)
var_tr = np.nanvar(sr_trials[np.isfinite(sr_trials)])
sub = rng("neff").choice(np.where(np.isfinite(GM["val_sharpe"]))[0], min(2000, int(np.isfinite(GM["val_sharpe"]).sum())), replace=False)
ev_ = np.clip(np.linalg.eigvalsh(np.corrcoef(xv[:, sub].T.astype(float))), 0, None)
N_EFF = float(min(len(P_MAIN), (ev_.sum() ** 2 / (ev_ ** 2).sum()) * len(P_MAIN) / len(sub)))
dsr_rows = []
for lab, n_tr in [("N = all trials", len(P_MAIN)), ("N = effective (eigen participation, scaled)", N_EFF)]:
    d_, sr0 = deflated_sharpe(prim_val, n_tr, var_tr)
    dsr_rows.append(dict(trials=lab, N=n_tr, primary_val_sharpe=sharpe(prim_val), expected_max_sharpe_null=sr0, DSR=d_,
                         PSR_vs_0=psr(prim_val.mean() / prim_val.std(ddof=1), len(prim_val), stats.skew(prim_val), stats.kurtosis(prim_val, fisher=False))))
DSR = pd.DataFrame(dsr_rows).assign(sample="IS (validation)")
show(DSR, "Deflated Sharpe of the primary", "IS", fmt="{:.4g}")

wrc_runs = pd.DataFrame([dict(seed=s, **wrc_spa(xv, f"wrc.seed{s}")) for s in CFG["bootstrap"]["seed_stability_seeds"]]).assign(sample="IS (validation)")
show(wrc_runs, "White Reality Check / Hansen SPA across bootstrap seeds (H0: no variation beats cash)", "IS", fmt="{:.3f}")

elig = GM["eligible"].values
PBO_ALL, lg_all = cscv_pbo(X_IS[WARM:I_IS_END])
PBO_EL, lg_el = cscv_pbo(X_IS[WARM:I_IS_END][:, elig])
tv = nw_t_cols(xv.astype(float))
pv = 1 - stats.norm.cdf(tv)
okp = np.isfinite(pv)
holm = multipletests(pv[okp], method="holm")[0].sum()
bh = multipletests(pv[okp], method="fdr_bh")[0].sum()
OVF = pd.DataFrame([dict(test="CSCV PBO (eligible universe; gate)", value=PBO_EL["pbo"], detail=f"K={PBO_EL['K']}, {PBO_EL['n_combos']} splits"),
                    dict(test="CSCV PBO (all variations)", value=PBO_ALL["pbo"], detail=f"K={PBO_ALL['K']}"),
                    dict(test="Holm discoveries (5%, one-sided NW)", value=holm, detail=f"of {okp.sum()} tests; raw p<0.05: {(pv[okp] < 0.05).sum()} vs {0.05 * okp.sum():.0f} expected"),
                    dict(test="BH discoveries (5%)", value=bh, detail=""),
                    dict(test="primary raw one-sided NW p", value=pv[PRIMARY_ID], detail="selected after the fact → not a valid test on its own"),
                    dict(test="effective number of trials", value=N_EFF, detail="participation ratio of validation-return correlation eigenvalues (2,000-column subsample, scaled)")]).assign(sample="IS")
show(OVF, "PBO and multiple-testing control", "IS")
G2_PASS = bool(DSR["DSR"].iloc[0] >= 0.95 and wrc_runs["spa_consistent_p"].median() < 0.05 and PBO_EL["pbo"] < 0.5 and holm >= 1)
check("3", "G2 search significance (DSR ≥ 0.95, SPA_c p < 0.05, PBO < 0.5, ≥1 Holm discovery)", G2_PASS,
      f"DSR {DSR['DSR'].iloc[0]:.3f}; SPA_c p {wrc_runs['spa_consistent_p'].median():.3f}; WRC p {wrc_runs['wrc_p'].median():.3f}; PBO {PBO_EL['pbo']:.3f}; Holm {holm}")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Validation Sharpe of all variations (primary = red)", "CSCV logits, eligible universe (≤0 ⇒ overfit)"])
fig.add_trace(go.Histogram(x=GM["val_sharpe"], nbinsx=120, name="validation Sharpe"), 1, 1)
fig.add_vline(x=GM.loc[PRIMARY_ID, "val_sharpe"], line_color="red", row=1, col=1)
fig.add_vline(x=DSR["expected_max_sharpe_null"].iloc[0], line_dash="dot", line_color="black", row=1, col=1)
fig.add_trace(go.Histogram(x=lg_el, nbinsx=60, name="logit"), 1, 2)
fig.update_layout(height=360, showlegend=False, title="Selection-window diagnostics (dotted = expected max Sharpe of N null trials)")
fig.show()

**Deflated Sharpe of the primary** · sample: `IS`

,trials,N,primary_val_sharpe,expected_max_sharpe_null,DSR,PSR_vs_0,sample
0,N = all trials,1.2e+04,1.582,2.029,0.1705,0.9996,IS (validation)
1,"N = effective (eigen participation, scaled)",387.7,1.582,1.546,0.5302,0.9996,IS (validation)


**White Reality Check / Hansen SPA across bootstrap seeds (H0: no variation beats cash)** · sample: `IS`

,seed,K,T,wrc_p,spa_lower_p,spa_consistent_p,spa_upper_p,sample
0,1.000,11227.000,1006.000,0.796,0.669,0.701,0.704,IS (validation)
1,2.000,11227.000,1006.000,0.807,0.610,0.643,0.644,IS (validation)
2,3.000,11227.000,1006.000,0.817,0.720,0.777,0.777,IS (validation)
3,4.000,11227.000,1006.000,0.778,0.597,0.631,0.634,IS (validation)
4,5.000,11227.000,1006.000,0.793,0.600,0.644,0.646,IS (validation)


**PBO and multiple-testing control** · sample: `IS`

,test,value,detail,sample
0,CSCV PBO (eligible universe; gate),0.727972,"K=1765, 12870 splits",IS
1,CSCV PBO (all variations),0.392774,K=11365,IS
2,"Holm discoveries (5%, one-sided NW)",0.000000,of 11227 tests; raw p<0.05: 308 vs 561 expected,IS
3,BH discoveries (5%),0.000000,,IS
4,primary raw one-sided NW p,0.000505,selected after the fact → not a valid test on its own,IS
5,effective number of trials,387.673439,"participation ratio of validation-return correlation eigenvalues (2,000-colu...",IS


### 3.3 Parameter stability, seed stability and cost sensitivity (IS)

In [19]:
def run_window(p, start, end, cost_mult=1.0, **kw):
    x, ne = port_fast(p, start, end, cost_mult, **kw)
    return sharpe(x), int(ne.sum())

oat = []
for par in NUMERIC + ["require_turn"] + CATEG:
    for v in SP[par]:
        q = dict(PRIMARY, **{par: v})
        if q["slow"] < q["fast"] + SP["min_fast_slow_gap"] or q["exit_z"] >= q["entry_z"]:
            continue
        s_, n_ = run_window(q, WARM, I_IS_END)
        oat.append(dict(param=par, value=str(v), is_primary=(v == PRIMARY[par]), is_sharpe=s_, is_trades=n_))
OAT = pd.DataFrame(oat)
fig = px.line(OAT, x="value", y="is_sharpe", facet_col="param", facet_col_wrap=4, markers=True, height=900,
              title="One-at-a-time sensitivity around the primary (IS Sharpe 2004–2018)")
fig.update_xaxes(matches=None, type="category")
fig.show()

seed_rows = []
for s in CFG["bootstrap"]["seed_stability_seeds"][:3]:
    Ps = sample_space(CFG["search"]["n_variations"], f"search.alt{s}")
    Xs, Ns = grid_run(Ps, I_IS_END, f"alt universe {s}")
    gs, cs = select(grid_metrics(Ps, Xs, Ns, tr_rows, va_rows))
    top = cs.iloc[0]
    seed_rows.append(dict(universe=f"alt seed {s}", val_sharpe=top["val_sharpe"], score=top["score"],
                          shared_params=sum(top[c] == PRIMARY[c] for c in PARAM_COLS), **{c: top[c] for c in PARAM_COLS}))
    del Xs, Ns
seed_rows.insert(0, dict(universe="main", val_sharpe=GM.loc[PRIMARY_ID, "val_sharpe"], score=GM.loc[PRIMARY_ID, "score"], shared_params=len(PARAM_COLS), **PRIMARY))
SEEDSTAB = pd.DataFrame(seed_rows)
show(SEEDSTAB, f"Seed stability: re-sample the universe and re-select (shared_params out of {len(PARAM_COLS)})", "IS", fmt={"val_sharpe": "{:.3f}", "score": "{:.3f}"})

cost_is = pd.DataFrame([dict(candidate=f"#{i + 1} ({k})", cost_mult=cm, is_sharpe=run_window(CAND_PARAMS[k], WARM, I_IS_END, cm)[0])
                        for i, k in enumerate(CAND_PARAMS) for cm in CC["cost_multipliers"]])
fig = px.line(cost_is, x="cost_mult", y="is_sharpe", color="candidate", markers=True, title="Cost sensitivity of the candidates (IS)", height=340)
fig.show()

  alt universe 1: 2000/12000 variations · 6s


  alt universe 1: 4000/12000 variations · 11s


  alt universe 1: 6000/12000 variations · 17s


  alt universe 1: 8000/12000 variations · 23s


  alt universe 1: 10000/12000 variations · 29s


  alt universe 1: 12000/12000 variations · 35s


... (10 outputs left out)

  alt universe 3: 10000/12000 variations · 29s


  alt universe 3: 12000/12000 variations · 34s


**Seed stability: re-sample the universe and re-select (shared_params out of 16)** · sample: `IS`

,universe,val_sharpe,score,shared_params,fast,slow,signal,z_window,entry_z,exit_z,stop_atr,tp_atr,max_hold,require_turn,signal_kind,trend_filter,vol_filter,volume_filter,event_filter,side
0,main,1.582,1.151,16,10,30,7,40,1.500000,1.000000,2.500000,4.000000,20,False,macd_z,own_sma200_align,vix_lt_30,relvol_lt_1,only_event_window,long_only
1,alt seed 1,1.480,0.832,4,8,35,12,40,1.750000,-0.500000,3.000000,2.000000,5,True,macd_z,none,vix_lt_20,none,only_event_window,long_only
2,alt seed 2,1.783,1.135,5,12,26,5,40,1.500000,-0.500000,1.000000,3.000000,30,False,macd_z,spy_sma200_align,rv_pct_lt_80,obv_divergence,skip_any_event_window,long_only
3,alt seed 3,1.644,1.123,4,8,39,9,60,1.750000,0.500000,4.000000,3.000000,5,True,macd_z,own_sma200_align,none,none,only_event_window,long_only


### 3.4 Placebo and sanity checks
Each null is compared with the primary's portfolio Sharpe over the window, and p = share of null draws ≥ actual.
- **Random signals:** the same number of long/short entries per ETF, on random valid days, with the same stop, target and max hold but no signal exit.
- **Shuffled returns:** days are permuted jointly across ETFs (cross-correlation kept). Price-derived features are rebuilt; VIX, volume and events stay in place.
- **Lagged signals:** entries and exits are delayed by L days.
- **Randomized parameters:** random configurations from the same space.
- **Random event flags:** event dates are permuted within each year.

On the IS window these p-values are **biased toward significance**, because the primary was selected on that window.

In [20]:
def perm_events_within_year(g):
    ev = EVENTS[EVENTS["scheduled"] & ~EVENTS["cancelled"]].copy()
    idx = np.clip(CAL.searchsorted(ev["date"].values), 0, NCAL - 1)
    yrs = CAL.year.values
    new = idx.copy()
    for y in np.unique(yrs[idx]):
        pool = np.where(yrs == y)[0]
        days = np.unique(idx[yrs[idx] == y])
        mp = dict(zip(days, g.choice(pool, len(days), replace=False)))
        sel = yrs[idx] == y
        new[sel] = [mp[i] for i in idx[sel]]
    ev["date"] = CAL[new]
    return ev

def shuffled_features(g, start, end):
    perm = np.arange(NCAL)
    perm[start:end] = start + g.permutation(end - start)
    feat = {}
    newbars = {}
    for t in TRADABLE:
        b = BARS[t]
        c = b["close"].values
        c1 = np.concatenate([[np.nan], c[:-1]])
        gap, oc, hc, lc = b["open"].values / c1, c / b["open"].values, b["high"].values / c, b["low"].values / c
        gap, oc, hc, lc = gap[perm], oc[perm], hc[perm], lc[perm]
        nc = c.copy()
        for i in range(start, end):
            prev = nc[i - 1] if i > 0 else np.nan
            nc[i] = prev * gap[i] * oc[i] if np.isfinite(prev) and np.isfinite(gap[i] * oc[i]) else (c[i] if not np.isfinite(prev) else prev)
        nb_ = b.copy()
        nb_["close"] = nc
        nb_["open"] = np.where(np.arange(NCAL) >= start, nc / oc, b["open"].values)
        nb_["high"] = np.where(np.arange(NCAL) >= start, np.maximum(nc * hc, np.maximum(nb_["open"], nc)), b["high"].values)
        nb_["low"] = np.where(np.arange(NCAL) >= start, np.minimum(nc * lc, np.minimum(nb_["open"], nc)), b["low"].values)
        nb_.loc[b["close"].isna(), ["open", "high", "low", "close"]] = np.nan
        newbars[t] = nb_
    for t in TRADABLE:
        feat[t] = static_features(newbars[t], newbars["SPY"]["close"])
    return feat

def placebo_suite(p, start, end, tag):
    actual, _ = run_window(p, start, end)
    daily, trades, _ = run_portfolio(p, start, end)
    res = []
    g = rng(f"placebo.random_signals.{tag}")
    holds = {}
    def make_rand(t, sig):
        le, se, lx, sx = sig
        tt = trades[trades["ticker"] == t]
        nl, ns = int((tt["side"] == 1).sum()), int((tt["side"] == -1).sum())
        okd = np.where(FEAT[t]["valid"] & np.isfinite(FEAT[t]["atr"]) & (np.arange(NCAL) >= start) & (np.arange(NCAL) < end - 1))[0]
        le2, se2 = np.zeros(NCAL, bool), np.zeros(NCAL, bool)
        if len(okd):
            pick = g.choice(okd, min(nl + ns, len(okd)), replace=False)
            le2[pick[:nl]] = True
            se2[pick[nl:]] = True
        z = np.zeros(NCAL, bool)
        return le2, se2, z, z
    null = [sharpe(port_fast(p, start, end, sig_fn=make_rand)[0]) for _ in range(CFG["placebo"]["n_random_signal"])]
    res.append(("random signals (matched count/side)", null))
    g2 = rng(f"placebo.shuffled.{tag}")
    null = []
    for _ in range(CFG["placebo"]["n_shuffled_returns"]):
        null.append(sharpe(port_fast(p, start, end, feat=shuffled_features(g2, start, end))[0]))
    res.append(("shuffled returns (joint across ETFs)", null))
    res.append(("lagged signals (lags 1,2,5,10)", [sharpe(port_fast(p, start, end, sig_fn=lambda t, s, L=L: tuple(np.roll(a, L) & (np.arange(NCAL) >= L) for a in s))[0])
                                                    for L in CFG["placebo"]["signal_lags"]]))
    Pr = sample_space(CFG["placebo"]["n_random_params"] + 1, f"placebo.params.{tag}").iloc[1:]
    res.append(("randomized parameters", [run_window(r, start, end)[0] for r in Pr.to_dict("records")]))
    g3 = rng(f"placebo.events.{tag}")
    null = []
    for _ in range(CFG["placebo"]["n_random_event_flags"]):
        evf = event_flags(perm_events_within_year(g3))
        evn = {k: np.append(evf[f"{k}_win"].values[1:], False) for k in ["any", "high"]}
        null.append(sharpe(port_fast(p, start, end, ev_next=evn)[0]))
    res.append(("random event flags (within-year permutation)", null))
    rows = []
    for name, nl in res:
        nl = np.array(nl, float)
        rows.append(dict(placebo=name, actual_sharpe=actual, n_null=len(nl), null_mean=np.nanmean(nl), null_q95=np.nanpercentile(nl, 95),
                         p_value=float((nl >= actual).mean()) if len(nl) > 4 else np.nan))
    MACD_CACHE.clear()
    return pd.DataFrame(rows), res

PLACEBO_IS, PL_IS_RAW = placebo_suite(PRIMARY, WARM, I_IS_END, "is")
show(PLACEBO_IS.assign(sample="IS (selection window; biased toward significance)"), "Placebo suite on the selection window", "IS", fmt="{:.3f}")
print("placebos IS ·", elapsed())

**Placebo suite on the selection window** · sample: `IS`

,placebo,actual_sharpe,n_null,null_mean,null_q95,p_value,sample
0,random signals (matched count/side),0.554,500.000,0.292,0.634,0.094,IS (selection window; biased toward significance)
1,shuffled returns (joint across ETFs),0.554,200.000,0.059,0.482,0.015,IS (selection window; biased toward significance)
2,"lagged signals (lags 1,2,5,10)",0.554,4.000,0.627,0.709,n/a,IS (selection window; biased toward significance)
3,randomized parameters,0.554,2000.000,-0.072,0.393,0.009,IS (selection window; biased toward significance)
4,random event flags (within-year permutation),0.554,500.000,0.561,0.702,0.556,IS (selection window; biased toward significance)


placebos IS · 5.2 min


### 3.5 Sealed test: IS → OOS degradation (**OOS**)
From here on the test window is read. Every access is logged through `test_access()`.

In [21]:
test_access("candidates + base + SPY on the test window")
deg = []
SUBJECTS = {f"#{i + 1} id {k}": CAND_PARAMS[k] for i, k in enumerate(CAND_PARAMS)} | {"base 12/26/9": SB}
OOS_RUNS, IS_RUNS = {}, {}
for name, p in SUBJECTS.items():
    IS_RUNS[name] = run_portfolio(p, WARM, I_IS_END)
    OOS_RUNS[name] = run_portfolio(p, I_TEST, NCAL)
    for samp, (d, tr_, _) in [("IS", IS_RUNS[name]), ("OOS", OOS_RUNS[name])]:
        deg.append({**perf_stats(d, tr_, f"{name}.{samp}"), "subject": name, "sample": samp})
SPY_OOS_D = buy_and_hold("SPY", I_TEST, NCAL)
for samp, d in [("IS", buy_and_hold("SPY", WARM, I_IS_END)), ("OOS", SPY_OOS_D)]:
    deg.append({**perf_stats(d, None, "SPY buy&hold"), "sample": samp})
DEG = pd.DataFrame(deg)
show(DEG[["subject", "sample", "start", "end", "sharpe", "sharpe_lo", "sharpe_hi", "CAGR", "ann_excess", "max_dd", "nw_p", "hit_rate",
          "profit_per_trade_pct", "mean_R", "trades_per_year", "exposure", "alpha_ann", "beta"]],
     "IS vs OOS for the registered candidates, base and SPY", "IS / OOS", fmt="{:.4g}")
pr_name = f"#1 id {PRIMARY_ID}"
P_OOS = DEG[(DEG["subject"] == pr_name) & (DEG["sample"] == "OOS")].iloc[0]
G3_PASS = bool(P_OOS["sharpe"] > 0 and P_OOS["sharpe_lo"] > 0)
check("3", "G3 sealed OOS: registration before access; primary OOS Sharpe > 0 with CI lower bound > 0", G3_PASS,
      f"registered {REGISTRATION['registered_utc']}, first access {TEST_ACCESS_LOG[0]['utc']} UTC; OOS Sharpe {P_OOS['sharpe']:.3f} [{P_OOS['sharpe_lo']:.2f}, {P_OOS['sharpe_hi']:.2f}]")

test_access("cost sensitivity + WRC/SPA + placebos on the test window")
cost_oos = pd.DataFrame([dict(cost_mult=cm, oos_sharpe=run_window(PRIMARY, I_TEST, NCAL, cm)[0], is_sharpe=run_window(PRIMARY, WARM, I_IS_END, cm)[0])
                         for cm in CC["cost_multipliers"]])
show(cost_oos, "Primary: Sharpe by cost multiplier", "IS / OOS", fmt="{:.3f}")
Xo = np.column_stack([OOS_RUNS[n][0]["excess"].values for n in SUBJECTS])
show(pd.DataFrame([wrc_spa(Xo, "wrc.oos")]).assign(sample="OOS"), "WRC / SPA on the test window across the 5 candidates + base", "OOS", fmt="{:.3f}")
PLACEBO_OOS, PL_OOS_RAW = placebo_suite(PRIMARY, I_TEST, NCAL, "oos")
show(PLACEBO_OOS.assign(sample="OOS"), "Placebo suite on the test window (primary)", "OOS", fmt="{:.3f}")

fig = go.Figure()
for name in SUBJECTS:
    d = OOS_RUNS[name][0]
    fig.add_trace(go.Scatter(x=d.index, y=np.cumprod(1 + d["ret"].values), name=name))
fig.add_trace(go.Scatter(x=SPY_OOS_D.index, y=np.cumprod(1 + SPY_OOS_D["ret"].values), name="SPY buy&hold"))
fig.update_layout(title="Sealed test window: growth of 1 (OOS)", yaxis_type="log", height=400)
fig.show()
print("Section 3 done ·", elapsed())

**IS vs OOS for the registered candidates, base and SPY** · sample: `IS / OOS`

,subject,sample,start,end,sharpe,sharpe_lo,sharpe_hi,CAGR,ann_excess,max_dd,nw_p,hit_rate,profit_per_trade_pct,mean_R,trades_per_year,exposure,alpha_ann,beta
0,#1 id 4115,IS,2004-01-14,2018-12-31,0.5542,0.04355,1.104,0.01592,0.003532,-0.01398,0.04023,0.6718,0.1882,0.1898,25.89,0.05474,0.002972,0.007191
1,#1 id 4115,OOS,2019-01-02,2026-09-21,0.2429,-0.4568,0.9251,0.02854,0.001258,-0.005174,0.5009,0.6205,0.1072,0.1111,25.33,0.03753,0.0004905,0.005005
2,#2 id 2395,IS,2004-01-14,2018-12-31,0.5702,0.0394,1.091,0.0155,0.003112,-0.008819,0.03059,0.7215,0.1459,0.1431,29.3,0.05341,0.002612,0.006418
3,#2 id 2395,OOS,2019-01-02,2026-09-21,0.2365,-0.3814,0.9406,0.02838,0.001094,-0.004713,0.5146,0.6455,0.08741,0.09051,28.58,0.03824,0.0002683,0.005388
4,#3 id 4783,IS,2004-01-14,2018-12-31,0.6586,0.1443,1.206,0.01595,0.003549,-0.01498,0.01518,0.5764,0.3143,0.3214,13.58,0.03825,0.003255,0.003772
5,#3 id 4783,OOS,2019-01-02,2026-09-21,-0.004736,-0.661,0.6481,0.02723,-2.101e-05,-0.003827,0.989,0.5444,0.09027,0.1022,11.69,0.03018,-0.0005958,0.003749
6,#4 id 3206,IS,2004-01-14,2018-12-31,0.2728,-0.2392,0.8098,0.01374,0.001372,-0.009917,0.3173,0.6442,0.08825,0.05303,20.87,0.01864,0.001148,0.00288
7,#4 id 3206,OOS,2019-01-02,2026-09-21,0.1876,-0.5488,0.8926,0.0281,0.0008299,-0.004289,0.6267,0.6642,0.06549,0.07684,17.8,0.01499,0.0005848,0.001599
8,#5 id 9484,IS,2004-01-14,2018-12-31,0.4785,-0.01262,0.9992,0.01668,0.004297,-0.01606,0.07849,0.5798,0.1794,0.1756,35.19,0.0907,0.003621,0.008685
9,#5 id 9484,OOS,2019-01-02,2026-09-21,0.1081,-0.5655,0.7792,0.02813,0.000879,-0.01474,0.761,0.5632,0.09006,0.1006,35.98,0.06776,-0.0005257,0.009163


**Primary: Sharpe by cost multiplier** · sample: `IS / OOS`

,cost_mult,oos_sharpe,is_sharpe
0,0.000,0.340,0.658
1,0.500,0.291,0.606
2,1.000,0.243,0.554
3,2.000,0.146,0.451
4,3.000,0.050,0.348
5,5.000,-0.141,0.143


**WRC / SPA on the test window across the 5 candidates + base** · sample: `OOS`

,K,T,wrc_p,spa_lower_p,spa_consistent_p,spa_upper_p,sample
0,6.000,1940.000,0.661,0.551,0.610,0.610,OOS


**Placebo suite on the test window (primary)** · sample: `OOS`

,placebo,actual_sharpe,n_null,null_mean,null_q95,p_value,sample
0,random signals (matched count/side),0.243,500.000,0.355,0.852,0.626,OOS
1,shuffled returns (joint across ETFs),0.243,200.000,0.120,0.708,0.335,OOS
2,"lagged signals (lags 1,2,5,10)",0.243,4.000,0.253,0.396,n/a,OOS
3,randomized parameters,0.243,2000.000,-0.074,0.531,0.193,OOS
4,random event flags (within-year permutation),0.243,500.000,0.210,0.443,0.374,OOS


Section 3 done · 7.2 min


## 4 · Multi-split stress testing

For splits 1–5 (IS = the first *k* years after the 260-day warm-up, OOS = the remainder to 2026-09-21) and split 6 (first half vs second half), two things are reported:
- **(a) Re-selection:** the selection rule is re-run on that split's IS window only (train = first 80%, validation = last 20%, minimum trades scaled to the window). The re-selected #1 is then evaluated on that split's OOS remainder with fresh sleeves. This is a true out-of-sample test **of the procedure**. Most of these OOS windows still lie inside the study's 2003–2018 IS years, which shaped the search space, hence the label `OOS-split`.
- **(b) Fixed registered primary** on the same OOS window, flagged `IS-selection-overlap` where the window overlaps the 2003–2018 selection data.

These runs need the full-history grid, so they come **after** registration. They are logged as test access.

In [22]:
del X_IS, NE_IS
test_access("full-history grid (multi-split, IS→OOS rank diagnostic)")
X_ALL, NE_ALL = grid_run(P_MAIN, NCAL, "full-history grid")
test_sh = col_sharpe(X_ALL[I_TEST:])
fin = np.isfinite(GM["val_sharpe"].values) & np.isfinite(test_sh)
RANK_DIAG = dict(spearman_val_vs_test_all=stats.spearmanr(GM["val_sharpe"].values[fin], test_sh[fin])[0],
                 spearman_score_vs_test_eligible=stats.spearmanr(GM.loc[elig & fin, "score"], test_sh[elig & fin])[0],
                 primary_test_percentile=float((test_sh[fin] < test_sh[PRIMARY_ID]).mean()))
md("IS→OOS carry-over (post-registration diagnostic, not used for selection): " + ", ".join(f"{k} = {v:.3f}" for k, v in RANK_DIAG.items()))

s0 = WARM
splits = [(f"IS {k}y", int(CAL.searchsorted(CAL[s0] + pd.DateOffset(years=k)))) for k in CFG["splits"]["multi_split_is_years"]]
if CFG["splits"]["multi_split_half"]:
    splits.append(("first half", (s0 + NCAL) // 2))
sc = CFG["search"]["selection"]
ms_rows, ms_curves = [], {}
for name, e in splits:
    L = e - s0
    cut = s0 + int(0.8 * L)
    min_tr = max(5, int(round(sc["min_trades_train"] * (cut - s0) / (I_TR_END - WARM))))
    min_va = max(5, int(round(sc["min_trades_validation"] * (e - cut) / (I_IS_END - I_TR_END))))
    gm_k, cand_k = select(grid_metrics(P_MAIN, X_ALL, NE_ALL, slice(s0, cut), slice(cut, e)), min_tr=min_tr, min_va=min_va)
    win = f"{CAL[e].date()}..{CAL[-1].date()}"
    spy_d = buy_and_hold("SPY", e, NCAL)
    spy_s = perf_stats(spy_d, None, "SPY buy&hold")
    if len(cand_k):
        k1 = int(cand_k.index[0])
        p1 = {c: (v.item() if hasattr(v, "item") else v) for c, v in P_MAIN.loc[k1, PARAM_COLS].items()}
        d1, t1, _ = run_portfolio(p1, e, NCAL)
        st = perf_stats(d1, t1, f"split.{name}")
        pct = float((col_sharpe(X_ALL[e:])[np.isfinite(col_sharpe(X_ALL[e:]))] < st["sharpe"]).mean())
        ms_rows.append(dict(split=name, method="(a) re-selected #1", id=k1, sample="OOS-split", oos_window=win, is_val_sharpe=cand_k.iloc[0]["val_sharpe"],
                            grid_oos_percentile=pct, spy_oos_sharpe=spy_s["sharpe"], beats_spy=st["sharpe"] > spy_s["sharpe"], **st))
        ms_curves[name] = (d1, spy_d)
    dp, tp_, _ = run_portfolio(PRIMARY, e, NCAL)
    stp = perf_stats(dp, tp_, f"split.{name}.fixed")
    ms_rows.append(dict(split=name, method="(b) fixed registered primary", id=PRIMARY_ID, oos_window=win,
                        sample="IS-selection-overlap" if e < I_IS_END else "OOS", spy_oos_sharpe=spy_s["sharpe"], beats_spy=stp["sharpe"] > spy_s["sharpe"], **stp))
MS = pd.DataFrame(ms_rows)
show(MS[["split", "method", "id", "sample", "oos_window", "is_val_sharpe", "sharpe", "sharpe_lo", "sharpe_hi", "nw_p", "CAGR", "ann_vol", "sortino", "max_dd",
         "hit_rate", "profit_per_trade_pct", "exposure", "turnover_ann", "alpha_ann", "alpha_t", "beta", "spy_oos_sharpe", "beats_spy", "grid_oos_percentile"]],
     "Multi-split results", "OOS-split / IS-selection-overlap", fmt="{:.4g}")
re_ = MS[MS["method"] == "(a) re-selected #1"]
n_sig = int(((re_["sharpe"] > 0) & (re_["nw_p"] < 0.05)).sum())
n_spy = int(re_["beats_spy"].sum())
G4_PASS = n_sig >= 4 and n_spy >= 4
check("4", "G4 multi-split: re-selected #1 positive & HAC-significant OOS in ≥4/6 and beats SPY in ≥4/6", G4_PASS,
      f"positive {int((re_['sharpe'] > 0).sum())}/6, significant {n_sig}/6, beats SPY {n_spy}/6")

fig = make_subplots(rows=2, cols=3, subplot_titles=list(ms_curves))
for i, (name, (d1, spy_d)) in enumerate(ms_curves.items()):
    r_, c_ = divmod(i, 3)
    fig.add_trace(go.Scatter(x=d1.index, y=np.cumprod(1 + d1["ret"].values), name="re-selected #1", line=dict(color="#1f77b4"), showlegend=i == 0), r_ + 1, c_ + 1)
    fig.add_trace(go.Scatter(x=spy_d.index, y=np.cumprod(1 + spy_d["ret"].values), name="SPY", line=dict(color="#ff7f0e"), showlegend=i == 0), r_ + 1, c_ + 1)
fig.update_yaxes(type="log")
fig.update_layout(height=560, title="Multi-split OOS remainders: re-selected #1 vs SPY buy&hold (growth of 1)")
fig.show()

  full-history grid: 2000/12000 variations · 5s


  full-history grid: 4000/12000 variations · 9s


  full-history grid: 6000/12000 variations · 14s


  full-history grid: 8000/12000 variations · 19s


  full-history grid: 10000/12000 variations · 23s


  full-history grid: 12000/12000 variations · 28s


IS→OOS carry-over (post-registration diagnostic, not used for selection): spearman_val_vs_test_all = 0.158, spearman_score_vs_test_eligible = 0.043, primary_test_percentile = 0.805

**Multi-split results** · sample: `OOS-split / IS-selection-overlap`

,split,method,id,sample,oos_window,is_val_sharpe,sharpe,sharpe_lo,sharpe_hi,nw_p,CAGR,ann_vol,sortino,max_dd,hit_rate,profit_per_trade_pct,exposure,turnover_ann,alpha_ann,alpha_t,beta,spy_oos_sharpe,beats_spy,grid_oos_percentile
0,IS 1y,(a) re-selected #1,3124,OOS-split,2005-01-14..2026-09-21,4.406,0.04838,-0.3762,0.4647,0.8228,0.01774,0.004303,0.02558,-0.01143,0.5919,0.06338,0.02843,2.09,-0.0001252,-0.1385,0.00308,0.5597,False,0.6432
1,IS 1y,(b) fixed registered primary,4115,IS-selection-overlap,2005-01-14..2026-09-21,n/a,0.5224,0.1006,0.9691,0.01749,0.02067,0.005991,0.3587,-0.01398,0.6589,0.1704,0.04711,2.661,0.00243,1.902,0.00613,0.5597,False,n/a
2,IS 2y,(a) re-selected #1,109,OOS-split,2006-01-17..2026-09-21,3.546,-0.2234,-0.592,0.1784,0.2812,0.01477,0.009044,-0.1095,-0.02134,0.4369,-0.04102,0.03074,7.176,-0.002816,-1.51,0.007584,0.5567,False,0.2671
3,IS 2y,(b) fixed registered primary,4115,IS-selection-overlap,2006-01-17..2026-09-21,n/a,0.6047,0.1675,1.067,0.007866,0.02039,0.005883,0.4146,-0.01398,0.6648,0.1815,0.04499,2.588,0.002884,2.224,0.005713,0.5567,True,n/a
4,IS 3y,(a) re-selected #1,9354,OOS-split,2007-01-16..2026-09-21,2.653,-0.5801,-0.955,-0.1902,0.00657,0.01313,0.003863,-0.3005,-0.01005,0.5473,-0.1502,0.02178,1.835,-0.002333,-2.871,0.001648,0.5523,False,0.04754
5,IS 3y,(b) fixed registered primary,4115,IS-selection-overlap,2007-01-16..2026-09-21,n/a,0.6301,0.1767,1.07,0.006283,0.019,0.005875,0.4342,-0.01398,0.668,0.1857,0.04377,2.525,0.003053,2.325,0.005443,0.5523,True,n/a
6,IS 4y,(a) re-selected #1,3081,OOS-split,2008-01-14..2026-09-21,2.311,-0.3778,-0.801,0.04951,0.09545,0.01195,0.005071,-0.1923,-0.01194,0.4917,-0.07412,0.02605,3.361,-0.002041,-1.795,0.001396,0.5857,False,0.1426
7,IS 4y,(b) fixed registered primary,4115,IS-selection-overlap,2008-01-14..2026-09-21,n/a,0.5977,0.1382,1.06,0.01055,0.0173,0.005762,0.4076,-0.01398,0.6631,0.1745,0.04176,2.412,0.002782,2.131,0.005287,0.5857,True,n/a
8,IS 5y,(a) re-selected #1,7700,OOS-split,2009-01-14..2026-09-21,2.925,0.3458,-0.07355,0.757,0.1053,0.01554,0.004733,0.1445,-0.009762,0.6976,0.05852,0.01222,2.158,0.001778,1.724,-0.001286,0.8111,False,0.9127
9,IS 5y,(b) fixed registered primary,4115,IS-selection-overlap,2009-01-14..2026-09-21,n/a,0.6147,0.141,1.095,0.01066,0.01752,0.005858,0.4233,-0.01398,0.6681,0.1773,0.04268,2.465,0.002512,1.838,0.007215,0.8111,False,n/a


### 4.1 Monte Carlo (10,000 paths), t-tests and SPY comparison
- **Trade bootstrap:** resample the trades' portfolio contributions (`ret_net / available sleeves`) with replacement and compound them. This ignores trade overlap, so treat it as an approximation.
- **Stationary block bootstrap** (mean block 10) of daily portfolio returns over the window length.
- **Paired bootstrap vs SPY:** the same resampled days for both series.

In [23]:
def n_avail(idx):
    return np.array([sum(i >= FIRST_IDX[t] for t in TRADABLE) for i in idx])

def mc_trades(tr, years, name, n=None):
    n = n or CFG["bootstrap"]["n_mc"]
    c = (tr["ret_net"] / n_avail(tr["entry_idx"].values)).values
    if len(c) < 5:
        return {}
    G = rng(f"mc.trades.{name}").integers(0, len(c), (n, len(c)))
    paths = np.cumprod(1 + c[G], axis=1)
    final = paths[:, -1] - 1
    dd = (paths / np.maximum.accumulate(paths, axis=1) - 1).min(1)
    return dict(method="trade bootstrap", P_loss=(final < 0).mean(), final_p05=np.percentile(final, 5), final_p50=np.median(final),
                final_p95=np.percentile(final, 95), ann_pnl_p50=(1 + np.median(final)) ** (1 / years) - 1, maxdd_p05=np.percentile(dd, 5))

def mc_block(d, name, n=None, bench=None):
    n = n or CFG["bootstrap"]["n_mc"]
    r, ex = d["ret"].values, d["excess"].values
    T = len(r)
    sh, cagr, mdd, fin_, sh_b, fin_b = [], [], [], [], [], []
    for b in range(0, n, 2000):
        I = sb_indices(T, min(2000, n - b), f"mc.block.{name}.{b}")
        rr, ee = r[I], ex[I]
        sh.append(ee.mean(1) / ee.std(1, ddof=1) * np.sqrt(252))
        eq = np.cumprod(1 + rr, axis=1)
        cagr.append(eq[:, -1] ** (252 / T) - 1); fin_.append(eq[:, -1] - 1)
        mdd.append((eq / np.maximum.accumulate(eq, axis=1) - 1).min(1))
        if bench is not None:
            br, be = bench["ret"].values[I], bench["excess"].values[I]
            sh_b.append(be.mean(1) / be.std(1, ddof=1) * np.sqrt(252)); fin_b.append(np.cumprod(1 + br, axis=1)[:, -1] - 1)
    sh, cagr, mdd, fin_ = map(np.concatenate, (sh, cagr, mdd, fin_))
    out = dict(method="stationary block bootstrap", P_sharpe_le_0=(sh <= 0).mean(), sharpe_p05=np.percentile(sh, 5), sharpe_p50=np.median(sh),
               sharpe_p95=np.percentile(sh, 95), cagr_p05=np.percentile(cagr, 5), cagr_p50=np.median(cagr), cagr_p95=np.percentile(cagr, 95),
               maxdd_p05=np.percentile(mdd, 5), maxdd_p50=np.median(mdd), P_loss=(fin_ < 0).mean())
    if bench is not None:
        out["P_sharpe_above_SPY"] = (sh > np.concatenate(sh_b)).mean()
        out["P_final_above_SPY"] = (fin_ > np.concatenate(fin_b)).mean()
    return out, sh

MC_ROWS, MC_DIST = [], {}
for nm in [pr_name, "base 12/26/9"]:
    for samp, (d, tr_, _), w0 in [("IS", IS_RUNS[nm], WARM), ("OOS", OOS_RUNS[nm], I_TEST)]:
        yrs = len(d) / 252
        bench = buy_and_hold("SPY", w0, I_IS_END if samp == "IS" else NCAL)
        blk, shd = mc_block(d, f"{nm}.{samp}", bench=bench)
        MC_DIST[(nm, samp)] = shd
        MC_ROWS.append(dict(subject=nm, sample=samp, **blk))
        MC_ROWS.append(dict(subject=nm, sample=samp, **mc_trades(tr_, yrs, f"{nm}.{samp}")))
MC = pd.DataFrame(MC_ROWS)
show(MC, f"Monte Carlo ({CFG['bootstrap']['n_mc']:,} paths each)", "IS / OOS", fmt="{:.4f}")

fig = go.Figure()
for (nm, samp), shd in MC_DIST.items():
    fig.add_trace(go.Histogram(x=shd, name=f"{nm} · {samp}", opacity=0.55, nbinsx=80))
fig.update_layout(barmode="overlay", title="Block-bootstrap Sharpe distributions (IS vs OOS)", height=380, xaxis_title="annualised Sharpe")
fig.show()

tt = []
for nm in [pr_name, "base 12/26/9"]:
    for samp, (d, tr_, _) in [("IS", IS_RUNS[nm]), ("OOS", OOS_RUNS[nm])]:
        a, b = nw_ttest(d["excess"].values), nw_ttest(tr_.sort_values("entry_idx")["ret_excess"].values)
        tt.append(dict(subject=nm, sample=samp, daily_excess_mean_bp=1e4 * a["mean"], daily_t=a["t"], daily_p=a["p"],
                       trade_excess_mean_pct=100 * b["mean"], trade_t=b["t"], trade_p=b["p"], n_trades=b["n"]))
TT = pd.DataFrame(tt)
show(TT, "Newey–West (HAC) t-tests on daily and per-trade excess returns", "IS / OOS", fmt="{:.4f}")
print("Section 4 done ·", elapsed())

**Monte Carlo (10,000 paths each)** · sample: `IS / OOS`

,subject,sample,method,P_sharpe_le_0,sharpe_p05,sharpe_p50,sharpe_p95,cagr_p05,cagr_p50,cagr_p95,maxdd_p05,maxdd_p50,P_loss,P_sharpe_above_SPY,P_final_above_SPY,final_p05,final_p50,final_p95,ann_pnl_p50
0,#1 id 4115,IS,stationary block bootstrap,0.0233,0.0973,0.5542,1.0073,0.0125,0.0159,0.0193,-0.0163,-0.0103,0.0000,0.6490,0.0743,n/a,n/a,n/a,n/a
1,#1 id 4115,IS,trade bootstrap,n/a,n/a,n/a,n/a,n/a,n/a,n/a,-0.0124,n/a,0.0000,n/a,n/a,0.0432,0.0711,0.0998,0.0046
2,#1 id 4115,OOS,stationary block bootstrap,0.2440,-0.3441,0.2455,0.8178,0.0243,0.0285,0.0328,-0.0060,-0.0041,0.0000,0.1019,0.0212,n/a,n/a,n/a,n/a
3,#1 id 4115,OOS,trade bootstrap,n/a,n/a,n/a,n/a,n/a,n/a,n/a,-0.0122,n/a,0.0332,n/a,n/a,0.0020,0.0188,0.0363,0.0024
4,base 12/26/9,IS,stationary block bootstrap,0.9212,-0.7406,-0.3496,0.0534,0.0035,0.0082,0.0132,-0.0422,-0.0245,0.0020,0.0080,0.0596,n/a,n/a,n/a,n/a
5,base 12/26/9,IS,trade bootstrap,n/a,n/a,n/a,n/a,n/a,n/a,n/a,-0.1033,n/a,0.9698,n/a,n/a,-0.0962,-0.0523,-0.0075,-0.0036
6,base 12/26/9,OOS,stationary block bootstrap,0.7999,-0.7965,-0.2769,0.2621,0.0163,0.0238,0.0313,-0.0226,-0.0141,0.0000,0.0067,0.0137,n/a,n/a,n/a,n/a
7,base 12/26/9,OOS,trade bootstrap,n/a,n/a,n/a,n/a,n/a,n/a,n/a,-0.0603,n/a,0.8472,n/a,n/a,-0.0535,-0.0212,0.0126,-0.0028


**Newey–West (HAC) t-tests on daily and per-trade excess returns** · sample: `IS / OOS`

,subject,sample,daily_excess_mean_bp,daily_t,daily_p,trade_excess_mean_pct,trade_t,trade_p,n_trades
0,#1 id 4115,IS,0.1402,2.0514,0.0402,0.1582,2.8270,0.0047,387.0000
1,#1 id 4115,OOS,0.0499,0.6731,0.5009,0.0599,0.7821,0.4342,195.0000
2,base 12/26/9,IS,-0.1573,-1.4083,0.1590,-0.0459,-1.2371,0.2160,1027.0000
3,base 12/26/9,OOS,-0.1284,-0.8037,0.4216,-0.0374,-0.7765,0.4375,629.0000


Section 4 done · 7.8 min


## 5 · ML sizing overlay (random forest)

**Task.** For each entry signal of the base 12/26/9 strategy, decide **stay out** (size 0), **base size** (1×) or **increase size** (1.5×, i.e. 1.5% risk, the pre-specified cap).

**Label (no leakage).** Rows are entry signals at close t, pooled across ETFs. Each signal the base strategy actually traded gets the triple-barrier outcome of that trade: entry at open t+1, then exit at the stop, the target, the signal or time, net of costs. The labels are **0** if R_net ≤ 0, **1** if 0 < R_net ≤ 1, **2** if R_net > 1. Untraded signals (the sleeve was already in a position) are unlabeled but are still scored.

**Validation.** Expanding walk-forward by calendar year (test years 2008…2026). Training rows must have exited ≥ 10 trading days (the embargo) before the test year starts, which is the purge. The decision thresholds (τ_out, τ_up) are chosen on an inner validation year, the last year of the training window, by maximising the mean of size × R_net. There is no shuffling across time. Development folds are 2008–2018 (pseudo-OOS for the ML, inside the strategy's IS period); the final sealed test is 2019+.

**Primary candidate.** The registered primary is also checked for sample size. With too few IS signals (< 500) the ML run on it is BLOCKED.

In [24]:
MLC = CFG["ml"]
test_access("ML: base strategy signals/trades over full history (labels ≤ test year are used only for later folds)")
def full_run(p):
    _, tr_, sl_ = run_portfolio(p, 0, NCAL)
    eq_ = (1 + sl_.fillna(0)).cumprod()
    return tr_, (eq_ / eq_.cummax() - 1).values                          # trades, sleeve drawdown known at close t

BASE_FULL_TR, SLEEVE_DD = full_run(SB)

ev_high_idx = np.where(EVF["high_d0"].values)[0]
nxt = np.searchsorted(ev_high_idx, np.arange(NCAL), side="right")
DAYS_TO_HIGH = np.where(nxt < len(ev_high_idx), ev_high_idx[np.minimum(nxt, len(ev_high_idx) - 1)] - np.arange(NCAL), 99).astype(float)
VIX_CHG5 = (VIX / VIX.shift(5) - 1).values

FEATURE_TIMESTAMPS = {
    "x (MACD z-score)": "close t", "hist": "close t", "hist_slope": "close t (hist_t − hist_{t−1})", "relvol": "close t (segment-normalised)",
    "obv_slope": "close t", "vix": "VIX close t (16:15 ET, before open t+1)", "vix_chg5": "VIX close t vs t−5", "spy_trend": "SPY close t / SMA200 − 1",
    "own_trend": "close t / SMA200 − 1", "adx": "close t", "rv": "close t (20-day)", "rv_pct": "close t (252-day percentile)",
    "ev_any_next / ev_high_next": "schedule known before t (window around entry day t+1)", "days_to_next_high": "schedule known before t",
    "dow / month": "calendar of t", "side": "signal at t", "sleeve_dd": "base sleeve equity ≤ close t", "ticker one-hot": "static"}

def build_ml_dataset(p, feat=FEAT, sleeve_dd=None, trades=None):
    rows = []
    tick_idx = {t: j for j, t in enumerate(TRADABLE)}
    for t in TRADABLE:
        F = feat[t]
        M = macd_for(t, p) if feat is FEAT else macd_features(F["close"], F["valid"], p["fast"], p["slow"], p["signal"], p["signal_kind"], p["z_window"])
        le, se, _, _ = build_signals(F, M, p)
        idx = np.where((le | se) & (np.arange(NCAL) >= WARM) & (np.arange(NCAL) < NCAL - 1))[0]
        j = tick_idx[t]
        for i in idx:
            rows.append(dict(ticker=t, signal_idx=i, side=1 if le[i] else -1, x=M["x"][i], hist=M["hist"][i], hist_slope=M["hist"][i] - M["hist_prev"][i],
                             relvol=F["relvol"][i], obv_slope=F["obv_slope"][i], vix=F["vix"][i], vix_chg5=VIX_CHG5[i], spy_trend=F["spy_trend"][i],
                             own_trend=F["close"][i] / F["sma200"][i] - 1, adx=F["adx"][i], rv=F["rv"][i], rv_pct=F["rv_pct"][i],
                             ev_any_next=EV_NEXT["any"][i], ev_high_next=EV_NEXT["high"][i], days_to_next_high=DAYS_TO_HIGH[i],
                             dow=CAL[i].dayofweek, month=CAL[i].month, sleeve_dd=(SLEEVE_DD if sleeve_dd is None else sleeve_dd)[i, j]))
    D = pd.DataFrame(rows).sort_values(["signal_idx", "ticker"]).reset_index(drop=True)
    for t in TRADABLE:
        D[f"tk_{t}"] = (D["ticker"] == t).astype(float)
    if trades is not None:
        D = D.join(trades.set_index(["ticker", "signal_idx"])[["R_net", "ret_excess", "exit_idx"]], on=["ticker", "signal_idx"])
        D["y"] = np.select([D["R_net"] <= 0, D["R_net"] <= MLC["label_r_hi"]], [0, 1], 2).astype(float)
        D.loc[D["R_net"].isna(), "y"] = np.nan
        D["year"] = CAL[D["signal_idx"]].year
    return D

DS = build_ml_dataset(SB, trades=BASE_FULL_TR)
ML_FEATS = [c for c in DS.columns if c not in ("ticker", "signal_idx", "R_net", "ret_excess", "exit_idx", "y", "year")]

# point-in-time check: blank every input after a cut date, rebuild, compare rows up to the cut
def ml_pit_check(cut="2016-12-30"):
    k = int(CAL.searchsorted(pd.Timestamp(cut)))
    feat2 = {}
    for t in TRADABLE:
        b2 = BARS[t].copy()
        b2.iloc[k + 1:, :4] = b2.iloc[k + 1:, :4].values * 1.37
        b2.iloc[k + 1:, b2.columns.get_loc("volume")] *= 3
        feat2[t] = static_features(b2, BARS["SPY"]["close"].where(np.arange(NCAL) <= k, BARS["SPY"]["close"] * 0.7))
    d2 = build_ml_dataset(SB, feat2)
    a = DS[DS["signal_idx"] <= k][ML_FEATS].reset_index(drop=True)
    b = d2[d2["signal_idx"] <= k][ML_FEATS].reset_index(drop=True)
    return bool(len(a) == len(b) and np.allclose(a.values.astype(float), b.values.astype(float), equal_nan=True))

PIT_OK = ml_pit_check()
cls_bal = DS["y"].value_counts().sort_index()
show(pd.DataFrame([dict(signals=len(DS), labelled=int(DS["y"].notna().sum()), **{f"class_{int(k)}": int(v) for k, v in cls_bal.items()},
                        features=len(ML_FEATS), pit_perturbation_identical=PIT_OK)]), "ML dataset (base 12/26/9 signals, 2004–2026)", "n/a")
show(pd.DataFrame(list(FEATURE_TIMESTAMPS.items()), columns=["feature", "known at"]), "Feature timestamps", "n/a")

**ML dataset (base 12/26/9 signals, 2004–2026)** · sample: `n/a`

,signals,labelled,class_0,class_1,class_2,features,pit_perturbation_identical
0,3454,1656,853,566,237,30,True


**Feature timestamps** · sample: `n/a`

,feature,known at
0,x (MACD z-score),close t
1,hist,close t
2,hist_slope,close t (hist_t − hist_{t−1})
3,relvol,close t (segment-normalised)
4,obv_slope,close t
5,vix,"VIX close t (16:15 ET, before open t+1)"
6,vix_chg5,VIX close t vs t−5
7,spy_trend,SPY close t / SMA200 − 1
8,own_trend,close t / SMA200 − 1
9,adx,close t


In [25]:
def fit_rf(Xtr, ytr, seed):
    m = RandomForestClassifier(n_estimators=MLC["n_estimators"], max_depth=MLC["max_depth"], min_samples_leaf=MLC["min_samples_leaf"],
                               max_features=MLC["max_features"], class_weight=MLC["class_weight"], n_jobs=MLC["n_jobs"], random_state=seed)
    m.fit(Xtr, ytr)
    m.n_jobs = 1                                                          # deterministic predict_proba
    return m

def proba3(m, X):
    p = np.zeros((len(X), 3))
    p[:, m.classes_.astype(int)] = m.predict_proba(X)
    return p

def decide(P, tau_out, tau_up):
    sm_ = MLC["size_multipliers"]
    return np.where(P[:, 0] >= tau_out, sm_["out"], np.where(P[:, 2] >= tau_up, sm_["up"], sm_["base"]))

from sklearn.metrics import make_scorer
LL_SCORER = make_scorer(log_loss, greater_is_better=False, response_method="predict_proba", labels=[0, 1, 2])
emb = MLC["embargo_days"]

def walk_forward(DS, tag):
    folds, preds, mdi, perm_rows, audit = [], [], [], [], []
    for Y in range(MLC["walk_forward_first_test_year"], CAL[-1].year + 1):
        i_y = int(CAL.searchsorted(pd.Timestamp(Y, 1, 1)))
        i_v = int(CAL.searchsorted(pd.Timestamp(Y - 1, 1, 1)))
        lab_ok = DS["y"].notna()
        tr_m = lab_ok & (DS["signal_idx"] < i_y) & (DS["exit_idx"] < i_y - emb)
        in_tr = lab_ok & (DS["signal_idx"] < i_v) & (DS["exit_idx"] < i_v - emb)
        va = tr_m & (DS["signal_idx"] >= i_v)
        te = DS["year"] == Y
        if tr_m.sum() < 200 or te.sum() == 0:
            continue
        m_in = fit_rf(DS.loc[in_tr, ML_FEATS].values, DS.loc[in_tr, "y"].values, seed_for(f"ml.rf.{tag}.{Y}.inner"))
        Pv = proba3(m_in, DS.loc[va, ML_FEATS].values)
        Rv = DS.loc[va, "R_net"].values
        best = max(((np.mean(decide(Pv, a, b) * Rv) if len(Rv) else 0.0, a, b) for a in MLC["threshold_grid_out"] for b in MLC["threshold_grid_up"]),
                   key=lambda z: (round(z[0], 12), z[1], z[2]))   # ties → fewest skips / ups
        m = fit_rf(DS.loc[tr_m, ML_FEATS].values, DS.loc[tr_m, "y"].values, seed_for(f"ml.rf.{tag}.{Y}"))
        Pt = proba3(m, DS.loc[te, ML_FEATS].values)
        preds.append(pd.DataFrame({"row": DS.index[te], "p0": Pt[:, 0], "p1": Pt[:, 1], "p2": Pt[:, 2], "mult": decide(Pt, best[1], best[2]), "year": Y}))
        mdi.append(pd.Series(m.feature_importances_, index=ML_FEATS, name=Y))
        te_l = te & lab_ok
        if te_l.sum() >= 20:
            pi = permutation_importance(m, DS.loc[te_l, ML_FEATS].values, DS.loc[te_l, "y"].values, n_repeats=5, random_state=seed_for(f"ml.perm.{Y}"),
                                        scoring=LL_SCORER, n_jobs=1)
            perm_rows.append(pd.Series(pi.importances_mean, index=ML_FEATS, name=Y))
        folds.append(dict(test_year=Y, window="dev" if Y <= IS_END.year else "final", n_train=int(tr_m.sum()), n_inner_train=int(in_tr.sum()),
                          n_val=int(va.sum()), n_test=int(te.sum()), tau_out=best[1], tau_up=best[2], val_objective=best[0]))
        audit.append(dict(test_year=Y, train_exit_before_embargo=bool((DS.loc[tr_m, "exit_idx"] < i_y - emb).all()),
                          train_signals_before_test=bool((DS.loc[tr_m, "signal_idx"] < i_y).all()),
                          thresholds_from_prior_year_only=bool((DS.loc[va, "signal_idx"] < i_y).all()),
                          no_shuffle=True, min_gap_days=int(i_y - DS.loc[tr_m, "exit_idx"].max())))
    PRED = pd.concat(preds).set_index("row").join(DS[["ticker", "signal_idx", "y", "R_net"]])
    return pd.DataFrame(folds), PRED, pd.DataFrame(audit), mdi, perm_rows

FOLDS, PRED, LEAK, mdi, perm_rows = walk_forward(DS, "base")
show(FOLDS, "Walk-forward folds (purged, embargoed, expanding)", "dev = IS-pseudo-OOS · final = OOS", fmt={"val_objective": "{:.3f}"})
leak_ok = bool(LEAK.drop(columns=["test_year", "min_gap_days"]).all().all() and PIT_OK and (LEAK["min_gap_days"] >= emb).all())
show(LEAK.assign(pit_perturbation=PIT_OK), "Leakage audit per fold", "n/a")
check("5", "ML leakage audit (purge, embargo, thresholds from prior year, no shuffle, PIT perturbation)", leak_ok, f"{len(LEAK)} folds; PIT {PIT_OK}")

**Walk-forward folds (purged, embargoed, expanding)** · sample: `dev = IS-pseudo-OOS · final = OOS`

,test_year,window,n_train,n_inner_train,n_val,n_test,tau_out,tau_up,val_objective
0,2008,dev,235,148,82,134,0.650000,1.010000,-0.142
1,2009,dev,305,235,66,144,0.650000,1.010000,-0.202
2,2010,dev,363,305,57,140,0.650000,1.010000,-0.140
3,2011,dev,437,363,68,152,0.650000,0.300000,0.368
4,2012,dev,511,437,73,155,0.650000,1.010000,-0.075
5,2013,dev,584,511,73,145,0.400000,1.010000,-0.243
6,2014,dev,663,584,74,171,0.650000,1.010000,-0.115
7,2015,dev,742,663,75,134,0.650000,0.350000,0.086
8,2016,dev,806,742,60,184,0.650000,0.400000,-0.015
9,2017,dev,891,806,83,120,0.650000,1.010000,-0.192


**Leakage audit per fold** · sample: `n/a`

,test_year,train_exit_before_embargo,train_signals_before_test,thresholds_from_prior_year_only,no_shuffle,min_gap_days,pit_perturbation
0,2008,True,True,True,True,16,True
1,2009,True,True,True,True,11,True
2,2010,True,True,True,True,18,True
3,2011,True,True,True,True,15,True
4,2012,True,True,True,True,19,True
5,2013,True,True,True,True,13,True
6,2014,True,True,True,True,19,True
7,2015,True,True,True,True,13,True
8,2016,True,True,True,True,14,True
9,2017,True,True,True,True,14,True


In [26]:
def clf_diag(win):
    P_ = PRED[(PRED["year"] <= IS_END.year) if win == "dev" else (PRED["year"] > IS_END.year)]
    P_ = P_[P_["y"].notna()]
    y_, Pm = P_["y"].astype(int).values, P_[["p0", "p1", "p2"]].values
    prior = np.bincount(DS.loc[DS["y"].notna() & (DS["year"] < P_["year"].min()), "y"].astype(int), minlength=3) / max(1, (DS["y"].notna() & (DS["year"] < P_["year"].min())).sum())
    return dict(window=win, n=len(y_), log_loss=log_loss(y_, Pm, labels=[0, 1, 2]), prior_log_loss=log_loss(y_, np.tile(prior, (len(y_), 1)), labels=[0, 1, 2]),
                balanced_accuracy=balanced_accuracy_score(y_, Pm.argmax(1)), share_out=(P_["mult"] == 0).mean(), share_up=(P_["mult"] > 1).mean()), \
           confusion_matrix(y_, Pm.argmax(1), labels=[0, 1, 2]), (y_, Pm)

diag_rows = []
fig = make_subplots(rows=1, cols=4, subplot_titles=["confusion · dev", "confusion · final (OOS)", "calibration · dev", "calibration · final (OOS)"])
for j, win in enumerate(["dev", "final"]):
    d_, cm_, (y_, Pm) = clf_diag(win)
    diag_rows.append(d_)
    fig.add_trace(go.Heatmap(z=cm_, x=["pred out", "pred base", "pred up"], y=["true 0", "true 1", "true 2"], text=cm_, texttemplate="%{text}",
                             colorscale="Blues", showscale=False), 1, j + 1)
    for c_ in range(3):
        bins = np.clip((Pm[:, c_] * 10).astype(int), 0, 9)
        obs = pd.Series((y_ == c_).astype(float)).groupby(bins).mean()
        prd = pd.Series(Pm[:, c_]).groupby(bins).mean()
        fig.add_trace(go.Scatter(x=prd.values, y=obs.values, mode="lines+markers", name=f"class {c_} ({win})", showlegend=(j == 0)), 1, j + 3)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dot", color="grey"), showlegend=False), 1, j + 3)
fig.update_layout(height=360, title="Classifier diagnostics (labelled signals only)")
fig.show()
show(pd.DataFrame(diag_rows), "Classifier quality (log loss vs training-prior log loss; chance balanced accuracy = 0.333)", "dev / OOS", fmt="{:.3f}")

imp = pd.DataFrame({"MDI (mean over folds)": pd.concat(mdi, axis=1).mean(1),
                    "permutation Δlog-loss dev": pd.concat([s for s in perm_rows if s.name <= IS_END.year], axis=1).mean(1),
                    "permutation Δlog-loss final": pd.concat([s for s in perm_rows if s.name > IS_END.year], axis=1).mean(1)}).sort_values("MDI (mean over folds)")
fig = go.Figure([go.Bar(y=imp.index, x=imp[c], name=c, orientation="h") for c in imp.columns])
fig.update_layout(barmode="group", height=700, title="Feature importance (MDI and out-of-fold permutation importance)")
fig.show()

**Classifier quality (log loss vs training-prior log loss; chance balanced accuracy = 0.333)** · sample: `dev / OOS`

,window,n,log_loss,prior_log_loss,balanced_accuracy,share_out,share_up
0,dev,788.000,1.085,1.019,0.362,0.027,0.096
1,final,629.000,1.066,0.986,0.407,0.172,0.293


### 5.1 ML-sized backtest vs base vs SPY

In [27]:
def size_map(PRED):
    sm_ = {t: np.ones(NCAL) for t in TRADABLE}
    for tk, si, mlt in PRED[["ticker", "signal_idx", "mult"]].itertuples(index=False):
        sm_[tk][int(si)] = mlt
    return sm_

def ml_backtest(p, smult, label):
    tab, runs = [], {}
    for win, (a, b), samp in [("dev 2008–2018", (I_DEV, I_IS_END), "IS-pseudo-OOS"), ("final 2019+", (I_TEST, NCAL), "OOS")]:
        db, tb, _ = run_portfolio(p, a, b)
        dm, tm, _ = run_portfolio(p, a, b, smults=smult)
        ds = buy_and_hold("SPY", a, b)
        runs[win] = (db, dm, ds, tb, tm)
        for nm, d, t_ in [(label, db, tb), (f"ML-sized {label}", dm, tm), ("SPY buy&hold", ds, None)]:
            tab.append({**perf_stats(d, t_, f"ml.{nm}.{win}"), "subject": nm, "window": win, "sample": samp})
    return pd.DataFrame(tab), runs

I_DEV = int(CAL.searchsorted(pd.Timestamp(MLC["walk_forward_first_test_year"], 1, 1)))
SMULT = size_map(PRED)
ML_TAB, ML_RUNS = ml_backtest(SB, SMULT, "base 12/26/9")
show(ML_TAB[["subject", "window", "sample", "sharpe", "sharpe_lo", "sharpe_hi", "CAGR", "ann_vol", "sortino", "max_dd", "hit_rate", "profit_per_trade_pct",
             "n_trades", "exposure", "turnover_ann", "alpha_ann", "alpha_t", "beta"]], "Base vs ML vs SPY", "IS-pseudo-OOS / OOS", fmt="{:.4g}")

db, dm, ds, tb, tm = ML_RUNS["final 2019+"]
fig = go.Figure()
for nm, d in [("base 12/26/9", db), ("ML-sized base", dm), ("SPY buy&hold", ds)]:
    fig.add_trace(go.Scatter(x=d.index, y=np.cumprod(1 + d["ret"].values), name=nm))
fig.update_layout(title="OOS 2019–2026: base vs ML vs SPY buy&hold (growth of 1, log)", yaxis_type="log", height=400)
fig.show()

diff = dm["excess"].values - db["excess"].values
nw_diff = nw_ttest(diff)
I = sb_indices(len(dm), CFG["bootstrap"]["n_ci_boot"], "ml.sharpe_diff")
sd = (lambda e: e.mean(1) / e.std(1, ddof=1) * np.sqrt(252))(dm["excess"].values[I]) - (lambda e: e.mean(1) / e.std(1, ddof=1) * np.sqrt(252))(db["excess"].values[I])
SD_CI = np.percentile(sd, [2.5, 97.5])
g = rng("ml.placebo")
null = []
pfin = PRED[PRED["year"] > IS_END.year]
for _ in range(200):
    sm2 = {t: np.ones(NCAL) for t in TRADABLE}
    for Y, grp in pfin.groupby("year"):
        mults = g.permutation(grp["mult"].values)
        for (tk, si), mlt in zip(grp[["ticker", "signal_idx"]].itertuples(index=False), mults):
            sm2[tk][si] = mlt
    null.append(sharpe(port_fast(SB, I_TEST, NCAL, smults=sm2)[0]))
ml_sh = sharpe(dm["excess"].values)
ML_PLACEBO_P = float((np.array(null) >= ml_sh).mean())
mc_ml, _ = mc_block(dm, "ml.final", bench=ds)
tr_t = nw_ttest(tm.sort_values("entry_idx")["ret_excess"].values)
base_sh = sharpe(db["excess"].values)
G5_PASS = bool(ml_sh > base_sh and SD_CI[0] > 0)
ML_VERDICT = pd.DataFrame([dict(sample="OOS", ml_sharpe=ml_sh, base_sharpe=base_sh, sharpe_diff_ci_lo=SD_CI[0], sharpe_diff_ci_hi=SD_CI[1],
                                nw_t_ml_minus_base=nw_diff["t"], nw_p_ml_minus_base=nw_diff["p"], ml_trade_t=tr_t["t"], ml_trade_p=tr_t["p"],
                                placebo_p_shuffled_decisions=ML_PLACEBO_P, mc_P_sharpe_le_0=mc_ml["P_sharpe_le_0"], mc_sharpe_p05=mc_ml["sharpe_p05"],
                                mc_sharpe_p95=mc_ml["sharpe_p95"], ML_improves_OOS=G5_PASS)])
show(ML_VERDICT, "ML verdict on the sealed test window", "OOS", fmt="{:.4f}")
check("5", "G5 ML improves OOS risk-adjusted return (Sharpe > base and paired CI > 0)", G5_PASS,
      f"ML {ml_sh:.3f} vs base {base_sh:.3f}; paired Sharpe-diff CI [{SD_CI[0]:.3f}, {SD_CI[1]:.3f}]")

PRIM_FULL_TR, PRIM_DD = full_run(PRIMARY)
DS_P = build_ml_dataset(PRIMARY, sleeve_dd=PRIM_DD, trades=PRIM_FULL_TR)
prim_is_sig = int((DS_P["signal_idx"] < I_IS_END).sum())
if prim_is_sig >= 500:
    F_P, PRED_P, LEAK_P, _, _ = walk_forward(DS_P, "primary")
    TAB_P, RUNS_P = ml_backtest(PRIMARY, size_map(PRED_P), "primary")
    show(TAB_P[["subject", "window", "sample", "sharpe", "sharpe_lo", "sharpe_hi", "CAGR", "max_dd", "n_trades"]], "ML overlay on the registered primary", "IS-pseudo-OOS / OOS", fmt="{:.4g}")
    _, dmp, _, _, _ = RUNS_P["final 2019+"]
    dbp = RUNS_P["final 2019+"][0]
    Ip = sb_indices(len(dmp), CFG["bootstrap"]["n_ci_boot"], "ml.sharpe_diff.primary")
    sdp = (lambda e: e.mean(1) / e.std(1, ddof=1) * np.sqrt(252))(dmp["excess"].values[Ip]) - (lambda e: e.mean(1) / e.std(1, ddof=1) * np.sqrt(252))(dbp["excess"].values[Ip])
    PRIMARY_ML_STATUS = f"run: OOS Sharpe ML {sharpe(dmp['excess'].values):.3f} vs primary {sharpe(dbp['excess'].values):.3f}, paired CI [{np.percentile(sdp, 2.5):.3f}, {np.percentile(sdp, 97.5):.3f}]"
    check("5", "ML on the registered primary (≥500 IS signals)", bool(sharpe(dmp["excess"].values) > sharpe(dbp["excess"].values) and np.percentile(sdp, 2.5) > 0),
          PRIMARY_ML_STATUS + f"; leakage audit {'PASS' if LEAK_P.drop(columns=['test_year', 'min_gap_days']).all().all() else 'FAIL'}")
else:
    PRIMARY_ML_STATUS = f"BLOCKED: {prim_is_sig} IS entry signals < 500"
    check("5", "ML on the registered primary", False, PRIMARY_ML_STATUS, status="BLOCKED")
md(f"**Verdict:** the ML overlay on the base strategy {'improves' if G5_PASS else '**does not** improve'} OOS risk-adjusted returns after costs under the pre-set rule. "
   f"Primary-candidate ML: {PRIMARY_ML_STATUS}.")
print("Section 5 done ·", elapsed())

**Base vs ML vs SPY** · sample: `IS-pseudo-OOS / OOS`

,subject,window,sample,sharpe,sharpe_lo,sharpe_hi,CAGR,ann_vol,sortino,max_dd,hit_rate,profit_per_trade_pct,n_trades,exposure,turnover_ann,alpha_ann,alpha_t,beta
0,base 12/26/9,dev 2008–2018,IS-pseudo-OOS,-0.1907,-0.7667,0.3869,0.002406,0.01131,-0.1617,-0.02981,0.4772,-0.02502,788,0.09858,7.764,-0.00272,-0.8102,0.006632
1,ML-sized base 12/26/9,dev 2008–2018,IS-pseudo-OOS,-0.2216,-0.7641,0.3695,0.001895,0.012,-0.1863,-0.03473,0.4716,-0.03156,774,0.0984,7.76,-0.003241,-0.9164,0.006856
2,SPY buy&hold,dev 2008–2018,IS-pseudo-OOS,0.4236,-0.06525,0.9825,0.07197,0.2006,0.4024,-0.5187,n/a,n/a,0,1,0,-3.006e-05,-1.415,1
3,base 12/26/9,final 2019+,OOS,-0.2712,-0.8796,0.355,0.02387,0.01201,-0.2221,-0.02189,0.5072,-0.03723,629,0.09875,8.115,-0.004528,-1.106,0.008431
4,ML-sized base 12/26/9,final 2019+,OOS,-0.1323,-0.7695,0.5191,0.02544,0.01291,-0.1083,-0.0224,0.519,-0.01048,553,0.09152,7.557,-0.003504,-0.7948,0.01179
5,SPY buy&hold,final 2019+,OOS,0.7955,0.142,1.538,0.1753,0.1927,0.7537,-0.3372,n/a,n/a,0,1,0,-4.153e-05,-1.415,1


**ML verdict on the sealed test window** · sample: `OOS`

,sample,ml_sharpe,base_sharpe,sharpe_diff_ci_lo,sharpe_diff_ci_hi,nw_t_ml_minus_base,nw_p_ml_minus_base,ml_trade_t,ml_trade_p,placebo_p_shuffled_decisions,mc_P_sharpe_le_0,mc_sharpe_p05,mc_sharpe_p95,ML_improves_OOS
0,OOS,-0.1323,-0.2712,-0.0677,0.3352,1.0764,0.2818,-0.3289,0.7422,0.0350,0.6567,-0.6598,0.4183,False


**ML overlay on the registered primary** · sample: `IS-pseudo-OOS / OOS`

,subject,window,sample,sharpe,sharpe_lo,sharpe_hi,CAGR,max_dd,n_trades
0,primary,dev 2008–2018,IS-pseudo-OOS,0.8141,0.2247,1.44,0.00953,-0.01398,277
1,ML-sized primary,dev 2008–2018,IS-pseudo-OOS,0.7887,0.214,1.409,0.0105,-0.02119,277
2,SPY buy&hold,dev 2008–2018,IS-pseudo-OOS,0.4236,-0.06525,0.9825,0.07197,-0.5187,0
3,primary,final 2019+,OOS,0.2429,-0.4463,0.9461,0.02854,-0.005174,195
4,ML-sized primary,final 2019+,OOS,0.3424,-0.3773,1.062,0.02942,-0.007215,194
5,SPY buy&hold,final 2019+,OOS,0.7955,0.142,1.538,0.1753,-0.3372,0


**Verdict:** the ML overlay on the base strategy **does not** improve OOS risk-adjusted returns after costs under the pre-set rule. Primary-candidate ML: run: OOS Sharpe ML 0.342 vs primary 0.243, paired CI [-0.014, 0.205].

Section 5 done · 8.9 min


## 6 · Prop-firm (Topstep Trading Combine) viability · **PROVISIONAL**

Prop firms trade futures, not ETFs. Topstep also requires every position to be **flat by 15:10 CT** and forbids overnight holds, so the multi-day ETF swing strategy is **inadmissible as-is** (A14). The simulation therefore uses an **intraday adaptation**:
- Take each strategy entry.
- Enter at the RTH open of the entry day with the same ATR stop distance.
- Exit at the stop if touched, otherwise at that day's close.
- Translate the ETF return into discrete futures contracts on a Topstep-permitted product.

This is a *different strategy* from the one evaluated above, and the overnight part of the swing return is lost.

Rule values were transcribed from Topstep's public help-centre pages on 2026-10-20 and **must be verified** before any real pass/fail conclusion. Fields not used by the simulator (payout tiers, XFA scaling plan) are omitted.

In [28]:
PROP_RULES = {
    "_status": "PROVISIONAL: transcribed from help.topstep.com (retrieved 2026-10-20); verify before any pass/fail conclusion",
    "accounts": {"50K": dict(start=50_000, target=3_000, mll=2_000, dll=1_000, max_minis=5, max_micros=50, monthly_fee=49),
                 "100K": dict(start=100_000, target=6_000, mll=3_000, dll=2_000, max_minis=10, max_micros=100, monthly_fee=99),
                 "150K": dict(start=150_000, target=9_000, mll=4_500, dll=3_000, max_minis=15, max_micros=150, monthly_fee=199)},
    "mll": "trails the end-of-day balance high-water mark, never moves down, locks at the starting balance; breached intraday on realised + unrealised P&L",
    "dll": "optional (chosen at checkout); a breach flattens the account for the day and is not a rule violation",
    "consistency_best_day_max_share": 0.55,
    "min_trading_days": 2,
    "flat_by_ct": "15:10", "overnight_holding_allowed": False,
    "fees_round_turn_usd": {"ES": 3.78, "MES": 1.22, "NQ": 3.78, "MNQ": 1.22, "YM": 3.78, "MYM": 1.22, "RTY": 3.78, "M2K": 1.22, "GC": 4.32,
                            "MGC": 1.92, "SI": 4.32, "SIL": 2.72, "HG": 4.32, "MHG": 1.92, "CL": 4.02, "MCL": 1.72, "ZT": 2.32, "ZN": 2.62, "ZB": 2.76},
    "temporary_caps": {"_order": ["50K", "100K", "150K"], "SI": [0, 0, 0], "HG": [0, 0, 0], "SIL": [2, 5, 7], "MHG": [2, 5, 7],
                       "GC": [3, 6, 9], "CL": [3, 6, 9], "MGC": [31, 62, 93], "MCL": [31, 62, 93]},
    "sources": ["https://help.topstep.com (Trading Combine parameters, Maximum Loss Limit, Daily Loss Limit, Consistency Target, Scaling/position limits, Fees, Risk adjustments, Trading hours)"],
}
# CME contract specs (multiplier per point, tick size, tick value) and Topstep-permitted mapping. Micro listing dates: CME launch notices.
FUT = {
    "SPY": dict(mini="ES", mm=50, micro="MES", um=5, tick=0.25, micro_from="2019-05-06", yf="ES"),
    "QQQ": dict(mini="NQ", mm=20, micro="MNQ", um=2, tick=0.25, micro_from="2019-05-06", yf="NQ"),
    "DIA": dict(mini="YM", mm=5, micro="MYM", um=0.5, tick=1.0, micro_from="2019-05-06", yf="YM"),
    "IWM": dict(mini="RTY", mm=50, micro="M2K", um=5, tick=0.1, micro_from="2019-05-06", yf="RTY", mini_from="2017-07-10"),
    "GLD": dict(mini="GC", mm=100, micro="MGC", um=10, tick=0.1, micro_from="2010-10-04", yf="GC"),
    "SLV": dict(mini="SI", mm=5000, micro="SIL", um=1000, tick=0.005, micro_from="2013-06-17", yf="SI"),
    "USO": dict(mini="CL", mm=1000, micro=None, um=None, tick=0.01, micro_from=None, yf="CL"),     # MCL listing date unverified → not used
    "CPER": dict(mini="HG", mm=25000, micro=None, um=None, tick=0.0005, micro_from=None, yf="HG"),  # MHG listing date unverified → not used
    "TLT": dict(mini="ZB", mm=1000, micro=None, um=None, tick=1 / 32, micro_from=None, yf="ZB"),
    "IEF": dict(mini="ZN", mm=1000, micro=None, um=None, tick=1 / 64, micro_from=None, yf="ZN"),
    "SHY": dict(mini="ZT", mm=2000, micro=None, um=None, tick=1 / 256, micro_from=None, yf="ZT"),
}
md(f"**Rules status:** {PROP_RULES['_status']}")
show(pd.DataFrame(PROP_RULES["accounts"]).T, "Combine parameters used (USD)", "n/a")
show(pd.DataFrame(FUT).T, "ETF → Topstep-permitted futures mapping (CME specs)", "n/a")

FUT_CLOSE = {k: fut_yf[v["yf"]]["Close"].reindex(CAL).ffill(limit=3).values for k, v in FUT.items()}

def intraday_trades(tr: pd.DataFrame) -> pd.DataFrame:
    """Entry-day-only adaptation of swing trades: open → stop or close of the entry day (flat by 15:10 CT)."""
    d = tr[["ticker", "signal_idx", "entry_idx", "side", "entry_px", "shares", "risk_usd"]].copy()
    out = []
    for t, g in d.groupby("ticker"):
        F = FEAT[t]
        e = g["entry_idx"].values
        o, h, l, c = F["open"][e], F["high"][e], F["low"][e], F["close"][e]
        dist = (g["risk_usd"] / g["shares"]).values / o
        sd = g["side"].values
        stop_hit = np.where(sd == 1, l <= o * (1 - dist), h >= o * (1 + dist))
        r = np.where(stop_hit, -dist, sd * (c / o - 1))
        worst = np.maximum(-dist, np.where(sd == 1, l / o - 1, -(h / o - 1)))
        out.append(g.assign(dist_pct=dist, ret=r, worst=worst, F=FUT_CLOSE[t][g["signal_idx"].values]))
    return pd.concat(out).sort_values("entry_idx")

def to_futures(it: pd.DataFrame, risk_usd: float, acct: str, slip_ticks=None) -> tuple[pd.DataFrame, dict]:
    A = PROP_RULES["accounts"][acct]
    caps = PROP_RULES["temporary_caps"]
    ai = caps["_order"].index(acct)
    slip_ticks = CFG["prop"]["slippage_ticks_per_side"] if slip_ticks is None else slip_ticks
    rows, unsized = [], 0
    used = {}
    for r in it.itertuples(index=False):
        s = FUT[r.ticker]
        date = CAL[r.entry_idx]
        use_micro = CFG["prop"]["use_micros"] and s["micro"] and date >= pd.Timestamp(s["micro_from"])
        if s.get("mini_from") and not use_micro and date < pd.Timestamp(s["mini_from"]):
            unsized += 1
            continue
        sym, mult = (s["micro"], s["um"]) if use_micro else (s["mini"], s["mm"])
        if not np.isfinite(r.F) or r.F <= 0:
            unsized += 1
            continue
        n = int(risk_usd // (r.dist_pct * r.F * mult))
        n = min(n, caps.get(sym, [10**6] * 3)[ai], A["max_micros"] if use_micro else A["max_minis"])
        mini_eq = n / 10 if use_micro else n
        room = A["max_minis"] - used.get(r.entry_idx, 0.0)
        if mini_eq > room:
            n = int(room * (10 if use_micro else 1))
            mini_eq = n / 10 if use_micro else n
        if n <= 0:
            unsized += 1
            continue
        used[r.entry_idx] = used.get(r.entry_idx, 0.0) + mini_eq
        tick_val = s["tick"] * mult
        if sym == "SIL" and date < pd.Timestamp("2020-09-21"):
            tick_val = 0.01 * mult
        if sym == "ZT" and date < pd.Timestamp("2019-01-14"):
            tick_val = 15.625
        fees = n * (PROP_RULES["fees_round_turn_usd"][sym] + 2 * slip_ticks * tick_val)
        notional = n * mult * r.F
        rows.append(dict(entry_idx=r.entry_idx, ticker=r.ticker, symbol=sym, contracts=n, notional=notional,
                         pnl=notional * r.ret - fees, worst=notional * r.worst - fees))
    f = pd.DataFrame(rows)
    return f, dict(n_trades=len(it), n_sized=len(f), share_sized=len(f) / max(1, len(it)), unsized=unsized)

def daily_account(f: pd.DataFrame, start: int, end: int):
    pnl, worst = np.zeros(end - start), np.zeros(end - start)
    if len(f):
        g = f.groupby("entry_idx")[["pnl", "worst"]].sum()
        g = g[(g.index >= start) & (g.index < end)]
        pnl[g.index - start], worst[g.index - start] = g["pnl"].values, g["worst"].values
    return pnl, worst

@nb.njit(cache=False)
def combine_sim(pnl, worst, I, start_bal, target, mll, dll, cons, min_days):
    n, H = I.shape
    outcome = np.zeros(n, np.int8)      # 0 timeout, 1 pass, 2 fail (MLL)
    day = np.full(n, H, np.int32)
    dll_hits = np.zeros(n, np.int32)
    for b in range(n):
        bal, hw, level, best, nd = start_bal, start_bal, start_bal - mll, 0.0, 0
        for k in range(H):
            p, w = pnl[I[b, k]], worst[I[b, k]]
            if dll > 0 and w < -dll:
                w, dll_hits[b] = -dll, dll_hits[b] + 1
                if p < -dll:
                    p = -dll
            if bal + w <= level:
                outcome[b], day[b] = 2, k + 1
                break
            bal += p
            if p != 0.0 or w != 0.0:
                nd += 1
            if p > best:
                best = p
            if bal > hw:
                hw = bal
            lv = min(start_bal, hw - mll)
            if lv > level:
                level = lv
            prof = bal - start_bal
            if nd >= min_days and prof >= target and prof >= best / cons:
                outcome[b], day[b] = 1, k + 1
                break
    return outcome, day, dll_hits

def prop_mc(pnl, worst, acct, name, n=None, use_dll=True):
    A = PROP_RULES["accounts"][acct]
    n = n or CFG["prop"]["n_mc"]
    I = sb_indices(len(pnl), n, f"prop.{name}")[:, :CFG["prop"]["max_days"]] if len(pnl) >= CFG["prop"]["max_days"] else \
        np.tile(sb_indices(len(pnl), n, f"prop.{name}"), (1, int(np.ceil(CFG["prop"]["max_days"] / len(pnl)))))[:, :CFG["prop"]["max_days"]]
    o, d, dh = combine_sim(pnl, worst, I, float(A["start"]), float(A["target"]), float(A["mll"]), float(A["dll"] if use_dll else 0),
                           PROP_RULES["consistency_best_day_max_share"], PROP_RULES["min_trading_days"])
    res = dict(P_pass=(o == 1).mean(), P_fail_mll=(o == 2).mean(), P_timeout=(o == 0).mean(),
               median_days_to_pass=np.median(d[o == 1]) if (o == 1).any() else np.nan,
               median_days_to_fail=np.median(d[o == 2]) if (o == 2).any() else np.nan, share_paths_with_dll_hit=(dh > 0).mean())
    res["expected_fee_per_attempt"] = A["monthly_fee"] * np.ceil(d / 21).mean()
    return res, o, d

subj6 = {"primary": (PRIMARY, OOS_RUNS[pr_name][1], IS_RUNS[pr_name][1]), "base 12/26/9": (SB, OOS_RUNS["base 12/26/9"][1], IS_RUNS["base 12/26/9"][1])}
for i, k in enumerate(list(CAND_PARAMS)[1:], start=2):
    nm = f"#{i} id {k}"
    subj6[nm] = (CAND_PARAMS[k], OOS_RUNS[nm][1], IS_RUNS[nm][1])
IT = {(s, w): intraday_trades(trs) for s, (p, tro, tri) in subj6.items() for w, trs in [("IS", tri), ("OOS", tro)]}

test_access("prop simulation on the test window")
grid_rows = []
for (s, w), it in IT.items():
    for risk in CFG["prop"]["risk_per_trade_usd_grid"]:
        f, info = to_futures(it, risk, "50K")
        a, b = (WARM, I_IS_END) if w == "IS" else (I_TEST, NCAL)
        pnl, worst = daily_account(f, a, b)
        res, _, _ = prop_mc(pnl, worst, "50K", f"opt.{s}.{w}.{risk}", n=2000 if w == "IS" else None)
        grid_rows.append(dict(subject=s, sample=w, risk_usd=risk, **info, pnl_per_year=pnl.sum() / ((b - a) / 252),
                              median_notional=f["notional"].median() if len(f) else np.nan, **res))
PG = pd.DataFrame(grid_rows)
is_g = PG[(PG["sample"] == "IS") & (PG["n_sized"] > 0)].sort_values(["P_pass", "P_fail_mll", "risk_usd"], ascending=[False, True, True])   # a config that sizes no trade is not a strategy
CHOSEN = is_g.iloc[0][["subject", "risk_usd"]].to_dict()
show(PG[PG["risk_usd"].isin([300, 750])][["subject", "sample", "risk_usd", "share_sized", "pnl_per_year", "P_pass", "P_fail_mll", "P_timeout",
                                          "median_days_to_pass", "median_days_to_fail"]], "Intraday adaptation, Topstep 50K: reference risk levels", "IS / OOS", fmt="{:.4g}")
f_is, _ = to_futures(IT[(CHOSEN["subject"], "IS")], CHOSEN["risk_usd"], "50K")
f_oos, _ = to_futures(IT[(CHOSEN["subject"], "OOS")], CHOSEN["risk_usd"], "50K")
opt_rows = []
for w, f_, (a, b) in [("IS", f_is, (WARM, I_IS_END)), ("OOS", f_oos, (I_TEST, NCAL))]:
    pnl, worst = daily_account(f_, a, b)
    for acct in ["50K"]:
        res, o, d = prop_mc(pnl, worst, acct, f"chosen.{w}")
        opt_rows.append(dict(config=f"{CHOSEN['subject']} @ ${CHOSEN['risk_usd']:.0f}", account=acct, sample=w, **res))
        if w == "OOS":
            PROP_O, PROP_D = o, d
OPT = pd.DataFrame(opt_rows)
show(OPT, f"Constrained bootstrap optimisation: chosen on IS P(pass) over {PG['subject'].nunique()} subjects × {len(CFG['prop']['risk_per_trade_usd_grid'])} risk levels, validated on the held-out test window", "IS / OOS", fmt="{:.4f}")
acc_rows = []
for acct, scale in [("100K", 1.5), ("150K", 2.25)]:
    for w, (a, b) in [("IS", (WARM, I_IS_END)), ("OOS", (I_TEST, NCAL))]:
        f_, _ = to_futures(IT[(CHOSEN["subject"], w)], CHOSEN["risk_usd"] * scale, acct)
        pnl, worst = daily_account(f_, a, b)
        acc_rows.append(dict(account=acct, sample=w, risk_usd=CHOSEN["risk_usd"] * scale, **prop_mc(pnl, worst, acct, f"acct.{acct}.{w}")[0]))
show(pd.DataFrame(acc_rows), "Larger accounts (risk scaled by the MLL ratio)", "IS / OOS", fmt="{:.4f}")
P_OOS_PASS = OPT.loc[OPT["sample"] == "OOS", "P_pass"].iloc[0]
P_OOS_FAIL = OPT.loc[OPT["sample"] == "OOS", "P_fail_mll"].iloc[0]
G6_PASS = bool(P_OOS_PASS >= 0.25 and P_OOS_PASS > P_OOS_FAIL)
check("6", "G6 prop viability: OOS P(pass) ≥ 25% and > P(fail) (PROVISIONAL rules)", G6_PASS, f"OOS P(pass) {P_OOS_PASS:.2%}, P(fail) {P_OOS_FAIL:.2%}")
check("6", "Prop admissibility of the swing strategy (no overnight holds at Topstep)", False, "multi-day holds forbidden; only the intraday adaptation is simulated", status="FAIL")

fig = go.Figure()
for oc, lab in [(1, "pass"), (2, "fail (MLL)"), (0, "timeout")]:
    fig.add_trace(go.Histogram(x=PROP_D[PROP_O == oc], name=f"{lab} ({(PROP_O == oc).mean():.1%})", opacity=0.6, nbinsx=60))
fig.update_layout(barmode="overlay", title=f"OOS Combine outcomes, {OPT['config'].iloc[0]} (days to outcome, {CFG['prop']['n_mc']:,} paths)", height=360)
fig.show()

**Rules status:** PROVISIONAL: transcribed from help.topstep.com (retrieved 2026-10-20); verify before any pass/fail conclusion

**Combine parameters used (USD)** · sample: `n/a`

,start,target,mll,dll,max_minis,max_micros,monthly_fee
50K,50000,3000,2000,1000,5,50,49
100K,100000,6000,3000,2000,10,100,99
150K,150000,9000,4500,3000,15,150,199


**ETF → Topstep-permitted futures mapping (CME specs)** · sample: `n/a`

,mini,mm,micro,um,tick,micro_from,yf,mini_from
SPY,ES,50,MES,5,0.25,2019-05-06,ES,NaN
QQQ,NQ,20,MNQ,2,0.25,2019-05-06,NQ,NaN
DIA,YM,5,MYM,0.5,1.0,2019-05-06,YM,NaN
IWM,RTY,50,M2K,5,0.1,2019-05-06,RTY,2017-07-10
GLD,GC,100,MGC,10,0.1,2010-10-04,GC,NaN
SLV,SI,5000,SIL,1000,0.005,2013-06-17,SI,NaN
USO,CL,1000,None,None,0.01,None,CL,NaN
CPER,HG,25000,None,None,0.0005,None,HG,NaN
TLT,ZB,1000,None,None,0.03125,None,ZB,NaN
IEF,ZN,1000,None,None,0.015625,None,ZN,NaN


**Intraday adaptation, Topstep 50K: reference risk levels** · sample: `IS / OOS`

,subject,sample,risk_usd,share_sized,pnl_per_year,P_pass,P_fail_mll,P_timeout,median_days_to_pass,median_days_to_fail
2,primary,IS,300,0.02842,-8.882,0,0,1,n/a,n/a
5,primary,IS,750,0.1395,-65.05,0,0,1,n/a,n/a
8,primary,OOS,300,0.04103,-25.68,0,0,1,n/a,n/a
11,primary,OOS,750,0.4,-97.72,0,0.0005,0.9995,n/a,235
14,base 12/26/9,IS,300,0.04382,-76.96,0,0,1,n/a,n/a
17,base 12/26/9,IS,750,0.1636,-440.6,0,0.008,0.992,n/a,196.5
20,base 12/26/9,OOS,300,0.05087,-155.9,0,0.0003,0.9997,n/a,223
23,base 12/26/9,OOS,750,0.3943,-1415,0.0001,0.4406,0.5593,212,172
26,#2 id 2395,IS,300,0.0137,-9.707,0,0,1,n/a,n/a
29,#2 id 2395,IS,750,0.1119,-44.05,0,0,1,n/a,n/a


**Constrained bootstrap optimisation: chosen on IS P(pass) over 6 subjects × 6 risk levels, validated on the held-out test window** · sample: `IS / OOS`

,config,account,sample,P_pass,P_fail_mll,P_timeout,median_days_to_pass,median_days_to_fail,share_paths_with_dll_hit,expected_fee_per_attempt
0,primary @ $200,50K,IS,0.0000,0.0000,1.0000,n/a,n/a,0.0000,588.0000
1,primary @ $200,50K,OOS,0.0000,0.0000,1.0000,n/a,n/a,0.0000,588.0000


**Larger accounts (risk scaled by the MLL ratio)** · sample: `IS / OOS`

,account,sample,risk_usd,P_pass,P_fail_mll,P_timeout,median_days_to_pass,median_days_to_fail,share_paths_with_dll_hit,expected_fee_per_attempt
0,100K,IS,300.0000,0.0000,0.0000,1.0000,n/a,n/a,0.0000,1188.0000
1,100K,OOS,300.0000,0.0000,0.0000,1.0000,n/a,n/a,0.0000,1188.0000
2,150K,IS,450.0000,0.0000,0.0000,1.0000,n/a,n/a,0.0000,2388.0000
3,150K,OOS,450.0000,0.0000,0.0000,1.0000,n/a,n/a,0.0000,2388.0000


### 6.1 ETF → futures correlation, normal vs stress
Daily log returns of the ETF vs the yfinance continuous future, which are raw front contracts. Roll days are masked (|r| > 20%). Commodity and rate futures settle before the 16:00 ET ETF close, which lowers measured correlation. QQQ is also checked against the **local NQ 1-minute data** on RTH open→close returns. A "break" is flagged when correlation drops by > 0.1 or beta moves by > 0.25 from the full-sample value.

In [29]:
STRESS = {"full": (START, END), "gfc 2008-09": ("2008-09-01", "2009-03-31"), "covid 2020": ("2020-02-15", "2020-04-30"), "rates 2022": ("2022-01-01", "2022-12-31")}
corr_rows = []
for t, s in FUT.items():
    e = np.log(BARS[t]["close"]).diff()
    fz = np.log(fut_yf[s["yf"]]["Close"].reindex(CAL)).diff()
    ok = e.notna() & fz.notna() & (fz.abs() < 0.2) & (e.abs() < 0.2)
    base_c = base_b = None
    for nm, (a, b) in STRESS.items():
        m = ok & (CAL >= pd.Timestamp(a)) & (CAL <= pd.Timestamp(b))
        if m.sum() < 20:
            continue
        c_ = np.corrcoef(e[m], fz[m])[0, 1]
        beta = np.cov(fz[m], e[m])[0, 1] / e[m].var()
        if nm == "full":
            base_c, base_b = c_, beta
        corr_rows.append(dict(etf=t, future=s["yf"], window=nm, n=int(m.sum()), corr=c_, beta_fut_on_etf=beta,
                              break_flag=nm != "full" and (base_c - c_ > 0.1 or abs(beta - base_b) > 0.25)))
qo = np.log(BARS["QQQ"]["close"] / BARS["QQQ"]["open"])
nq = NQ.reindex(CAL)
no = np.log(nq["close"] / nq["open"])
for nm, (a, b) in STRESS.items():
    m = qo.notna() & no.notna() & (CAL >= pd.Timestamp(a)) & (CAL <= pd.Timestamp(b))
    if m.sum() >= 20:
        corr_rows.append(dict(etf="QQQ (RTH open→close)", future="NQ local 1-min front", window=nm, n=int(m.sum()), corr=np.corrcoef(qo[m], no[m])[0, 1],
                              beta_fut_on_etf=np.cov(no[m], qo[m])[0, 1] / qo[m].var(), break_flag=False))
CORR = pd.DataFrame(corr_rows)
show(CORR, "ETF ↔ futures mapping quality", "n/a", fmt={"corr": "{:.3f}", "beta_fut_on_etf": "{:.3f}"})
md("**Flags.** ETF→futures correlation and leverage assumptions can break in stress regimes: USO vs CL around the April 2020 negative-price event, and Treasury ETFs vs CTD-driven futures (duration mismatch, so unit beta overstates ZB P&L). "
   "Topstep products, caps and rules change over time. The temporary caps above are a single 2026-10-19 snapshot applied to all history. Micros did not exist before 2019 (equity index), so early-period sizing uses minis and many trades cannot be sized at small risk budgets.")
print("Section 6 done ·", elapsed())

**ETF ↔ futures mapping quality** · sample: `n/a`

,etf,future,window,n,corr,beta_fut_on_etf,break_flag
0,SPY,ES,full,5962,0.979,0.990,False
1,SPY,ES,gfc 2008-09,146,0.986,1.000,False
2,SPY,ES,covid 2020,52,0.947,0.961,False
3,SPY,ES,rates 2022,251,0.997,0.990,False
4,QQQ,NQ,full,5962,0.974,0.983,False
5,QQQ,NQ,gfc 2008-09,146,0.968,1.032,False
6,QQQ,NQ,covid 2020,52,0.949,0.926,False
7,QQQ,NQ,rates 2022,251,0.993,1.000,False
8,DIA,YM,full,5962,0.978,0.978,False
9,DIA,YM,gfc 2008-09,146,0.988,0.979,False


**Flags.** ETF→futures correlation and leverage assumptions can break in stress regimes: USO vs CL around the April 2020 negative-price event, and Treasury ETFs vs CTD-driven futures (duration mismatch, so unit beta overstates ZB P&L). Topstep products, caps and rules change over time. The temporary caps above are a single 2026-10-19 snapshot applied to all history. Micros did not exist before 2019 (equity index), so early-period sizing uses minis and many trades cannot be sized at small risk budgets.

Section 6 done · 9.1 min


## 7 · Viability report

Everything in this section is generated from the numbers computed above. Nothing is typed in by hand.

In [30]:
def row(df, **kw):
    m = np.ones(len(df), bool)
    for k, v in kw.items():
        m &= (df[k] == v).values
    return df[m].iloc[0]

b_is, b_oos = row(DEG, subject="base 12/26/9", sample="IS"), row(DEG, subject="base 12/26/9", sample="OOS")
p_is, p_oos = row(DEG, subject=pr_name, sample="IS"), row(DEG, subject=pr_name, sample="OOS")
s_is, s_oos = row(DEG, subject="SPY buy&hold", sample="IS"), row(DEG, subject="SPY buy&hold", sample="OOS")
m_dev, m_oos = row(ML_TAB, subject="ML-sized base 12/26/9", window="dev 2008–2018"), row(ML_TAB, subject="ML-sized base 12/26/9", window="final 2019+")
o_is, o_oos = row(OPT, sample="IS"), row(OPT, sample="OOS")
cols = ["sharpe", "sharpe_lo", "sharpe_hi", "CAGR", "ann_excess", "ann_vol", "sortino", "max_dd", "alpha_ann", "alpha_t", "beta", "hit_rate",
        "profit_per_trade_pct", "trades_per_year", "exposure", "turnover_ann"]
COMP = pd.DataFrame({"base MACD · IS": b_is[cols], "base MACD · OOS": b_oos[cols], "registered primary · IS": p_is[cols], "registered primary · OOS": p_oos[cols],
                     "ML-sized base · IS-pseudo-OOS (2008–18)": m_dev[cols], "ML-sized base · OOS": m_oos[cols], "SPY B&H · IS": s_is[cols], "SPY B&H · OOS": s_oos[cols]}).T
COMP["prop P(pass) / P(fail)"] = np.nan
COMP = COMP.astype(object)
COMP.loc["prop sim · IS", "prop P(pass) / P(fail)"] = f"{o_is['P_pass']:.2%} / {o_is['P_fail_mll']:.2%} ({o_is['config']})"
COMP.loc["prop sim · OOS", "prop P(pass) / P(fail)"] = f"{o_oos['P_pass']:.2%} / {o_oos['P_fail_mll']:.2%} ({o_oos['config']})"
show(COMP, "Comparison table (IS = 2004–2018 after warm-up; OOS = 2019-01-02..2026-09-21)", "IS / OOS",
     fmt={c: "{:.4g}" for c in cols})

GATES = pd.DataFrame([
    ("G0 data integrity", G0_PASS), ("G1 IS baseline edge", G1_PASS), ("G2 search significance", G2_PASS), ("G3 sealed OOS", G3_PASS),
    ("G4 multi-split robustness", G4_PASS), ("G5 ML improvement", G5_PASS), ("G6 prop viability (PROVISIONAL)", G6_PASS)], columns=["gate", "pass"])
GATES["outcome"] = np.where(GATES["pass"], "PASS", "FAIL")
GATES.loc[len(GATES)] = ["G7 paper trading", False, "eligible" if GATES["pass"].iloc[1:].all() else "NOT REACHED"]
show(GATES, "Decision gates", "n/a")

check("7", "Survivorship / ex-post universe (A15)", False, "11 surviving ETFs chosen ex post; no delisted ETFs", status="FAIL")
check("7", "Transaction costs realistic", True, "assumed bps (not measured); Section 3 cost-sensitivity tables", status="WARN")
check("7", "Unrealistic fills", True, "next-open fills, stop-first, gap fills at open, costs on every fill; fractional shares and no market impact", status="WARN")
check("7", "Volume features consistent across the 2022-03-07 source splice", True, "single-venue IEX volume after the splice (A10)", status="WARN")
check("7", "Test window pristine", True, "an earlier version of this study evaluated the same 2019–2026 window; this run reproduces it", status="WARN")
check("7", "Event timing point-in-time", True, "scheduled dates only; shutdown-delayed releases announced days ahead (A13)", status="WARN")
check("7", "Prop rules verified", True, "PROVISIONAL transcription; temporary caps are a 2026-10-19 snapshot", status="WARN")
check("3", "Selection used train/validation only; registration before test access", REGISTRATION["registered_utc"] <= TEST_ACCESS_LOG[0]["utc"],
      f"selection grid simulated to {CAL[I_IS_END - 1].date()} only; registered {REGISTRATION['registered_utc']}; {len(TEST_ACCESS_LOG)} logged test accesses after it")
CHK = pd.DataFrame(CHECKS)
show(CHK, "Leakage / overfitting checklist (PASS / FAIL / WARN / BLOCKED)", "n/a")
show(pd.DataFrame(TEST_ACCESS_LOG), "Sealed-test access log (this run)", "n/a")

if G1_PASS and G2_PASS and G3_PASS and G4_PASS and G5_PASS and G6_PASS:
    VERDICT = "PROCEED to paper trading (subject to the gates below)"
elif G3_PASS and G4_PASS:
    VERDICT = "REVISE"
else:
    VERDICT = "REJECT"

f = lambda x, d=3: "n/a" if x is None or (isinstance(x, float) and not np.isfinite(x)) else f"{x:.{d}f}"
pct = lambda x, d=2: "n/a" if not np.isfinite(x) else f"{100 * x:.{d}f}%"
mc_p = row(MC, subject=pr_name, sample="OOS", method="stationary block bootstrap")
mc_b = row(MC, subject="base 12/26/9", sample="OOS", method="stationary block bootstrap")
tt_p = row(TT, subject=pr_name, sample="OOS")
REPORT = f"""
### Verdict: **{VERDICT}**

**1. Strategy and exact rules.** MACD mean reversion on 11 ETFs (SPY QQQ DIA IWM GLD SLV USO CPER TLT IEF SHY; VIX/VXX not traded). Signals are evaluated at close t and filled at open t+1. Each trade risks 1% of sleeve equity to an ATR(14) stop (notional ≤ 2× equity). Costs are per-ETF bps assumptions plus borrow and financing. The portfolio is the equal-weight average of the sleeves.
- *Base (A5):* `{json.dumps(SB)}`
- *Registered primary (id {PRIMARY_ID}, selected on 2003–2018 only, registration sha1 `{REGISTRATION['sha1']}`):* `{json.dumps(PRIMARY)}`

**2. Data limitations.**
- The vendor files splice two differently adjusted sources on 2022-03-07; prices were rebased to the yfinance total-return basis and {int(AUDIT['bad_bars_replaced'].sum())} bad bars were replaced.
- IEX volume is single-venue (A10).
- The event calendar has dates only, with no surprise values. "myforexfunds" is a defunct prop firm, not a FRED dataset.
- The universe was chosen ex post (survivorship). Costs are assumed, not measured.

**3. Out-of-sample results (sealed 2019-01-02..2026-09-21).**

| | Sharpe [95% CI] | CAGR | annual excess | alpha (t) | beta | profit/trade | hit rate | max DD | turnover/yr | exposure |
|---|---|---|---|---|---|---|---|---|---|---|
| Registered primary | {f(p_oos['sharpe'])} [{f(p_oos['sharpe_lo'], 2)}, {f(p_oos['sharpe_hi'], 2)}] | {pct(p_oos['CAGR'])} | {pct(p_oos['ann_excess'], 3)} | {pct(p_oos['alpha_ann'], 3)} ({f(p_oos['alpha_t'], 2)}) | {f(p_oos['beta'])} | {f(p_oos['profit_per_trade_pct'], 4)}% | {pct(p_oos['hit_rate'], 1)} | {pct(p_oos['max_dd'])} | {f(p_oos['turnover_ann'], 2)} | {f(p_oos['exposure'])} |
| Base 12/26/9 | {f(b_oos['sharpe'])} [{f(b_oos['sharpe_lo'], 2)}, {f(b_oos['sharpe_hi'], 2)}] | {pct(b_oos['CAGR'])} | {pct(b_oos['ann_excess'], 3)} | {pct(b_oos['alpha_ann'], 3)} ({f(b_oos['alpha_t'], 2)}) | {f(b_oos['beta'])} | {f(b_oos['profit_per_trade_pct'], 4)}% | {pct(b_oos['hit_rate'], 1)} | {pct(b_oos['max_dd'])} | {f(b_oos['turnover_ann'], 2)} | {f(b_oos['exposure'])} |
| ML-sized base | {f(m_oos['sharpe'])} [{f(m_oos['sharpe_lo'], 2)}, {f(m_oos['sharpe_hi'], 2)}] | {pct(m_oos['CAGR'])} | {pct(m_oos['ann_excess'], 3)} | {pct(m_oos['alpha_ann'], 3)} | {f(m_oos['beta'])} | {f(m_oos['profit_per_trade_pct'], 4)}% | {pct(m_oos['hit_rate'], 1)} | {pct(m_oos['max_dd'])} | {f(m_oos['turnover_ann'], 2)} | {f(m_oos['exposure'])} |
| SPY buy & hold | {f(s_oos['sharpe'])} [{f(s_oos['sharpe_lo'], 2)}, {f(s_oos['sharpe_hi'], 2)}] | {pct(s_oos['CAGR'])} | {pct(s_oos['ann_excess'], 3)} | n/a | 1 | n/a | n/a | {pct(s_oos['max_dd'])} | — | 1.0 |

- Monte Carlo ({CFG['bootstrap']['n_mc']:,} stationary-block paths): for the primary, P(Sharpe ≤ 0) = {f(mc_p['P_sharpe_le_0'])}, Sharpe 5–95% [{f(mc_p['sharpe_p05'], 2)}, {f(mc_p['sharpe_p95'], 2)}], P(Sharpe above SPY) = {f(mc_p['P_sharpe_above_SPY'])}. For the base, P(Sharpe ≤ 0) = {f(mc_b['P_sharpe_le_0'])}.
- Newey–West t-test of the primary's OOS daily excess returns: t = {f(tt_p['daily_t'], 2)} (p = {f(tt_p['daily_p'])}).
- IS → OOS: the primary's validation Sharpe of {f(GM.loc[PRIMARY_ID, 'val_sharpe'], 2)} became {f(p_oos['sharpe'], 2)} OOS. The base went from {f(b_is['sharpe'], 2)} IS to {f(b_oos['sharpe'], 2)} OOS.
- **Capacity/liquidity:** exposure is tiny (≈{f(p_oos['exposure'], 3)} average gross for the primary). CPER spreads (~10 bp half-spread assumed) dominate its P&L. The 2× leverage cap binds on SHY/IEF. Any edge would need measured NBBO costs before scaling.

**4. Overfitting controls (IS).**
- DSR = {f(DSR['DSR'].iloc[0])} with N = {len(P_MAIN):,}, and {f(DSR['DSR'].iloc[1])} with the effective N.
- SPA consistent p (median over seeds) = {f(wrc_runs['spa_consistent_p'].median())}; WRC p = {f(wrc_runs['wrc_p'].median())}.
- PBO = {f(PBO_EL['pbo'])} on the eligible universe and {f(PBO_ALL['pbo'])} on all variations.
- Holm discoveries: {int(holm)}.
- OOS placebo p-values: {', '.join(f"{r.placebo.split(' (')[0]} {f(r.p_value, 2)}" for r in PLACEBO_OOS.itertuples() if np.isfinite(r.p_value))}.

**5. Multi-split.** The re-selected #1 had a positive OOS Sharpe in {int((re_['sharpe'] > 0).sum())}/6 splits, was HAC-significant in {n_sig}/6 and beat SPY in {n_spy}/6.

**6. ML.** OOS Sharpe was {f(ml_sh)} for ML vs {f(base_sh)} for the base. Paired Sharpe-difference CI [{f(SD_CI[0])}, {f(SD_CI[1])}] → {'improvement' if G5_PASS else 'no demonstrated improvement'}. Primary: {PRIMARY_ML_STATUS}.

**7. Prop firm (PROVISIONAL rules, intraday adaptation only).** Chosen on IS: {o_is['config']}. P(pass) was {pct(o_is['P_pass'])} IS and {pct(o_oos['P_pass'])} OOS, against P(fail) {pct(o_oos['P_fail_mll'])} OOS and a gate of 25%.

**8. Failure modes, with evidence here.**
- **Regime shifts:** the IS → OOS Sharpe collapse, and multi-split instability.
- **Event shocks:** event-window effects were associational and not significant after BH (Section 2.4). The event-flag placebo on OOS is listed above.
- **Cost increases and slippage:** the primary's OOS Sharpe falls monotonically with cost (Section 3 table), and SHY/CPER costs are a large share of gross P&L.
- **Correlation breaks:** ETF↔futures flags in Section 6.1 (USO/CL 2020, Treasury duration mismatch).
- **Overfitting and multiple testing:** DSR, SPA, PBO and Holm above. The best-of-{len(P_MAIN):,} validation Sharpe is close to the expected maximum under the null.
- **Data leakage:** the perturbation and purge audits passed. Residual risks are the ex-post universe and a test window that was previously examined.

**9. Recommendations.** **{VERDICT}.** Do not paper-trade, fund or submit this to a prop firm. Next experiments, in order:
1. Measure pre-cost signal content on a **new, untouched universe** (e.g. sector/country ETFs) with base and primary frozen and pre-registered. A positive pre-cost spread with a CI > 0 would reopen the question.
2. Run a paired comparison against 2–3 simple mean-reversion rules (N-day low in an uptrend, 2-day RSI) to see whether MACD adds anything.
3. Shrink the search to ≤ 20 economically motivated variants, and report DSR including all trials already run.
4. Compute all volume features from consolidated volume.
5. Design a native RTH-intraday strategy for any prop-firm use instead of translating swing trades.

**Paper-trading gates, if a future spec ever passes 1–3 on fresh data:**
- Frozen hashes of code, config and universe.
- Realised costs ≤ assumed after 50 fills.
- Replay parity (0 signal mismatches).
- A kill switch at the OOS Monte Carlo p05 drawdown.
- No size-up until the Sharpe CI lower bound is > 0.
- Human approval before any capital.

**Disclaimer.** This is a research artifact only. It is not financial advice and guarantees no performance; past and simulated results do not predict future results. Live trading is out of scope.
"""
md(REPORT)
print(f"Notebook complete · total {elapsed()}")

**Comparison table (IS = 2004–2018 after warm-up; OOS = 2019-01-02..2026-09-21)** · sample: `IS / OOS`

,sharpe,sharpe_lo,sharpe_hi,CAGR,ann_excess,ann_vol,sortino,max_dd,alpha_ann,alpha_t,beta,hit_rate,profit_per_trade_pct,trades_per_year,exposure,turnover_ann,prop P(pass) / P(fail)
base MACD · IS,-0.347325,-0.832921,0.113337,0.008292,-0.003964,0.011446,-0.292621,-0.029813,-0.004496,-1.581303,0.006827,0.471276,-0.046612,68.702947,0.106743,8.349208,n/a
base MACD · OOS,-0.271187,-0.899415,0.373060,0.023871,-0.003235,0.012014,-0.222138,-0.021888,-0.004528,-1.106244,0.008431,0.507154,-0.037231,81.705155,0.098747,8.115290,n/a
registered primary · IS,0.554195,0.043554,1.104244,0.015924,0.003532,0.006441,0.373453,-0.013977,0.002972,1.756220,0.007191,0.671835,0.188176,25.889036,0.054736,3.124344,n/a
registered primary · OOS,0.242895,-0.456825,0.925145,0.028541,0.001258,0.005301,0.168455,-0.005174,0.000490,0.267203,0.005005,0.620513,0.107231,25.329897,0.037530,2.077544,n/a
ML-sized base · IS-pseudo-OOS (2008–18),-0.221614,-0.764123,0.369502,0.001895,-0.002658,0.011997,-0.186322,-0.034734,-0.003241,-0.916434,0.006856,0.471576,-0.031560,70.439870,0.098402,7.759934,n/a
ML-sized base · OOS,-0.132326,-0.769483,0.519146,0.025436,-0.001696,0.012907,-0.108283,-0.022395,-0.003504,-0.794778,0.011789,0.518987,-0.010480,71.832990,0.091524,7.557333,n/a
SPY B&H · IS,0.425960,-0.021252,0.904147,0.076238,0.077907,0.182886,0.403340,-0.551895,n/a,n/a,1.000000,n/a,n/a,n/a,1.000000,0.000000,n/a
SPY B&H · OOS,0.795504,0.178159,1.549269,0.175279,0.153261,0.192661,0.753704,-0.337173,n/a,n/a,1.000000,n/a,n/a,n/a,1.000000,0.000000,n/a
prop sim · IS,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,0.00% / 0.00% (primary @ $200)
prop sim · OOS,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,n/a,0.00% / 0.00% (primary @ $200)


**Decision gates** · sample: `n/a`

,gate,pass,outcome
0,G0 data integrity,True,PASS
1,G1 IS baseline edge,False,FAIL
2,G2 search significance,False,FAIL
3,G3 sealed OOS,False,FAIL
4,G4 multi-split robustness,False,FAIL
5,G5 ML improvement,False,FAIL
6,G6 prop viability (PROVISIONAL),False,FAIL
7,G7 paper trading,False,NOT REACHED


**Leakage / overfitting checklist (PASS / FAIL / WARN / BLOCKED)** · sample: `n/a`

,section,item,status,evidence
0,2,No look-ahead: perturbing data after k leaves features/signals/equity ≤ k un...,PASS,"[{'ticker': 'SPY', 'cut': 2012-06-29, 'features_identical': True, 'signals_i..."
1,2,G1 IS baseline edge (Sharpe > 0 and NW p < 0.05),FAIL,"Sharpe -0.347, NW p 0.159"
2,3,"G2 search significance (DSR ≥ 0.95, SPA_c p < 0.05, PBO < 0.5, ≥1 Holm disco...",FAIL,DSR 0.170; SPA_c p 0.644; WRC p 0.796; PBO 0.728; Holm 0
3,3,G3 sealed OOS: registration before access; primary OOS Sharpe > 0 with CI lo...,FAIL,"registered 2026-10-25T21:58:13Z, first access 2026-10-25T22:02:40Z UTC; OOS ..."
4,4,G4 multi-split: re-selected #1 positive & HAC-significant OOS in ≥4/6 and be...,FAIL,"positive 2/6, significant 0/6, beats SPY 0/6"
5,5,"ML leakage audit (purge, embargo, thresholds from prior year, no shuffle, PI...",PASS,19 folds; PIT True
6,5,G5 ML improves OOS risk-adjusted return (Sharpe > base and paired CI > 0),FAIL,"ML -0.132 vs base -0.271; paired Sharpe-diff CI [-0.068, 0.335]"
7,5,ML on the registered primary (≥500 IS signals),FAIL,"run: OOS Sharpe ML 0.342 vs primary 0.243, paired CI [-0.014, 0.205]; leakag..."
8,6,G6 prop viability: OOS P(pass) ≥ 25% and > P(fail) (PROVISIONAL rules),FAIL,"OOS P(pass) 0.00%, P(fail) 0.00%"
9,6,Prop admissibility of the swing strategy (no overnight holds at Topstep),FAIL,multi-day holds forbidden; only the intraday adaptation is simulated


**Sealed-test access log (this run)** · sample: `n/a`

,utc,what,registration
0,2026-10-25T22:02:40Z,candidates + base + SPY on the test window,7616018ed24b
1,2026-10-25T22:02:42Z,cost sensitivity + WRC/SPA + placebos on the test window,7616018ed24b
2,2026-10-25T22:04:39Z,"full-history grid (multi-split, IS→OOS rank diagnostic)",7616018ed24b
3,2026-10-25T22:05:18Z,ML: base strategy signals/trades over full history (labels ≤ test year are u...,7616018ed24b
4,2026-10-25T22:06:26Z,prop simulation on the test window,7616018ed24b



### Verdict: **REJECT**

**1. Strategy and exact rules.** MACD mean reversion on 11 ETFs (SPY QQQ DIA IWM GLD SLV USO CPER TLT IEF SHY; VIX/VXX not traded). Signals are evaluated at close t and filled at open t+1. Each trade risks 1% of sleeve equity to an ATR(14) stop (notional ≤ 2× equity). Costs are per-ETF bps assumptions plus borrow and financing. The portfolio is the equal-weight average of the sleeves.
- *Base (A5):* `{"fast": 12, "slow": 26, "signal": 9, "signal_kind": "hist_z", "z_window": 60, "entry_z": 1.5, "exit_z": 0.0, "require_turn": true, "stop_atr": 2.0, "tp_atr": 0.0, "max_hold": 10, "atr_window": 14, "trend_filter": "none", "vol_filter": "none", "volume_filter": "none", "event_filter": "none", "side": "both"}`
- *Registered primary (id 4115, selected on 2003–2018 only, registration sha1 `7616018ed24b`):* `{"fast": 10, "slow": 30, "signal": 7, "z_window": 40, "entry_z": 1.5, "exit_z": 1.0, "stop_atr": 2.5, "tp_atr": 4.0, "max_hold": 20, "require_turn": false, "signal_kind": "macd_z", "trend_filter": "own_sma200_align", "vol_filter": "vix_lt_30", "volume_filter": "relvol_lt_1", "event_filter": "only_event_window", "side": "long_only"}`

**2. Data limitations.**
- The vendor files splice two differently adjusted sources on 2022-03-07; prices were rebased to the yfinance total-return basis and 240 bad bars were replaced.
- IEX volume is single-venue (A10).
- The event calendar has dates only, with no surprise values. "myforexfunds" is a defunct prop firm, not a FRED dataset.
- The universe was chosen ex post (survivorship). Costs are assumed, not measured.

**3. Out-of-sample results (sealed 2019-01-02..2026-09-21).**

| | Sharpe [95% CI] | CAGR | annual excess | alpha (t) | beta | profit/trade | hit rate | max DD | turnover/yr | exposure |
|---|---|---|---|---|---|---|---|---|---|---|
| Registered primary | 0.243 [-0.46, 0.93] | 2.85% | 0.126% | 0.049% (0.27) | 0.005 | 0.1072% | 62.1% | -0.52% | 2.08 | 0.038 |
| Base 12/26/9 | -0.271 [-0.90, 0.37] | 2.39% | -0.324% | -0.453% (-1.11) | 0.008 | -0.0372% | 50.7% | -2.19% | 8.12 | 0.099 |
| ML-sized base | -0.132 [-0.77, 0.52] | 2.54% | -0.170% | -0.350% | 0.012 | -0.0105% | 51.9% | -2.24% | 7.56 | 0.092 |
| SPY buy & hold | 0.796 [0.18, 1.55] | 17.53% | 15.326% | n/a | 1 | n/a | n/a | -33.72% | — | 1.0 |

- Monte Carlo (10,000 stationary-block paths): for the primary, P(Sharpe ≤ 0) = 0.244, Sharpe 5–95% [-0.34, 0.82], P(Sharpe above SPY) = 0.102. For the base, P(Sharpe ≤ 0) = 0.800.
- Newey–West t-test of the primary's OOS daily excess returns: t = 0.67 (p = 0.501).
- IS → OOS: the primary's validation Sharpe of 1.58 became 0.24 OOS. The base went from -0.35 IS to -0.27 OOS.
- **Capacity/liquidity:** exposure is tiny (≈0.038 average gross for the primary). CPER spreads (~10 bp half-spread assumed) dominate its P&L. The 2× leverage cap binds on SHY/IEF. Any edge would need measured NBBO costs before scaling.

**4. Overfitting controls (IS).**
- DSR = 0.170 with N = 12,000, and 0.530 with the effective N.
- SPA consistent p (median over seeds) = 0.644; WRC p = 0.796.
- PBO = 0.728 on the eligible universe and 0.393 on all variations.
- Holm discoveries: 0.
- OOS placebo p-values: random signals 0.63, shuffled returns 0.34, randomized parameters 0.19, random event flags 0.37.

**5. Multi-split.** The re-selected #1 had a positive OOS Sharpe in 2/6 splits, was HAC-significant in 0/6 and beat SPY in 0/6.

**6. ML.** OOS Sharpe was -0.132 for ML vs -0.271 for the base. Paired Sharpe-difference CI [-0.068, 0.335] → no demonstrated improvement. Primary: run: OOS Sharpe ML 0.342 vs primary 0.243, paired CI [-0.014, 0.205].

**7. Prop firm (PROVISIONAL rules, intraday adaptation only).** Chosen on IS: primary @ $200. P(pass) was 0.00% IS and 0.00% OOS, against P(fail) 0.00% OOS and a gate of 25%.

**8. Failure modes, with evidence here.**
- **Regime shifts:** the IS → OOS Sharpe collapse, and multi-split instability.
- **Event shocks:** event-window effects were associational and not significant after BH (Section 2.4). The event-flag placebo on OOS is listed above.
- **Cost increases and slippage:** the primary's OOS Sharpe falls monotonically with cost (Section 3 table), and SHY/CPER costs are a large share of gross P&L.
- **Correlation breaks:** ETF↔futures flags in Section 6.1 (USO/CL 2020, Treasury duration mismatch).
- **Overfitting and multiple testing:** DSR, SPA, PBO and Holm above. The best-of-12,000 validation Sharpe is close to the expected maximum under the null.
- **Data leakage:** the perturbation and purge audits passed. Residual risks are the ex-post universe and a test window that was previously examined.

**9. Recommendations.** **REJECT.** Do not paper-trade, fund or submit this to a prop firm. Next experiments, in order:
1. Measure pre-cost signal content on a **new, untouched universe** (e.g. sector/country ETFs) with base and primary frozen and pre-registered. A positive pre-cost spread with a CI > 0 would reopen the question.
2. Run a paired comparison against 2–3 simple mean-reversion rules (N-day low in an uptrend, 2-day RSI) to see whether MACD adds anything.
3. Shrink the search to ≤ 20 economically motivated variants, and report DSR including all trials already run.
4. Compute all volume features from consolidated volume.
5. Design a native RTH-intraday strategy for any prop-firm use instead of translating swing trades.

**Paper-trading gates, if a future spec ever passes 1–3 on fresh data:**
- Frozen hashes of code, config and universe.
- Realised costs ≤ assumed after 50 fills.
- Replay parity (0 signal mismatches).
- A kill switch at the OOS Monte Carlo p05 drawdown.
- No size-up until the Sharpe CI lower bound is > 0.
- Human approval before any capital.

**Disclaimer.** This is a research artifact only. It is not financial advice and guarantees no performance; past and simulated results do not predict future results. Live trading is out of scope.


Notebook complete · total 9.1 min
